# 21 — YOLOv8s e treinamento ampliado

Esta rodada reabre o desenvolvimento com **treinamento de um modelo maior**, a
partir de pesos públicos. Usa as 12 classes existentes, até **12 mil imagens
Open Images selecionadas do TRAIN histórico de 80 mil**, além de Home TRAIN.
Não exige novos rótulos, formulários ou edição de parâmetros.

**Objetivo de desenvolvimento: precisão do detector ≥80% e recall ≥70%, ao
mesmo tempo, na validação externa.** Isso é uma meta a verificar, não uma
promessa. Aumentar a confiança e deixar de reconhecer a maioria dos objetos
não atende ao objetivo. Todas as classes continuam na avaliação.

- Até **40 épocas**, parada antecipada após 8 épocas sem melhora externa.
- Cobertura parcial de anotações mantida na perda e nas métricas.
- Escolha de pesos e confiança somente nas validações já expostas.
- Checkpoint completo a cada época, com retomada automática pelo Drive.
- As 300 imagens avaliadas na rodada 20 ficam fora de treino e seleção.

Esta etapa produz resultados de **desenvolvimento**. Uma nova avaliação final,
com dados independentes, será necessária depois da escolha da nova versão.


## 1. Preparar o ambiente

Importe como **novo notebook no Kaggle**, ative **Internet** e uma **GPU T4**.
Habilite o Secret existente **GDRIVE_RCLONE_CONFIG_B64** em **Add-ons → Secrets**.
Use **Save Version → Save & Run All**, sem copiar células de versões anteriores.
O código usa somente a primeira GPU, mesmo se o Kaggle disponibilizar duas.

**Planeje várias horas.** Como referência inicial ampla: **30–120 minutos ou
mais para preparação** e **4–12 horas ou mais para treinamento em T4**. Não é
um tempo medido nesta base. Downloads, número de imagens elegíveis, validação
e parada antecipada mudam bastante o total. Após a primeira época, o notebook
mostra uma estimativa baseada no tempo observado.

O treino pausa no fim de uma época após aproximadamente 8 horas de sessão de
treino. A preparação tem duração adicional. Se aparecer **pausado**, execute
**Run All** novamente; o checkpoint é recuperado, sem preencher nada.


In [ ]:
%pip install -q ultralytics==8.4.150 scipy pyyaml


In [ ]:
import os
os.environ['YOLO_OFFLINE']='true'
os.environ['YOLO_CONFIG_DIR']='/tmp/tcc_yolo21_config'
os.environ['WANDB_DISABLED']='true'
import sys,json,hashlib,platform,shutil,time,random
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import PIL,torch,ultralytics
from IPython.display import display,FileLink,HTML
if ultralytics.__version__!='8.4.150':
    raise RuntimeError('A versão importada difere da fixada. Reinicie a sessão e use Run All.')
if not torch.cuda.is_available():
    raise RuntimeError('Ative GPU T4 nas configurações do Kaggle e execute Run All. Esta rodada contém treinamento longo.')
DEVICE=0
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
torch.backends.cudnn.benchmark=False
random.seed(21002026);np.random.seed(21002026);torch.manual_seed(21002026)
AMBIENTE={'python':platform.python_version(),'numpy':np.__version__,'pillow':PIL.__version__,
    'torch':torch.__version__,'ultralytics':ultralytics.__version__,'cuda_runtime':torch.version.cuda,
    'dispositivo':str(DEVICE),'gpu':torch.cuda.get_device_name(0),'amp':False,'tf32':False}
BASE_ID='2887e24c8a53dbbcb748f196'

PASTA_LOCAL=Path('/tmp/tcc_treinamento21')/BASE_ID
PASTA_BOOT=Path('/kaggle/working/resultados_tcc')/('preparacao21_'+BASE_ID)
PASTA_LOCAL.mkdir(parents=True,exist_ok=True);PASTA_BOOT.mkdir(parents=True,exist_ok=True)
PASTA_RESULTADOS=PASTA_BOOT;NOME_EXPERIMENTO=PASTA_BOOT.name;DRIVE=None
INICIO=time.monotonic();ULTIMA_ETAPA=INICIO;TEMPOS=[]
print('GPU:',AMBIENTE['gpu'],'| YOLOv8s | até 40 épocas | entrada 768')

EXPECTED={'home_zip': 'ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676', 'test_home': 'f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89', 'preparacao': '9865042654c51e83955c2aeb186a1b5e27e1028a5c56f1f163e2891abc0bec49', 'divisao_10': 'a5dd35ff484dda77e8988aefdc961f05d07ea409d1336cbe90c4a5420632bab9', 'manifesto_pacotes': '655094dea28755a7bddc817ae66660d06fab28ccb8f9d140d3ffebb07f03a0d4', 'dataset_novo': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'zip_novo': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'politica': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'associador': '1da5dff21a6c6f6ec53d041247ce18d012008c4b26023b52c3fd2745f4c4105d', 'pacote_19': '33b9937613b3cddae984a4499adc2788318cb957628c858d525bc2a86be674e4', 'pasta_19': 'resultados/consolidacao19_4d5ea2227cb3d7146cc78d8b', 'oi_root': 'datasets/openimages_v7_tcc_100k_v1', 'pasta10': 'resultados/yolov8n_home_openimages_80_20_1ep_kaggle_20261015_030228_068010', 'dataset_url': 'datasets/validacao_independente_11_v1/695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0/validacao_independente_11_695b445e15c9.zip', 'pretrained_url': 'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8s.pt', 'pretrained_sha256': '1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a'}
MODULOS_EMBUTIDOS={'conf14_data.py': '"""Entradas autenticadas do diagnóstico 14; extrai somente Home VAL."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\nfrom PIL import Image\nimport diag11_core as core\n\nCOHORTS = (\'home_val_historica\', \'openimages_val_nova\')\nBOOLS = {\'True\':True, \'False\':False}\n\ndef rows(raw):\n    return list(csv.DictReader(io.StringIO(raw.decode(\'utf-8-sig\'))))\n\ndef parse_events(raw):\n    result = rows(raw)\n    for r in result:\n        r[\'classe_id\'] = int(r[\'classe_id\'])\n        for field in [\'bbox_ref\',\'bbox_pred\']:\n            r[field] = json.loads(r[field]) if r[field] else None\n        for field in [\'ha_alerta_ref\',\'ha_alerta_pred\']:\n            r[field] = BOOLS[r[field]]\n        r[\'confianca\'] = float(r[\'confianca\']) if r[\'confianca\'] else None\n    return result\n\ndef gt_key(r):\n    return (r[\'id\'],int(r[\'classe_id\']),tuple(round(float(v),6) for v in r[\'bbox_ref\']))\n\ndef gt_map(events):\n    result = {}\n    for r in events:\n        if r[\'bbox_ref\'] is None: continue\n        key = gt_key(r)\n        if key in result: raise ValueError(\'Objeto GT repetido nos registros.\')\n        result[key] = r\n    return result\n\ndef load_source12(path, expected, output=None):\n    if core.sha256_file(path)!=expected[\'pacote_12\']:\n        raise RuntimeError(\'O pacote 12 mudou em relação ao recebido.\')\n    bundle = {\'evaluations\':{}}\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())):\n            raise RuntimeError(\'Pacote 12 com membros repetidos.\')\n        for name in z.namelist(): core.safe_member(name)\n        if z.testzip() is not None: raise RuntimeError(\'Falha de CRC no pacote 12.\')\n        cfg = json.loads(z.read(\'configuracao_experimento_12.json\'))\n        summary = json.loads(z.read(\'resumo_experimento_12.json\'))\n        if not summary[\'concluido\'] or summary[\'reserva_futura_inferida\'] or summary[\'test_home_inferido\'] or summary[\'test_oi20k_inferido\']:\n            raise RuntimeError(\'Execução 12 incompatível com o protocolo.\')\n        if cfg[\'expected\'][\'candidato\']!=expected[\'candidato\'] or cfg[\'expected\'][\'politica\']!=expected[\'politica\'] or summary[\'dataset_sha256\']!=expected[\'dataset_novo\']:\n            raise RuntimeError(\'Referências ou política diferentes do experimento 12.\')\n        if cfg[\'codigo_embutido_sha256\']!=expected[\'codigo_12\']:\n            raise RuntimeError(\'Código do experimento 12 diferente do recebido.\')\n        for name, digest in expected[\'codigo_12\'].items():\n            if hashlib.sha256(z.read(\'codigo/\'+name)).hexdigest()!=digest:\n                raise RuntimeError(\'Código alterado no pacote 12: \'+name)\n        rule_raw = z.read(\'regra_selecao_congelada.json\')\n        if hashlib.sha256(rule_raw).hexdigest()!=cfg[\'regra_sha256\']:\n            raise RuntimeError(\'A regra original do experimento 12 foi alterada.\')\n        selection = json.loads(z.read(\'selecao_checkpoints.json\'))\n        checkpoint = next(e for e in selection[\'assessed\'] if e[\'id\']==\'B_home_replay_e03\')\n        if checkpoint[\'weight_sha256\']!=expected[\'B_e03\']:\n            raise RuntimeError(\'Checkpoint B_e03 diferente do esperado.\')\n        for model,prefix in [(\'baseline\',\'referencias/baseline\'),(\'candidato_10\',\'referencias/candidato_10\'),(\'B_e03\',\'avaliacoes/B_home_replay_e03\')]:\n            bundle[\'evaluations\'][model] = {}\n            for cohort in COHORTS:\n                base=prefix+\'/\'+cohort+\'/\'\n                members={name:z.read(base+name) for name in [\'metricas.json\',\'alertas_por_imagem.csv\',\'avaliacao_objetos.csv\']}\n                metrics=json.loads(members[\'metricas.json\'])\n                images=rows(members[\'alertas_por_imagem.csv\'])\n                for r in images:\n                    for key in [\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']:\n                        r[key]=int(r[key])\n                if len({r[\'id\'] for r in images})!=len(images) or len(images)!=metrics[\'imagens\']:\n                    raise RuntimeError(\'IDs ou quantidades inconsistentes nos registros 12.\')\n                for field, n in [(\'tp_alerta\',metrics[\'alertas_por_objeto\'][\'tp\']),(\'fn_alerta\',metrics[\'alertas_por_objeto\'][\'fn\']),(\'fp_alerta\',metrics[\'alertas_por_objeto\'][\'fp\']),(\'alta_ref\',metrics[\'alta_ref\']),(\'alta_sem_alerta\',metrics[\'alta_sem_alerta\'])]:\n                    if sum(r[field] for r in images)!=n: raise RuntimeError(\'Contagens inconsistentes no pacote 12.\')\n                events=parse_events(members[\'avaliacao_objetos.csv\'])\n                gt_map(events)\n                bundle[\'evaluations\'][model][cohort]={\'summary\':metrics,\'images\':images,\'events\':events}\n                if output is not None:\n                    dest=Path(output)/\'referencias_12\'/model/cohort\n                    dest.mkdir(parents=True,exist_ok=True)\n                    for name, raw in members.items(): (dest/name).write_bytes(raw)\n        for cohort in COHORTS:\n            target=gt_map(bundle[\'evaluations\'][\'B_e03\'][cohort][\'events\'])\n            for model in [\'baseline\',\'candidato_10\']:\n                source=gt_map(bundle[\'evaluations\'][model][cohort][\'events\'])\n                if source.keys()!=target.keys(): raise RuntimeError(\'Modelos do 12 usam referências GT diferentes.\')\n                if any(source[k][\'ha_alerta_ref\']!=target[k][\'ha_alerta_ref\'] or source[k][\'prioridade_ref\']!=target[k][\'prioridade_ref\'] for k in target):\n                    raise RuntimeError(\'Referências de prioridade diferentes entre modelos.\')\n        bundle.update(config=cfg,summary=summary,rule=json.loads(rule_raw),\n                      audit=json.loads(z.read(\'auditoria_dados.json\')))\n    return bundle\n\ndef extract_home_val(path, destination, expected_zip, expected_test, expected_policy):\n    if core.sha256_file(path)!=expected_zip: raise RuntimeError(\'ZIP Home alterado.\')\n    destination=Path(destination)\n    records=[];frozen=[]\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())): raise RuntimeError(\'Membros repetidos no Home.\')\n        for name in z.namelist():core.safe_member(name)\n        manifest=json.loads(z.read(\'verificacao_preparacao.json\'))\n        if (manifest[\'test\'][\'sha256\']!=expected_test or manifest[\'test\'][\'imagens\']!=403 or\n            {int(k):v for k,v in manifest[\'classes\'].items()}!=dict(enumerate(core.NAMES))):\n            raise RuntimeError(\'Manifesto Home incompatível.\')\n        root=manifest[\'dataset_dir\']\n        items={r[\'arquivo\']:r for r in manifest[\'arquivos\']}\n        if len(items)!=len(manifest[\'arquivos\']): raise RuntimeError(\'Manifesto Home repetido.\')\n        train_hash=set();val_hash=set();all_hash=set();counts=Counter()\n        for name,item in items.items():\n            is_image=name.startswith(root+\'/images/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}\n            if not is_image:continue\n            all_hash.add(item[\'sha256\'])\n            if name.startswith(root+\'/images/train/raw_train/\'):\n                counts[\'train\']+=1;train_hash.add(item[\'sha256\'])\n            elif name.startswith(root+\'/images/val/raw_train/\'):\n                counts[\'val\']+=1;val_hash.add(item[\'sha256\'])\n        if counts!={\'train\':1824,\'val\':456} or train_hash&val_hash:\n            raise RuntimeError(\'A divisão atual de Home mudou ou tem bytes repetidos entre TRAIN e VAL.\')\n        for name in z.namelist():\n            if not(name.startswith(\'codigo/\') and name.endswith(\'.py\')):continue\n            raw=z.read(name)\n            if name in items and hashlib.sha256(raw).hexdigest()!=items[name][\'sha256\']:\n                raise RuntimeError(\'Código Home alterado.\')\n            out=destination/name;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n            frozen.append((out,hashlib.sha256(raw).hexdigest()))\n        for name,item in sorted(items.items()):\n            if not(name.startswith(root+\'/images/val/raw_train/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}):continue\n            label=name.replace(\'/images/\',\'/labels/\').rsplit(\'.\',1)[0]+\'.txt\'\n            if label not in items:raise RuntimeError(\'Label Home ausente.\')\n            paths={}\n            for field,member in [(\'image\',name),(\'label\',label)]:\n                raw=z.read(member)\n                if hashlib.sha256(raw).hexdigest()!=items[member][\'sha256\']:raise RuntimeError(\'Home alterado: \'+member)\n                out=destination/member;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n                paths[field]=out;frozen.append((out,items[member][\'sha256\']))\n            with Image.open(paths[\'image\']) as im:\n                width,height=im.size;im.verify()\n            records.append({\'id\':\'home:\'+name,\'name\':name.split(\'/raw_train/\')[-1],**paths,\n                \'role\':\'home_val\',\'coorte\':\'home_val_historica\',\'width\':width,\'height\':height,\n                \'known_mask\':core.ALL_KNOWN,\'sha256\':item[\'sha256\'],\'label_sha256\':items[label][\'sha256\'],\n                \'gt\':core.parse_yolo(paths[\'label\'].read_bytes(),width,height,source=label)})\n    if len(records)!=456 or core.sha256_file(destination/\'codigo/src/politica_alertas.py\')!=expected_policy:\n        raise RuntimeError(\'Quantidade Home VAL ou política V2 incompatível.\')\n    audit={\'home_train_manifesto\':1824,\'home_val_extraida\':456,\'home_train_extraida\':False,\n           \'home_test_extraido\':False,\'test_home_manifesto\':403,\'intersecao_bytes_train_val\':0,\n           \'historico_original_baseline_nao_comprovado_pela_separacao_atual\':True}\n    return records,all_hash,frozen,audit\n\ndef validate_cohorts(cohorts,bundle,reserved_ids):\n    if set(cohorts)!=set(COHORTS) or {k:len(v) for k,v in cohorts.items()}!={\'home_val_historica\':456,\'openimages_val_nova\':1000}:\n        raise ValueError(\'Coortes diferentes das duas validações congeladas.\')\n    for cohort,records in cohorts.items():\n        ids={r[\'id\'] for r in records}\n        if len(ids)!=len(records) or ids&set(reserved_ids):raise ValueError(\'ID repetido ou reservado na avaliação.\')\n        expected_role=\'home_val\' if cohort==\'home_val_historica\' else \'val_nova\'\n        if any(r[\'role\']!=expected_role for r in records):raise ValueError(\'Papel incompatível com a coorte de validação.\')\n        old=bundle[\'evaluations\'][\'B_e03\'][cohort]\n        if ids!={r[\'id\'] for r in old[\'images\']}:raise ValueError(\'IDs diferentes dos avaliados no 12.\')\n        actual={}\n        for r in records:\n            for g in r[\'gt\']:\n                if r[\'known_mask\']&(1<<g[\'classe_id\']):\n                    k=(r[\'id\'],g[\'classe_id\'],tuple(round(float(x),6) for x in g[\'bbox\']))\n                    if k in actual:raise ValueError(\'GT repetido na entrada atual.\')\n                    actual[k]=True\n        if actual.keys()!=gt_map(old[\'events\']).keys():raise ValueError(\'Caixas avaliáveis diferentes das referências originais.\')\n    return True\n', 'conf14_evaluation.py': '"""Varredura de confiança: uma inferência por imagem, política V2 fixa."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport diag11_core as core\nimport conf14_data as data\nfrom exp12_protocol import judge\n\nTHRESHOLDS=(.05,.10,.15,.20,.25,.30,.40)\nPREDICT_ARGS={\'conf\':.001,\'imgsz\':640,\'rect\':True,\'iou\':.70,\'max_det\':300,\'half\':False,\'batch\':1}\nVERSION=\'14-confianca-automatica-1.0\'\n\ndef point_name(conf):return \'conf_\'+f\'{conf:.2f}\'.replace(\'.\',\'\')\n\n\ndef cache_request(records,weight_sha,environment):\n    if not records or len({r[\'id\'] for r in records})!=len(records):raise ValueError(\'Cache exige IDs únicos.\')\n    if any(r[\'role\'] not in {\'home_val\',\'val_nova\'} for r in records):raise ValueError(\'Cache somente para validação.\')\n    signature=[{k:r[k] for k in [\'id\',\'sha256\',\'label_sha256\',\'width\',\'height\',\'known_mask\',\'role\']} for r in records]\n    return {\'versao\':VERSION,\'peso_sha256\':weight_sha,\'parametros\':PREDICT_ARGS,\n            \'ambiente\':environment,\'coorte\':records[0][\'coorte\'],\'imagens\':len(records),\n            \'entradas_sha256\':hashlib.sha256(core.canonical(signature).encode()).hexdigest()}\n\ndef read_cache(path,records):\n    expected={r[\'id\']:r for r in records};result={}\n    with gzip.open(path,\'rt\',encoding=\'utf-8\') as f:\n        for line in f:\n            row=json.loads(line);iid=row[\'id\']\n            if iid not in expected or iid in result:raise ValueError(\'Cache com ID desconhecido ou repetido.\')\n            r=expected[iid]\n            if row[\'image_sha256\']!=r[\'sha256\'] or row[\'label_sha256\']!=r[\'label_sha256\']:\n                raise ValueError(\'Cache pertence a outros bytes de imagem/label.\')\n            validate_predictions(row[\'predicoes\'],r);result[iid]=row[\'predicoes\']\n    if result.keys()!=expected.keys():raise ValueError(\'Cache incompleto.\')\n    return result\n\n\ndef event(record,kind,g,p,overlap,pred_low,conf):\n    row=core.record_event(record,\'B_e03\',kind,g,p,overlap,pred_low)\n    row[\'limiar_confianca\']=conf\n    row[\'melhor_confianca_mesma_classe\']=None\n    if kind==\'objeto_nao_detectado\':\n        flags=[v for v in row[\'indicios\'].split(\'|\') if v and v not in {\'deteccao_abaixo_da_confianca_025\',\'sobreposicao_com_previsao_de_outra_classe\',\'omissao_sem_indicio_automatico\'}]\n        matrix=core.iou_matrix([g],pred_low)[0]\n        same=[i for i,p0 in enumerate(pred_low) if p0[\'classe_id\']==g[\'classe_id\']]\n        if same:\n            best=max(same,key=lambda i:matrix[i]);row[\'melhor_confianca_mesma_classe\']=pred_low[best][\'confianca\']\n            if matrix[best]>=.5 and pred_low[best][\'confianca\']<conf:flags.append(\'deteccao_abaixo_do_limiar_avaliado\')\n        if any(matrix[i]>=.5 and p0[\'classe_id\']!=g[\'classe_id\'] and p0[\'confianca\']>=conf for i,p0 in enumerate(pred_low)):\n            flags.append(\'sobreposicao_com_previsao_de_outra_classe\')\n        row[\'indicios\']=\'|\'.join(flags or [\'omissao_sem_indicio_automatico\'])\n    return row\n\ndef analyse(record,pred_low,conf,policy,matcher):\n    if record[\'role\'] not in {\'home_val\',\'val_nova\'} or conf not in THRESHOLDS:\n        raise ValueError(\'Papel ou limiar fora do protocolo.\')\n    known=int(record[\'known_mask\'])\n    gt=core.with_policy([g for g in record[\'gt\'] if known&(1<<g[\'classe_id\'])],record[\'width\'],record[\'height\'],policy)\n    low=[p for p in pred_low if known&(1<<p[\'classe_id\'])]\n    all_pred=core.with_policy([p for p in pred_low if p[\'confianca\']>=conf],record[\'width\'],record[\'height\'],policy)\n    pred=[p for p in all_pred if known&(1<<p[\'classe_id\'])]\n    pairs,missing,extras=matcher(gt,pred)\n    events=[event(record,\'correspondencia\',gt[gi],pred[pi],float(iou),low,conf) for gi,pi,iou in pairs]\n    events += [event(record,\'objeto_nao_detectado\',gt[gi],None,None,low,conf) for gi in missing]\n    events += [event(record,\'previsao_nao_associada\',None,pred[pi],None,low,conf) for pi in extras]\n    messages=[policy.gerar_mensagem(p,\'medio\') for p in all_pred if core.alert_on(p)]\n    if any(not isinstance(s,str) or not s.strip() for s in messages):raise RuntimeError(\'Alerta sem mensagem.\')\n    im={\'coorte\':record[\'coorte\'],\'imagem\':record[\'name\'],\'id\':record[\'id\'],\'modelo\':\'B_e03\',\'limiar_confianca\':conf,\n        \'tp_alerta\':sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'fn_alerta\':sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in events),\n        \'fp_alerta\':sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'alta_ref\':sum(g[\'prioridade\']==\'alto\' for g in gt),\n        \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in events),\n        \'tp_detector\':len(pairs),\'fn_detector\':len(missing),\'fp_detector\':len(extras),\n        \'gt_avaliaveis\':len(gt),\'gt_ignorados\':len(record[\'gt\'])-len(gt),\n        \'predicoes_ignoradas_conf\':len(all_pred)-len(pred),\n        \'cobertura_completa_alertas\':known&core.ALERT_MASK==core.ALERT_MASK,\n        \'ha_alerta_ref\':any(core.alert_on(g) for g in gt),\'ha_alerta_pred\':any(core.alert_on(p) for p in pred),\n        \'mensagem_alerta\':\'\\n\'.join(messages)}\n    return im,events\n\ndef summarise(images,events,conf):\n    if not images:raise ValueError(\'Validação vazia.\')\n    total=lambda k:sum(r[k] for r in images)\n    complete=[r for r in images if r[\'cobertura_completa_alertas\']]\n    summary={\'modelo\':\'B_e03\',\'coorte\':images[0][\'coorte\'],\'imagens\':len(images),\'conf_alertas\':conf,\n        \'alertas_por_objeto\':core.alert_metrics(total(\'tp_alerta\'),total(\'fn_alerta\'),total(\'fp_alerta\')),\n        \'alta_ref\':total(\'alta_ref\'),\'alta_sem_alerta\':total(\'alta_sem_alerta\'),\n        \'taxa_alta_sem_alerta\':core.ratio(total(\'alta_sem_alerta\'),total(\'alta_ref\')),\n        \'detector_no_limiar\':core.alert_metrics(total(\'tp_detector\'),total(\'fn_detector\'),total(\'fp_detector\')),\n        \'gt_ignorados_sem_cobertura\':total(\'gt_ignorados\'),\'predicoes_ignoradas_conf\':total(\'predicoes_ignoradas_conf\'),\n        \'imagens_com_cobertura_completa_alertas\':len(complete),\n        \'alertas_por_imagem_cobertura_completa\':core.alert_metrics(\n            sum(r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete),\n            sum(r[\'ha_alerta_ref\'] and not r[\'ha_alerta_pred\'] for r in complete),\n            sum(not r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete)),\n        \'referencia\':\'Concordância com política V2 nas caixas; não valida risco físico.\',\n        \'nao_e_openimages_challenge\':True,\'ap_recalculada\':False}\n    classes=[]\n    for cid,name in enumerate(core.NAMES):\n        part=[e for e in events if e[\'classe_id\']==cid];n=sum(e[\'bbox_ref\']!=\'\' for e in part)\n        classes.append({\'classe_id\':cid,\'classe\':name,\'gt_avaliaveis\':n,\'suporte_reduzido\':n<30,\'conf_alertas\':conf,\n            **core.alert_metrics(sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part),\n                                 sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in part),\n                                 sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part)),\n            \'alta_ref\':sum(e[\'prioridade_ref\']==\'alto\' for e in part),\n            \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in part)})\n    return summary,classes\n\ndef evaluate_cache(records,cache,conf,policy,matcher,output=None):\n    if cache.keys()!={r[\'id\'] for r in records}:raise ValueError(\'Cache não corresponde aos IDs da validação.\')\n    images=[];events=[]\n    for r in records:\n        im,ev=analyse(r,cache[r[\'id\']],conf,policy,matcher);images.append(im);events.extend(ev)\n    summary,classes=summarise(images,events,conf)\n    if output is not None:\n        output=Path(output);output.mkdir(parents=True,exist_ok=True)\n        core.save_json(output/\'metricas.json\',summary)\n        core.save_csv(output/\'metricas_por_classe.csv\',classes)\n        core.save_csv(output/\'alertas_por_imagem.csv\',images)\n        core.save_csv(output/\'avaliacao_objetos.csv\',events)\n    return {\'summary\':summary,\'classes\':classes,\'images\':images,\'events\':events}\n\ndef parse_current_events(events):\n    result=[]\n    for r in events:\n        v=dict(r)\n        for key in [\'bbox_ref\',\'bbox_pred\']:v[key]=json.loads(v[key]) if v[key] else None\n        result.append(v)\n    return result\n\ndef check_reproduction(result,original,cohort):\n    old={r[\'id\']:r for r in original[\'images\']};new={r[\'id\']:r for r in result[\'images\']}\n    if old.keys()!=new.keys():raise ValueError(\'Reprodução exige os mesmos IDs.\')\n    rows=[]\n    fields=[\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']\n    for iid in sorted(old):\n        for key in fields:\n            key_new=\'predicoes_ignoradas_conf\' if key==\'predicoes_ignoradas_conf_025\' else key\n            a,b=old[iid][key],new[iid][key_new]\n            rows.append({\'coorte\':cohort,\'id\':iid,\'metrica\':key,\'notebook12\':a,\'notebook14\':b,\'igual\':a==b})\n    old_gt=data.gt_map(original[\'events\']);new_gt=data.gt_map(parse_current_events(result[\'events\']))\n    if old_gt.keys()!=new_gt.keys():raise ValueError(\'Referências GT mudaram na reprodução.\')\n    object_rows=[]\n    for key,a in old_gt.items():\n        b=new_gt[key]\n        fields_gt=[\'evento\',\'prioridade_ref\',\'prioridade_pred\',\'ha_alerta_ref\',\'ha_alerta_pred\']\n        diffs=[field for field in fields_gt if a[field]!=b[field]]\n        object_rows.append({\'coorte\':cohort,\'id\':key[0],\'classe\':a[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                            \'campos_diferentes\':\'|\'.join(diffs),\'igual\':not diffs})\n    return {\'coorte\':cohort,\'checagens_imagem\':len(rows),\'divergencias_imagem\':sum(not r[\'igual\'] for r in rows),\n            \'objetos_GT\':len(object_rows),\'divergencias_objetos\':sum(not r[\'igual\'] for r in object_rows),\n            \'reproduziu\':all(r[\'igual\'] for r in rows+object_rows)},rows,object_rows\n\ndef select_threshold(results,rule,reproduction_ok):\n    assessed=[]\n    for conf in THRESHOLDS:\n        home=results[\'home_val_historica\'][conf][\'summary\'];ext=results[\'openimages_val_nova\'][conf][\'summary\']\n        supported=all(s[\'alertas_por_objeto\'][k] is not None and math.isfinite(s[\'alertas_por_objeto\'][k])\n                      for s in [home,ext] for k in [\'precision\',\'recall\',\'f1\'])\n        if supported:\n            checks={**judge(home,ext,rule),\'metricas_com_suporte\':True}\n        else:\n            # Precisão sem nenhum alerta é indefinida; não vira zero ou aprovação.\n            checks={k:None for k in [\'home_precision_superior\',\'home_recall_preservado\',\'home_altas_preservadas\',\n                                    \'externa_precision_preservada\',\'externa_recall_preservado\',\'externa_altas_preservadas\']}\n            checks.update(elegivel=False,retencao_externa=False,metricas_com_suporte=False)\n        assessed.append({\'conf\':conf,\'home\':home,\'external\':ext,\'checks\':checks,\n                         \'comparavel_historico\':reproduction_ok,\'elegivel\':checks[\'elegivel\'] and reproduction_ok})\n    def ranking(r):\n        h=r[\'home\'][\'alertas_por_objeto\'];e=r[\'external\'][\'alertas_por_objeto\']\n        return (-h[\'precision\'],-h[\'recall\'],r[\'home\'][\'alta_sem_alerta\'],-e[\'f1\'],abs(r[\'conf\']-.25),-r[\'conf\'])\n    good=[r for r in assessed if r[\'elegivel\']]\n    chosen=min(good,key=ranking) if good else None\n    status=\'limiar_candidato_em_validacao\' if chosen else \'nenhum_limiar_elegivel\'\n    if not reproduction_ok:status=\'reproducao_divergente_sem_sugestao_de_limiar\'\n    return {\'status\':status,\'limiar_sugerido\':chosen[\'conf\'] if chosen else None,\n            \'comparabilidade_confirmada\':bool(reproduction_ok),\'assessed\':assessed,\n            \'limiar_galeria\':chosen[\'conf\'] if chosen else .10,\n            \'criterio_galeria_sem_elegivel\':\'0,10 fixado antes da inferência; somente diagnóstico.\',\n            \'modelo_em_uso_continua_baseline\':True,\'substituicao_automatica\':False}\n\ndef effects(results,bundle,cohort,cache):\n    old=data.gt_map(bundle[\'evaluations\'][\'candidato_10\'][cohort][\'events\'])\n    anchor=data.gt_map(parse_current_events(results[cohort][.25][\'events\']))\n    lost={k for k in anchor if old[k][\'ha_alerta_ref\'] and old[k][\'ha_alerta_pred\'] and not anchor[k][\'ha_alerta_pred\']}\n    changed=[];losses=[];priority=[]\n    for key in sorted(lost):\n        a,b=old[key],anchor[key]\n        if b[\'evento\']==\'correspondencia\':\n            priority.append({\'coorte\':cohort,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_candidato10\':a[\'prioridade_pred\'],\'prioridade_B_025\':b[\'prioridade_pred\'],\n                \'confianca_candidato10\':a[\'confianca\'],\'confianca_B_025\':b[\'confianca\'],\n                \'observacao\':\'Objeto ainda associado; alerta pode mudar pela geometria/prioridade.\'})\n    for conf in THRESHOLDS:\n        current=data.gt_map(parse_current_events(results[cohort][conf][\'events\']))\n        for key,b in current.items():\n            a=anchor[key]\n            if a[\'ha_alerta_ref\'] and a[\'ha_alerta_pred\']!=b[\'ha_alerta_pred\']:\n                changed.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                    \'alerta_B_025\':a[\'ha_alerta_pred\'],\'alerta_no_limiar\':b[\'ha_alerta_pred\'],\n                    \'prioridade_ref\':b[\'prioridade_ref\'],\'confianca_associada\':b[\'confianca\']})\n            if key not in lost:continue\n            same=[p for p in cache[key[0]] if p[\'classe_id\']==key[1]]\n            ious=core.iou_matrix([{\'bbox\':list(key[2])}],same)[0]\n            best=max(range(len(same)),key=lambda i:ious[i]) if same else None\n            losses.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_ref\':b[\'prioridade_ref\'],\'recuperado\':b[\'ha_alerta_pred\'],\n                \'evento_B_025\':a[\'evento\'],\'evento_no_limiar\':b[\'evento\'],\n                \'melhor_iou_cache_mesma_classe\':float(ious[best]) if best is not None else None,\n                \'confianca_dessa_proposta\':same[best][\'confianca\'] if best is not None else None,\n                \'limite\':\'Melhor IoU não garante associação nem alerta; coluna recuperado usa avaliação completa.\'})\n    return changed,losses,priority\n\n\ndef validate_predictions(pred,record):\n    """Valida estrutura; preserva coordenadas numéricas como no avaliador 11/12.\n\n    Área zero e limites da imagem são diagnósticos, não filtros do cache.\n    Nenhuma proposta é removida, recortada ou corrigida aqui.\n    """\n    totals={\'predicoes\':len(pred),\'area_zero\':0,\'fora_da_imagem\':0,\n            \'area_zero_na_grade\':0,\'fora_da_imagem_na_grade\':0,\'amostras\':[]}\n    for index,p in enumerate(pred):\n        reason=None\n        if not isinstance(p,dict):\n            raise ValueError(f"Previsão não é um dicionário: {record[\'id\']}; índice={index}; valor={p!r}")\n        cid=p.get(\'classe_id\');box=p.get(\'bbox\');score=p.get(\'confianca\')\n        if type(cid)!=int or not 0<=cid<len(core.NAMES):reason=\'classe_id inválida\'\n        elif p.get(\'classe\')!=core.NAMES[cid]:reason=\'nome da classe incompatível\'\n        elif type(score) not in (int,float) or not math.isfinite(score) or not .001-1e-8<=score<=1:\n            reason=\'confiança não finita ou fora da faixa de captura\'\n        elif not isinstance(box,(list,tuple)) or len(box)!=4:reason=\'bbox deve ter quatro coordenadas\'\n        elif not all(type(v) in (int,float) and math.isfinite(v) for v in box):reason=\'coordenada não numérica ou não finita\'\n        elif box[2]<box[0] or box[3]<box[1]:reason=\'coordenadas xyxy invertidas\'\n        if reason:\n            raise ValueError(f"Previsão incompatível com o cache: {record[\'id\']}; motivo={reason}; "\n                f"índice={index}; classe_id={cid!r}; confiança={score!r}; bbox={box!r}; "\n                f"dimensões=(w={record[\'width\']}, h={record[\'height\']})")\n        zero=box[2]==box[0] or box[3]==box[1]\n        outside=(min(box)<-.01 or box[2]>record[\'width\']+.01 or box[3]>record[\'height\']+.01)\n        totals[\'area_zero\']+=int(zero);totals[\'fora_da_imagem\']+=int(outside)\n        totals[\'area_zero_na_grade\']+=int(zero and score>=min(THRESHOLDS))\n        totals[\'fora_da_imagem_na_grade\']+=int(outside and score>=min(THRESHOLDS))\n        if (zero or outside) and len(totals[\'amostras\'])<5:\n            totals[\'amostras\'].append({\'indice\':index,\'classe\':p[\'classe\'],\'confianca\':score,\n                \'bbox\':list(box),\'area_zero\':zero,\'fora_da_imagem\':outside,\n                \'width\':record[\'width\'],\'height\':record[\'height\']})\n    return totals\n\ndef save_geometry_diagnostics(directory,items):\n    totals={key:sum(item[\'diagnostico\'][key] for item in items) for key in\n            [\'predicoes\',\'area_zero\',\'fora_da_imagem\',\'area_zero_na_grade\',\'fora_da_imagem_na_grade\']}\n    unusual=[item for item in items if item[\'diagnostico\'][\'area_zero\'] or item[\'diagnostico\'][\'fora_da_imagem\']]\n    report={\'correcao\':\'14-cache-validacao-1.0.1\',\'imagens\':len(items),\n        \'imagens_com_geometria_incomum\':len(unusual),\'totais\':totals,\n        \'propostas_removidas\':0,\'coordenadas_alteradas\':False,\n        \'observacao\':\'Geometria incomum preservada no cache e no protocolo; não constitui objeto aprovado. Reprodução em 0,25 continua obrigatória.\',\n        \'casos\':unusual}\n    core.save_json(Path(directory)/\'diagnostico_geometria_previsoes.json\',report)\n    return {k:v for k,v in report.items() if k!=\'casos\'}\n\ndef obtain_cache(model,records,directory,weight_sha,environment,device,predict_fn=None):\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    path=directory/\'predicoes_conf_0p001.jsonl.gz\';meta_path=directory/\'cache_manifesto.json\'\n    request=cache_request(records,weight_sha,environment)\n    if path.is_file() and meta_path.is_file():\n        meta=json.loads(meta_path.read_text())\n        if meta[\'solicitacao\']!=request or meta[\'cache_sha256\']!=core.sha256_file(path):\n            raise RuntimeError(\'Cache existente não corresponde aos pesos/entradas/ambiente. Não será reutilizado.\')\n        cached=read_cache(path,records)\n        diagnostics=[{\'id\':r[\'id\'],\'diagnostico\':validate_predictions(cached[r[\'id\']],r)} for r in records]\n        geometry=save_geometry_diagnostics(directory,diagnostics)\n        print(\'Cache conferido e reutilizado:\',records[0][\'coorte\'],flush=True)\n        return path,{**meta,\'geometria_resumo\':geometry,\'reutilizado_nesta_execucao\':True}\n    predict=predict_fn or core.model_predictions\n    tmp=path.with_name(path.name+\'.tmp\')\n    saturated=0;diagnostics=[]\n    with gzip.open(tmp,\'wt\',encoding=\'utf-8\',compresslevel=4) as f:\n        for i,r in enumerate(records,1):\n            if r[\'role\'] not in {\'home_val\',\'val_nova\'}:raise ValueError(\'Papel proibido na inferência.\')\n            pred=predict(model,r,device)\n            try:diagnostic=validate_predictions(pred,r)\n            except ValueError as exc:\n                core.save_json(directory/\'previsao_rejeitada.json\',{\'id\':r[\'id\'],\'mensagem\':str(exc),\n                    \'width\':r[\'width\'],\'height\':r[\'height\'],\'propostas\':pred})\n                raise\n            diagnostics.append({\'id\':r[\'id\'],\'diagnostico\':diagnostic})\n            if len(pred)==PREDICT_ARGS[\'max_det\']:saturated+=1\n            f.write(json.dumps({\'id\':r[\'id\'],\'imagem\':r[\'name\'],\'image_sha256\':r[\'sha256\'],\n                                \'label_sha256\':r[\'label_sha256\'],\'predicoes\':pred},ensure_ascii=False)+\'\\n\')\n            if i%100==0 or i==len(records):print(f"Checkpoint / {r[\'coorte\']}: {i}/{len(records)}",flush=True)\n    tmp.replace(path)\n    geometry=save_geometry_diagnostics(directory,diagnostics)\n    meta={\'solicitacao\':request,\'cache_sha256\':core.sha256_file(path),\'concluido\':True,\n          \'validacao_previsoes\':\'14-cache-validacao-1.0.1\',\'geometria_resumo\':geometry,\n          \'imagens_atingiram_max_det\':saturated,\'limite\':\'Propostas pós-NMS em confiança 0,001; max_det=300, sem acesso aos logits brutos.\'}\n    core.save_json(meta_path,meta)\n    read_cache(path,records)\n    print(\'Diagnóstico geométrico do cache:\',geometry[\'totais\'],flush=True)\n    return path,{**meta,\'reutilizado_nesta_execucao\':False}\n', 'diag11_core.py': '"""Núcleo do diagnóstico 11. Não contém treinamento nem seleção de pesos."""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image\n\nNAMES = ["bed", "sofa", "chair", "table", "lamp", "tv", "laptop",\n         "wardrobe", "window", "door", "potted plant", "photo frame"]\nALL_KNOWN = (1 << len(NAMES)) - 1\nALERT_IDS = {0, 1, 2, 3, 7, 10}\nALERT_MASK = sum(1 << c for c in ALERT_IDS)\nLEVEL = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef canonical(x):\n    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(2**20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef json_clean(value):\n    if isinstance(value, dict):\n        return {str(k): json_clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_clean(v) for v in value]\n    if isinstance(value, np.generic):\n        return json_clean(value.item())\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(json_clean(value), ensure_ascii=False, indent=2,\n                              allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef save_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    rows = list(rows)\n    if fields is None:\n        fields = list(dict.fromkeys(k for row in rows for k in row))\n    with path.open("w", newline="", encoding="utf-8-sig") as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in name:\n        raise ValueError(f"Membro inseguro no ZIP: {name}")\n    return p\n\n\ndef parse_yolo(raw, width, height, names=NAMES, source="label"):\n    objects = []\n    for line_no, line in enumerate(raw.decode("utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        parts = line.split()\n        if len(parts) != 5:\n            raise ValueError(f"{source}:{line_no}: esperados cinco campos.")\n        vals = [float(s) for s in parts]\n        if not all(math.isfinite(v) for v in vals):\n            raise ValueError(f"{source}:{line_no}: valor não finito.")\n        c, x, y, w, h = vals\n        if int(c) != c or not 0 <= c < len(names) or min(w, h) <= 0:\n            raise ValueError(f"{source}:{line_no}: classe ou dimensão inválida.")\n        box = [x-w/2, y-h/2, x+w/2, y+h/2]\n        if min(box) < -0.002 or max(box) > 1.002:\n            raise ValueError(f"{source}:{line_no}: caixa fora da imagem.")\n        box = [max(0, min(1, v)) for v in box]\n        objects.append({"classe_id": int(c), "classe": names[int(c)],\n                        "bbox": [box[0]*width, box[1]*height,\n                                 box[2]*width, box[3]*height],\n                        "ocluido": None, "truncado": None})\n    return objects\n\n\ndef hierarchy_mapping(cfg, class_rows, hierarchy):\n    if cfg["nomes"] != NAMES or len(cfg["mids_raizes"]) != len(NAMES):\n        raise ValueError("Classes da preparação incompatíveis com o projeto.")\n    descriptions = {r[0]: r[1] for r in class_rows if len(r) >= 2}\n    roots = cfg["mids_raizes"]\n    if any(mid not in descriptions for mid in roots):\n        raise ValueError("A descrição oficial não contém todas as raízes.")\n    descendants, ancestors = defaultdict(set), defaultdict(set)\n\n    def walk(node, parents=()):\n        mid = node["LabelName"]\n        ancestors[mid].update(parents)\n        descendants[mid].add(mid)\n        for parent in parents:\n            descendants[parent].add(mid)\n        for child in node.get("Subcategory", []):\n            walk(child, parents + (mid,))\n\n    walk(hierarchy)\n    excluded_names = set(cfg.get("ambiguas_excluidas", []))\n    excluded = {mid for mid, name in descriptions.items() if name in excluded_names}\n    candidates = defaultdict(set)\n    root_desc = {}\n    for cid, mid in enumerate(roots):\n        if mid not in descendants:\n            raise ValueError(f"Raiz ausente da hierarquia: {mid}")\n        root_desc[cid] = set(descendants[mid])\n        for sub in root_desc[cid]:\n            candidates[sub].add(cid)\n    ambiguous = excluded | {mid for mid, ids in candidates.items() if len(ids) > 1}\n    mapping = {mid: next(iter(ids)) for mid, ids in candidates.items()\n               if mid not in ambiguous}\n    return {"roots": roots, "descendants": root_desc, "ancestors": ancestors,\n            "mapping": mapping, "ambiguous": ambiguous,\n            "descriptions": descriptions,\n            "relevant_labels": set().union(*root_desc.values(),\n                                            *(ancestors[r] for r in roots))}\n\n\ndef prepare_oi_annotations(box_rows, label_rows, info):\n    """Cobertura conservadora: positivo na raiz ou negativo na raiz/ancestral.\n\n    Uma positiva apenas numa subclasse não prova cobertura exaustiva da classe\n    agregada. Grupos, representações, atributos desconhecidos ou caixas inválidas\n    tornam essa classe/imagem não avaliável, em vez de criarem falsos negativos.\n    """\n    labels = defaultdict(lambda: {"positive": set(), "negative": set()})\n    for row in label_rows:\n        if row.get("Source") not in {"verification", "crowdsource-verification"}:\n            continue\n        mid, confidence = row["LabelName"], float(row["Confidence"])\n        if mid not in info["relevant_labels"]:\n            continue\n        if confidence in (0, 1):\n            key = "positive" if confidence == 1 else "negative"\n            labels[row["ImageID"]][key].add(mid)\n    boxes, blocked = defaultdict(list), defaultdict(lambda: defaultdict(set))\n    audited = Counter()\n    for row in box_rows:\n        mid, iid = row["LabelName"], row["ImageID"]\n        affected = [cid for cid, mids in info["descendants"].items() if mid in mids]\n        if not affected:\n            continue\n        if mid in info["ambiguous"]:\n            for cid in affected:\n                blocked[iid][cid].add("subclasse_ambigua_excluida")\n            audited["caixas_subclasse_ambigua"] += 1\n            continue\n        cid = info["mapping"].get(mid)\n        if cid is None:\n            continue\n        flags = {k: int(float(row.get(k, -1))) for k in\n                 ["IsGroupOf", "IsDepiction", "IsOccluded", "IsTruncated"]}\n        reasons = []\n        if row.get("Source") not in {"xclick", "activemil"}:\n            reasons.append("origem_caixa_nao_reconhecida")\n        if flags["IsGroupOf"] != 0:\n            reasons.append("grupo_ou_atributo_desconhecido")\n        if flags["IsDepiction"] != 0:\n            reasons.append("representacao_ou_atributo_desconhecido")\n        xyxy = [float(row[k]) for k in ["XMin", "YMin", "XMax", "YMax"]]\n        if (not all(math.isfinite(v) for v in xyxy) or min(xyxy) < 0\n                or max(xyxy) > 1 or xyxy[2] <= xyxy[0] or xyxy[3] <= xyxy[1]):\n            reasons.append("coordenadas_invalidas")\n        if reasons:\n            blocked[iid][cid].update(reasons)\n            audited.update(reasons)\n            continue\n        boxes[iid].append({"classe_id": cid, "classe": NAMES[cid],\n                           "bbox_norm": xyxy, "mid_original": mid,\n                           "ocluido": flags["IsOccluded"] == 1 if flags["IsOccluded"] >= 0 else None,\n                           "truncado": flags["IsTruncated"] == 1 if flags["IsTruncated"] >= 0 else None})\n    records = {}\n    for iid in sorted(set(labels) | set(boxes)):\n        gt = []\n        seen = set()\n        for obj in boxes[iid]:\n            key = (obj["classe_id"], *(round(x, 8) for x in obj["bbox_norm"]))\n            if key not in seen:\n                seen.add(key)\n                gt.append(obj)\n        known, positive, negative, reasons = 0, 0, 0, {}\n        present = {o["classe_id"] for o in gt}\n        for cid, root in enumerate(info["roots"]):\n            pos = root in labels[iid]["positive"]\n            neg = bool(({root} | info["ancestors"][root]) & labels[iid]["negative"])\n            has_positive_label = bool(info["descendants"][cid] & labels[iid]["positive"])\n            reason = set(blocked[iid].get(cid, set()))\n            if neg and (cid in present or has_positive_label):\n                reason.add("contradicao_positivo_negativo")\n            if pos and cid not in present:\n                reason.add("positivo_sem_caixa_utilizavel")\n            if reason:\n                reasons[str(cid)] = sorted(reason)\n                continue\n            if neg:\n                known |= 1 << cid\n                negative |= 1 << cid\n            elif pos and cid in present:\n                known |= 1 << cid\n                positive |= 1 << cid\n        if positive or known & ALERT_MASK == ALERT_MASK:\n            records[iid] = {"id": iid, "source_split": "validation",\n                            "known_mask": known, "positive_mask": positive,\n                            "negative_mask": negative, "gt_normalizado": gt,\n                            "classes_positivas_avaliaveis": sorted(c for c in range(12)\n                                                                 if positive & (1 << c)),\n                            "restricoes_cobertura": reasons}\n    return records, dict(audited)\n\n\ndef iou_matrix(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)))\n    a = np.asarray([o["bbox"] for o in gt], dtype=float)\n    b = np.asarray([o["bbox"] for o in pred], dtype=float)\n    lt = np.maximum(a[:, None, :2], b[None, :, :2])\n    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    intersection = np.maximum(rb-lt, 0).prod(axis=2)\n    aa = np.maximum(a[:, 2:]-a[:, :2], 0).prod(axis=1)\n    ba = np.maximum(b[:, 2:]-b[:, :2], 0).prod(axis=1)\n    return intersection / np.maximum(aa[:, None] + ba[None, :] - intersection, 1e-12)\n\n\ndef ap_matches(gt, pred, thresholds=np.linspace(.5, .95, 10)):\n    correct = np.zeros((len(pred), len(thresholds)), dtype=bool)\n    matrix = iou_matrix(gt, pred)\n    order = sorted(range(len(pred)), key=lambda pi: (-pred[pi]["confianca"], pi))\n    for ti, threshold in enumerate(thresholds):\n        used = set()\n        for pi in order:\n            options = [gi for gi, g in enumerate(gt) if gi not in used\n                       and g["classe_id"] == pred[pi]["classe_id"]\n                       and matrix[gi, pi] >= threshold]\n            if options:\n                gi = max(options, key=lambda k: (matrix[k, pi], -k))\n                used.add(gi)\n                correct[pi, ti] = True\n    return correct\n\n\ndef ratio(n, d):\n    return n/d if d else None\n\n\ndef alert_metrics(tp, fn, fp):\n    return {"tp": int(tp), "fn": int(fn), "fp": int(fp),\n            "precision": ratio(tp, tp+fp), "recall": ratio(tp, tp+fn),\n            "f1": ratio(2*tp, 2*tp+fp+fn)}\n\n\ndef alert_on(obj):\n    return bool(obj and LEVEL[obj["prioridade"]] >= LEVEL["medio"])\n\n\ndef with_policy(objects, width, height, policy):\n    return [{**o, **policy.avaliar_bbox(o["classe"], o["bbox"], width, height,\n                                       versao="v2")} for o in objects]\n\n\ndef record_event(record, model_name, event, g=None, p=None, iou=None, all_pred=()):\n    gbox = g["bbox"] if g else None\n    pbox = p["bbox"] if p else None\n    area = ((gbox[2]-gbox[0])*(gbox[3]-gbox[1]) /\n            (record["width"]*record["height"])) if gbox else None\n    side_at_640 = (min(gbox[2]-gbox[0], gbox[3]-gbox[1]) *\n                   640/max(record["width"], record["height"])) if gbox else None\n    flags = []\n    near_best = None\n    if event == "objeto_nao_detectado":\n        if side_at_640 < 32:\n            flags.append("objeto_pequeno_na_entrada_640")\n        if g.get("ocluido") is True:\n            flags.append("oclusao_anotada")\n        if g.get("truncado") is True:\n            flags.append("truncamento_anotado")\n        overlaps = iou_matrix([g], all_pred)[0] if all_pred else []\n        same = [i for i, o in enumerate(all_pred) if o["classe_id"] == g["classe_id"]]\n        if same:\n            best = max(same, key=lambda i: overlaps[i])\n            near_best = float(overlaps[best])\n            if overlaps[best] >= .5 and all_pred[best]["confianca"] < .25:\n                flags.append("deteccao_abaixo_da_confianca_025")\n            elif .1 <= overlaps[best] < .5:\n                flags.append("caixa_mesma_classe_iou_010_a_050")\n        if any(overlaps[i] >= .5 and o["classe_id"] != g["classe_id"]\n               and o["confianca"] >= .25 for i, o in enumerate(all_pred)):\n            flags.append("sobreposicao_com_previsao_de_outra_classe")\n        if not flags:\n            flags.append("omissao_sem_indicio_automatico")\n    if event == "previsao_nao_associada":\n        flags.append("previsao_sem_correspondencia_na_classe_avaliavel")\n    return {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n            "modelo": model_name, "evento": event,\n            "classe_id": g["classe_id"] if g else p["classe_id"],\n            "classe": g["classe"] if g else p["classe"],\n            "bbox_ref": json.dumps(gbox) if g else "",\n            "bbox_pred": json.dumps(pbox) if p else "",\n            "iou": iou, "confianca": p["confianca"] if p else None,\n            "prioridade_ref": g["prioridade"] if g else "sem_prioridade",\n            "prioridade_pred": p["prioridade"] if p else "sem_prioridade",\n            "ha_alerta_ref": alert_on(g), "ha_alerta_pred": alert_on(p),\n            "area_relativa_gt": area, "menor_lado_gt_em_640": side_at_640,\n            "oclusao_anotada": g.get("ocluido") if g else None,\n            "truncamento_anotado": g.get("truncado") if g else None,\n            "melhor_iou_mesma_classe": near_best,\n            "indicios": "|".join(flags)}\n\n\ndef analyse_prediction(record, pred_low, model_name, policy, matcher):\n    if record.get("role") not in {"home_val", "val_nova"}:\n        raise ValueError("Somente validação pode entrar na inferência do notebook 11.")\n    known = int(record["known_mask"])\n    gt = with_policy([g for g in record["gt"] if known & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    pred_low_known = [p for p in pred_low if known & (1 << p["classe_id"])]\n    pred_all = with_policy([p for p in pred_low if p["confianca"] >= .25],\n                           record["width"], record["height"], policy)\n    pred = [p for p in pred_all if known & (1 << p["classe_id"])]\n    pairs, missing, extras = matcher(gt, pred)\n    events = []\n    for gi, pi, iou in pairs:\n        events.append(record_event(record, model_name, "correspondencia",\n                                   gt[gi], pred[pi], float(iou)))\n    for gi in missing:\n        events.append(record_event(record, model_name, "objeto_nao_detectado",\n                                   g=gt[gi], all_pred=pred_low_known))\n    for pi in extras:\n        events.append(record_event(record, model_name, "previsao_nao_associada", p=pred[pi]))\n    tp = sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    fn = sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in events)\n    fp = sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    high_n = sum(g["prioridade"] == "alto" for g in gt)\n    high_missing = sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in events)\n    messages = [policy.gerar_mensagem(p, "medio") for p in pred_all if alert_on(p)]\n    if any(not isinstance(m, str) or not m.strip() for m in messages):\n        raise RuntimeError("A política produziu alerta sem texto.")\n    image = {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n             "modelo": model_name, "tp_alerta": tp, "fn_alerta": fn, "fp_alerta": fp,\n             "alta_ref": high_n, "alta_sem_alerta": high_missing,\n             "tp_detector": len(pairs), "fn_detector": len(missing), "fp_detector": len(extras),\n             "gt_avaliaveis": len(gt), "gt_ignorados": len(record["gt"])-len(gt),\n             "predicoes_ignoradas_conf_025": len(pred_all)-len(pred),\n             "cobertura_completa_alertas": known & ALERT_MASK == ALERT_MASK,\n             "ha_alerta_ref": any(alert_on(g) for g in gt),\n             "ha_alerta_pred": any(alert_on(p) for p in pred),\n             "mensagem_alerta": "\\n".join(messages)}\n    stats = (ap_matches(gt, pred_low_known),\n             np.array([p["confianca"] for p in pred_low_known], dtype=float),\n             np.array([p["classe_id"] for p in pred_low_known], dtype=int),\n             np.array([g["classe_id"] for g in gt], dtype=int))\n    return image, events, stats\n\n\ndef summarise(images, events, stats, ap_function=None):\n    if not images:\n        raise ValueError("Coorte de avaliação vazia.")\n    total = lambda k: sum(int(r[k]) for r in images)\n    output = {"imagens": len(images), "conf_alertas": .25,\n              "alertas_por_objeto": alert_metrics(total("tp_alerta"), total("fn_alerta"), total("fp_alerta")),\n              "alta_ref": total("alta_ref"), "alta_sem_alerta": total("alta_sem_alerta"),\n              "taxa_alta_sem_alerta": ratio(total("alta_sem_alerta"), total("alta_ref")),\n              "detector_conf_025": alert_metrics(total("tp_detector"), total("fn_detector"), total("fp_detector")),\n              "gt_ignorados_sem_cobertura": total("gt_ignorados"),\n              "predicoes_ignoradas_conf_025": total("predicoes_ignoradas_conf_025")}\n    complete = [r for r in images if r["cobertura_completa_alertas"]]\n    output["imagens_com_cobertura_completa_alertas"] = len(complete)\n    output["alertas_por_imagem_cobertura_completa"] = alert_metrics(\n        sum(r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete),\n        sum(r["ha_alerta_ref"] and not r["ha_alerta_pred"] for r in complete),\n        sum(not r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete))\n    correct = np.concatenate([s[0] for s in stats], axis=0)\n    scores = np.concatenate([s[1] for s in stats])\n    pred_cls = np.concatenate([s[2] for s in stats])\n    target_cls = np.concatenate([s[3] for s in stats])\n    ap = {}\n    if ap_function is not None and len(target_cls) and len(pred_cls):\n        result = ap_function(correct, scores, pred_cls, target_cls, plot=False,\n                             names=dict(enumerate(NAMES)))\n        values, ids = np.asarray(result[5]), np.asarray(result[6], dtype=int)\n        if values.shape != (len(ids), 10):\n            raise RuntimeError("Retorno de AP incompatível com a versão fixada.")\n        ap = {int(c): a for c, a in zip(ids, values)}\n    elif ap_function is not None and len(target_cls):\n        ap = {int(c): np.zeros(10) for c in np.unique(target_cls)}\n    output["map50_recorte"] = float(np.mean([a[0] for a in ap.values()])) if ap else None\n    output["map50_95_recorte"] = float(np.mean(list(ap.values()))) if ap else None\n    class_rows = []\n    for cid, name in enumerate(NAMES):\n        part = [e for e in events if e["classe_id"] == cid]\n        a = ap.get(cid)\n        n_gt = int((target_cls == cid).sum())\n        class_rows.append({"classe_id": cid, "classe": name, "gt_avaliaveis": n_gt,\n                           "ap50": float(a[0]) if a is not None else None,\n                           "ap50_95": float(a.mean()) if a is not None else None,\n                           "suporte_reduzido": n_gt < 30,\n                           **alert_metrics(sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part),\n                                           sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                           sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part)),\n                           "alta_ref": sum(e["prioridade_ref"] == "alto" for e in part),\n                           "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part)})\n    return output, class_rows\n\n\ndef model_predictions(model, record, device):\n    if record["role"] not in {"home_val", "val_nova"}:\n        raise ValueError("Tentativa de inferência em reserva ou conjunto não autorizado.")\n    result = model.predict(str(record["image"]), conf=.001, imgsz=640, rect=True,\n                           iou=.70, max_det=300, device=device, half=False,\n                           verbose=False, save=False)[0]\n    if Path(result.path).resolve() != Path(record["image"]).resolve():\n        raise RuntimeError("A inferência retornou uma imagem diferente.")\n    if result.boxes is None:\n        return []\n    return [{"classe_id": int(c), "classe": NAMES[int(c)],\n             "bbox": [float(v) for v in box], "confianca": float(score)}\n            for c, box, score in zip(result.boxes.cls.cpu().numpy(),\n                                      result.boxes.xyxy.cpu().numpy(),\n                                      result.boxes.conf.cpu().numpy())]\n\n\ndef evaluate_model(model, records, model_name, output_dir, policy, matcher, device,\n                   ap_function):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    images, events, stats = [], [], []\n    cache_file = output_dir / "predicoes_conf_0001.jsonl.gz"\n    with gzip.open(cache_file, "wt", encoding="utf-8") as cache:\n        for i, record in enumerate(records, 1):\n            pred = model_predictions(model, record, device)\n            cache.write(json.dumps({"id": record["id"], "imagem": record["name"],\n                                    "predicoes": pred}, ensure_ascii=False) + "\\n")\n            im, ev, st = analyse_prediction(record, pred, model_name, policy, matcher)\n            images.append(im)\n            events.extend(ev)\n            stats.append(st)\n            if i % 100 == 0 or i == len(records):\n                print(f"{record[\'coorte\']} / {model_name}: {i}/{len(records)}", flush=True)\n    summary, classes = summarise(images, events, stats, ap_function)\n    summary.update(modelo=model_name, coorte=records[0]["coorte"],\n                   protocolo="12 classes; AP por confiança; alertas por IoU; cobertura conservadora",\n                   nao_e_openimages_challenge=True, selecao_automatica_de_modelo=False)\n    save_json(output_dir / "metricas.json", summary)\n    save_csv(output_dir / "metricas_por_classe.csv", classes)\n    save_csv(output_dir / "alertas_por_imagem.csv", images)\n    save_csv(output_dir / "avaliacao_objetos.csv", events)\n    (output_dir / "alertas_texto.txt").write_text(\n        "\\n\\n".join(r["imagem"]+"\\n"+r["mensagem_alerta"] for r in images if r["mensagem_alerta"]),\n        encoding="utf-8")\n    return {"summary": summary, "classes": classes, "images": images, "events": events,\n            "cache_file": cache_file}\n', 'diag11_data.py': '"""Entradas, auditoria documental e nova validação, sem consulta a previsões."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport time\nimport urllib.request\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.fft import dctn\n\nfrom diag11_core import (ALL_KNOWN, ALERT_MASK, NAMES, canonical, parse_yolo,\n                         safe_member, save_csv, save_json, sha256_file)\n\nSOURCE_URLS = {\n    "classes": "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv",\n    "hierarquia": "https://storage.googleapis.com/openimages/2018_04/bbox_labels_600_hierarchy.json",\n    "caixas": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",\n    "rotulos": "https://storage.googleapis.com/openimages/v5/validation-annotations-human-imagelabels-boxable.csv",\n    "metadata": "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv",\n}\n\n\ndef download(url, path, retries=3):\n    """Download atômico, com recibo local e validação de tamanho quando informado."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    receipt = path.with_name(path.name + ".download.json")\n    if path.is_file() and receipt.is_file():\n        prior = json.loads(receipt.read_text())\n        if prior.get("url") == url and prior.get("sha256") == sha256_file(path):\n            return path\n    last = None\n    for attempt in range(retries):\n        tmp = path.with_name(path.name + ".tmp")\n        try:\n            req = urllib.request.Request(url, headers={"User-Agent": "TCC-Diagnostico-11/1.0"})\n            with urllib.request.urlopen(req, timeout=45) as response, tmp.open("wb") as out:\n                expected = response.headers.get("Content-Length")\n                for block in iter(lambda: response.read(2**20), b""):\n                    out.write(block)\n            if expected and tmp.stat().st_size != int(expected):\n                raise IOError("Tamanho recebido difere do Content-Length.")\n            if not tmp.stat().st_size:\n                raise IOError("Arquivo recebido vazio.")\n            tmp.replace(path)\n            save_json(receipt, {"url": url, "sha256": sha256_file(path),\n                                "bytes": path.stat().st_size})\n            return path\n        except Exception as exc:\n            last = type(exc).__name__ + ": " + str(exc)[:180]\n            tmp.unlink(missing_ok=True)\n            if attempt + 1 < retries:\n                time.sleep(min(2**attempt, 4))\n    raise IOError(f"Download não concluído: {url}; {last}")\n\n\ndef csv_rows(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as f:\n        yield from csv.DictReader(f)\n\n\ndef load_history(csv_path, json_path, expected_hash):\n    split = json.loads(Path(json_path).read_text())\n    stated = split.pop("divisao_sha256")\n    computed = hashlib.sha256(canonical(split).encode()).hexdigest()\n    if stated != expected_hash or computed != stated:\n        raise RuntimeError("Divisão histórica não corresponde à execução 10.")\n    rows = list(csv_rows(csv_path))\n    if len(rows) != 100000 or len({r["id"] for r in rows}) != 100000:\n        raise RuntimeError("O manifesto histórico precisa de 100.000 IDs distintos.")\n    if Counter(r["split"] for r in rows) != {"train": 80000, "test": 20000}:\n        raise RuntimeError("Divisão histórica diferente de 80/20.")\n    for label, key in [("train", "ids_treino"), ("test", "ids_teste")]:\n        if {r["id"] for r in rows if r["split"] == label} != set(split[key]):\n            raise RuntimeError("CSV e JSON históricos divergem.")\n    hashes = {r["imagem_sha256"] for r in rows}\n    if len(hashes) != len(rows):\n        raise RuntimeError("O histórico contém imagens repetidas em bytes.")\n    return {"ids": {r["id"] for r in rows}, "hashes": hashes, "rows": rows,\n            "split_hash": stated}\n\n\ndef phash(path):\n    with Image.open(path) as image:\n        small = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS),\n                           dtype=float)\n    low = dctn(small, norm="ortho")[:8, :8]\n    bits = low > np.median(low.flatten()[1:])\n    bits[0, 0] = False\n    return int.from_bytes(np.packbits(bits.flatten()).tobytes(), "big")\n\n\nclass NearIndex:\n    """pHash de 64 bits, distância <=4; sinal de semelhança, não prova semântica."""\n    def __init__(self):\n        self.tables = [defaultdict(list) for _ in range(8)]\n\n    def add(self, value, name):\n        for band in range(8):\n            self.tables[band][(value >> (8*band)) & 255].append((value, name))\n\n    def find(self, value, max_distance=4):\n        checked = set()\n        for band in range(8):\n            for prior, name in self.tables[band].get((value >> (8*band)) & 255, []):\n                if (prior, name) in checked:\n                    continue\n                checked.add((prior, name))\n                if (prior ^ value).bit_count() <= max_distance:\n                    return name\n        return None\n\n\ndef extract_home(zip_path, destination, expected_zip, expected_test, expected_policy):\n    if sha256_file(zip_path) != expected_zip:\n        raise RuntimeError("ZIP Home diferente da entrada preservada.")\n    destination = Path(destination)\n    records, hashes, near = [], set(), NearIndex()\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Há membros repetidos no ZIP Home.")\n        manifest = json.loads(z.read("verificacao_preparacao.json"))\n        if (manifest["test"]["sha256"] != expected_test or\n                manifest["test"]["imagens"] != 403 or\n                {int(k): v for k, v in manifest["classes"].items()} != dict(enumerate(NAMES))):\n            raise RuntimeError("Manifesto Home incompatível.")\n        root = manifest["dataset_dir"]\n        items = {r["arquivo"]: r for r in manifest["arquivos"]}\n        if len(items) != len(manifest["arquivos"]):\n            raise RuntimeError("Manifesto Home com membros repetidos.")\n        for name in z.namelist():\n            safe_member(name)\n            if name.startswith("codigo/") and name.endswith(".py"):\n                raw = z.read(name)\n                if name in items and hashlib.sha256(raw).hexdigest() != items[name]["sha256"]:\n                    raise RuntimeError("Código Home diferente do manifesto.")\n                out = destination / name\n                out.parent.mkdir(parents=True, exist_ok=True)\n                out.write_bytes(raw)\n        for name, item in sorted(items.items()):\n            is_image = (name.startswith(f"{root}/images/") and\n                        Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})\n            if is_image:\n                hashes.add(item["sha256"])\n            role = None\n            if name.startswith(f"{root}/images/train/raw_train/"):\n                role = "home_train"\n            elif name.startswith(f"{root}/images/val/raw_train/"):\n                role = "home_val"\n            if not is_image or role is None:\n                continue\n            raw = z.read(name)\n            if hashlib.sha256(raw).hexdigest() != item["sha256"]:\n                raise RuntimeError(f"Imagem Home alterada: {name}")\n            image = destination / name\n            image.parent.mkdir(parents=True, exist_ok=True)\n            image.write_bytes(raw)\n            label_name = name.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"\n            if label_name not in items:\n                raise RuntimeError("Label Home ausente do manifesto.")\n            label_raw = z.read(label_name)\n            if hashlib.sha256(label_raw).hexdigest() != items[label_name]["sha256"]:\n                raise RuntimeError("Label Home alterado.")\n            with Image.open(image) as im:\n                width, height = im.size\n                im.verify()\n            gt = parse_yolo(label_raw, width, height, source=label_name)\n            record = {"id": "home:" + name, "name": name.split("/raw_train/")[-1],\n                      "image": image, "role": role, "coorte": "home_val_historica",\n                      "width": width, "height": height, "known_mask": ALL_KNOWN,\n                      "sha256": item["sha256"], "label_sha256": items[label_name]["sha256"],\n                      "gt": gt}\n            record["phash"] = phash(image)\n            near.add(record["phash"], name)\n            records.append(record)\n    counts = Counter(r["role"] for r in records)\n    if counts != {"home_train": 1824, "home_val": 456}:\n        raise RuntimeError(f"Home inesperado: {dict(counts)}")\n    train_hash = {r["sha256"] for r in records if r["role"] == "home_train"}\n    val_hash = {r["sha256"] for r in records if r["role"] == "home_val"}\n    if train_hash & val_hash:\n        raise RuntimeError("Home TRAIN e VAL atuais compartilham imagens em bytes.")\n    policy_file = destination / "codigo/src/politica_alertas.py"\n    if not policy_file.is_file() or sha256_file(policy_file) != expected_policy:\n        raise RuntimeError("Política V2 diferente da versão preservada.")\n    audit = {"home_train_atual": 1824, "home_val_atual": 456,\n             "test_home_preservado": 403, "test_extraido": False,\n             "intersecao_bytes_home_train_val_atual": 0,\n             "divergencia_documental_1822_1824": "Não reconciliada; o pacote atual tem 1824.",\n             "separacao_atual_nao_comprova_historico_baseline": True}\n    return records, hashes, near, audit\n\n\ndef audit_baseline_metadata(checkpoint, expected_hash, home_records, evidence=None):\n    if sha256_file(checkpoint) != expected_hash:\n        raise RuntimeError("Peso baseline diferente da referência.")\n    # Só o checkpoint autenticado pelo SHA esperado é desserializado.\n    import torch\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    args = obj.get("train_args", {})\n    fields = ["data", "epochs", "batch", "imgsz", "seed", "optimizer", "lr0",\n              "project", "name", "fraction"]\n    audit = {"modelo_sha256": expected_hash, "data_checkpoint": str(obj.get("date")),\n             "train_args_preservados": {k: args[k] for k in fields if k in args},\n             "estado_proveniencia_original": "nao_determinado",\n             "home_val_usada_no_treino_original": None,\n             "motivo": "Caminhos e argumentos não contêm a lista de imagens vistas no treino original."}\n    del obj\n    if evidence is not None:\n        required = {"modelo_sha256", "train_image_sha256", "val_image_sha256"}\n        if not required <= evidence.keys() or evidence["modelo_sha256"] != expected_hash:\n            audit["evidencia_original"] = "incompativel_ou_sem_vinculo_ao_peso"\n        else:\n            train = set(evidence["train_image_sha256"])\n            val = set(evidence["val_image_sha256"])\n            valid_sha = lambda x: isinstance(x, str) and len(x) == 64 and all(c in "0123456789abcdef" for c in x)\n            if not train or any(not valid_sha(h) for h in train | val):\n                raise ValueError("Manifesto original precisa de hashes SHA-256 válidos.")\n            current = {r["sha256"] for r in home_records if r["role"] == "home_val"}\n            overlap = train & current\n            complete = evidence.get("manifesto_treino_completo") is True\n            audit.update(estado_proveniencia_original="manifesto_documental_analisado",\n                         evidencia_original="documento_vinculado_ao_sha_do_modelo",\n                         home_val_usada_no_treino_original=True if overlap else (False if complete else None),\n                         manifesto_declarado_completo=complete,\n                         imagens_home_val_no_manifesto_train_original=len(overlap),\n                         imagens_home_val_no_manifesto_val_original=len(current & val),\n                         motivo="Conclusão limitada à completude e correção do manifesto apresentado.")\n    return audit\n\n\ndef stable_order(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef balanced_ids(records, n, seed=42, per_class=45):\n    """Diversidade por classe e preenchimento determinístico, sem usar previsões."""\n    if len(records) < n:\n        raise ValueError(f"Há somente {len(records)} imagens elegíveis; exigidas {n}.")\n    groups = {c: sorted((iid for iid, r in records.items() if\n                         r["positive_mask"] & (1 << c)), key=lambda x: stable_order(x, seed))\n              for c in range(12)}\n    order = sorted(groups, key=lambda c: (len(groups[c]), c))\n    chosen, counts = set(), Counter()\n    # Diversificar exemplos negativos quando as seis classes têm ausência verificada.\n    negatives = sorted((iid for iid, r in records.items() if\n                        r["known_mask"] & ALERT_MASK == ALERT_MASK and\n                        not r["positive_mask"] & ALERT_MASK), key=lambda x: stable_order(x, seed))\n    for iid in negatives[:min(len(negatives), n//10)]:\n        chosen.add(iid)\n        counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n    for target in range(1, per_class+1):\n        for cid in order:\n            if len(chosen) >= n:\n                break\n            if counts[cid] >= target:\n                continue\n            for iid in groups[cid]:\n                if iid not in chosen:\n                    chosen.add(iid)\n                    counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n                    break\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return sorted(chosen, key=lambda x: stable_order(x, seed))\n\n\ndef split_pool(records, n_reserve, seed=42):\n    """Estratos pela classe positiva mais rara; a reserva não entra na inferência."""\n    frequencies = Counter(c for r in records.values() for c in range(12)\n                          if r["positive_mask"] & (1 << c))\n    groups = defaultdict(list)\n    for iid, record in records.items():\n        classes = record["classes_positivas_avaliaveis"]\n        primary = min(classes, key=lambda c: (frequencies[c], c)) if classes else -1\n        groups[primary].append(iid)\n    total = len(records)\n    if not 0 < n_reserve < total:\n        raise ValueError("A reserva precisa ser menor que o conjunto novo.")\n    quotas = {c: len(ids)*n_reserve//total for c, ids in groups.items()}\n    order = sorted(groups, key=lambda c: (-(len(groups[c])*n_reserve % total), c))\n    for cid in order[:n_reserve-sum(quotas.values())]:\n        quotas[cid] += 1\n    selected = set()\n    for cid, ids in groups.items():\n        selected.update(sorted(ids, key=lambda x: stable_order(x, seed+1))[:quotas[cid]])\n    if len(selected) != n_reserve:\n        raise RuntimeError("A divisão não produziu a reserva planejada.")\n    return {iid: ("reserva_futura" if iid in selected else "val_nova") for iid in records}\n\n\ndef filter_metadata(records, metadata_path, prior_ids):\n    metadata, exclusions = {}, Counter()\n    for row in csv_rows(metadata_path):\n        iid = row["ImageID"]\n        if iid not in records:\n            continue\n        if row["Subset"] != "validation":\n            raise ValueError("A metadata contém uma origem diferente de validation.")\n        if iid in prior_ids:\n            exclusions["id_presente_nos_100k"] += 1\n            continue\n        try:\n            rotation = float(row["Rotation"])\n        except (TypeError, ValueError):\n            rotation = float("nan")\n        # Evita suposições sobre transformação das caixas e orientação do JPEG.\n        if rotation != 0:\n            exclusions["rotacao_nao_zero_ou_desconhecida"] += 1\n            continue\n        metadata[iid] = {k: row.get(k, "") for k in\n                         ["ImageID", "OriginalLandingURL", "License", "Author", "Rotation"]}\n    result = {iid: {**r, "creditos": metadata[iid]} for iid, r in records.items()\n              if iid in metadata}\n    return result, dict(exclusions)\n\n\ndef download_images(candidates, ids, directory, workers=8):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n\n    def one(iid):\n        record = dict(candidates[iid])\n        path = directory / f"oi_val_{iid}.jpg"\n        try:\n            download(f"https://open-images-dataset.s3.amazonaws.com/validation/{iid}.jpg", path)\n            with Image.open(path) as image:\n                image.verify()\n            with Image.open(path) as image:\n                width, height = image.size\n                orientation = image.getexif().get(274, 1)\n                if width < 32 or height < 32 or orientation != 1:\n                    raise ValueError("JPEG pequeno ou com orientação EXIF não trivial.")\n                image.load()\n            tiny = {g["classe_id"] for g in record["gt_normalizado"] if\n                    (g["bbox_norm"][2]-g["bbox_norm"][0])*width < 1 or\n                    (g["bbox_norm"][3]-g["bbox_norm"][1])*height < 1}\n            # A avaliação da ausência fica desconhecida para a classe afetada.\n            record["restricoes_cobertura"] = dict(record["restricoes_cobertura"])\n            for cid in tiny:\n                for key in ["known_mask", "positive_mask", "negative_mask"]:\n                    record[key] &= ~(1 << cid)\n                record["restricoes_cobertura"][str(cid)] = ["caixa_menor_que_um_pixel"]\n            record["classes_positivas_avaliaveis"] = [c for c in range(12)\n                                                      if record["positive_mask"] & (1 << c)]\n            if not (record["positive_mask"] or record["known_mask"] & ALERT_MASK == ALERT_MASK):\n                raise ValueError("Nenhuma cobertura útil após conferir as dimensões reais.")\n            record.update(image=path, width=width, height=height, sha256=sha256_file(path),\n                          phash=phash(path), name=path.name)\n            return record, None\n        except Exception as exc:\n            return None, {"id": iid, "motivo": type(exc).__name__, "detalhe": str(exc)[:240]}\n\n    valid, failures = {}, []\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        for done, (record, failure) in enumerate(executor.map(one, ids), 1):\n            if record:\n                valid[record["id"]] = record\n            else:\n                failures.append(failure)\n            if done % 100 == 0 or done == len(ids):\n                print(f"Novas imagens: {done}/{len(ids)}; válidas: {len(valid)}", flush=True)\n    return valid, failures\n\n\ndef deduplicate(records, prior_hashes, home_near, seed=42):\n    accepted, excluded, seen_hash, near = {}, [], set(), NearIndex()\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        record = records[iid]\n        reason, similar = None, None\n        if record["sha256"] in prior_hashes:\n            reason = "bytes_presentes_nos_dados_anteriores"\n        elif record["sha256"] in seen_hash:\n            reason = "bytes_repetidos_no_conjunto_novo"\n        elif (similar := home_near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_com_home_train_ou_val"\n        elif (similar := near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_dentro_do_conjunto_novo"\n        if reason:\n            excluded.append({"id": iid, "motivo": reason, "semelhante_a": similar})\n            continue\n        accepted[iid] = record\n        seen_hash.add(record["sha256"])\n        near.add(record["phash"], iid)\n    return accepted, excluded\n\n\ndef freeze_dataset(records, roles, directory, source_hashes, prior_split_hash, seed=42):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    manifest, runtime = [], []\n    for iid in sorted(roles):\n        r, role = records[iid], roles[iid]\n        boxes = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            boxes.append({**g, "bbox": [x1*r["width"], y1*r["height"],\n                                       x2*r["width"], y2*r["height"]]})\n        coverage = {k: r[k] for k in ["known_mask", "positive_mask", "negative_mask",\n                                      "restricoes_cobertura"]}\n        label = directory / "labels" / role / f"oi_val_{iid}.txt"\n        coverage_file = directory / "coverage" / role / f"oi_val_{iid}.json"\n        label.parent.mkdir(parents=True, exist_ok=True)\n        lines = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            lines.append(f"{g[\'classe_id\']} {(x1+x2)/2:.10f} {(y1+y2)/2:.10f} {(x2-x1):.10f} {(y2-y1):.10f}")\n        label.write_text("\\n".join(lines)+("\\n" if lines else ""), encoding="utf-8")\n        save_json(coverage_file, coverage)\n        image_rel = f"images/{role}/oi_val_{iid}.jpg"\n        item = {"id": iid, "role": role, "image": image_rel,\n                "label": label.relative_to(directory).as_posix(),\n                "coverage": coverage_file.relative_to(directory).as_posix(),\n                "sha256": r["sha256"], "label_sha256": sha256_file(label),\n                "coverage_sha256": sha256_file(coverage_file),\n                "width": r["width"], "height": r["height"], "known_mask": r["known_mask"],\n                "positive_mask": r["positive_mask"], "negative_mask": r["negative_mask"],\n                "phash": f"{r[\'phash\']:016x}", "creditos": r["creditos"],\n                "gt_normalizado": r["gt_normalizado"]}\n        manifest.append(item)\n        runtime.append({**r, "role": role, "coorte": "openimages_val_nova",\n                        "gt": boxes, "label": label, "coverage": coverage_file})\n    metadata = {"versao": "11-validacao-independente-v1", "seed": seed, "nomes": NAMES,\n                "fontes_metadata_sha256": source_hashes, "origem_split": "validation",\n                "divisao_100k_anterior_sha256": prior_split_hash,\n                "contagens": dict(Counter(roles.values())),\n                "criterio": "Diversidade de classes, hash determinístico e pHash; sem previsões",\n                "cobertura": "Positivo explícito na raiz ou negativo na raiz/ancestral; classes problemáticas excluídas",\n                "independencia_limitada_ao_historico_conhecido": True,\n                "pretreinamentos_publicos_nao_auditados": True,\n                "reserva_futura_avaliada": False, "arquivos": manifest}\n    dataset_hash = hashlib.sha256(canonical(metadata).encode()).hexdigest()\n    metadata["dataset_sha256"] = dataset_hash\n    save_json(directory / "manifesto_validacao_nova.json", metadata)\n    save_csv(directory / "divisao.csv", [{k: r[k] for k in\n                                        ["id", "role", "sha256", "known_mask", "positive_mask"]}\n                                       for r in manifest])\n    archive = directory.parent / f"validacao_independente_11_{dataset_hash[:12]}.zip"\n    # Os JPEGs permanecem sem recompressão.\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as z:\n        z.write(directory / "manifesto_validacao_nova.json", "manifesto_validacao_nova.json")\n        z.write(directory / "divisao.csv", "divisao.csv")\n        for r, item in zip(sorted(runtime, key=lambda r: r["id"]), manifest):\n            z.write(r["image"], item["image"])\n            z.write(r["label"], item["label"])\n            z.write(r["coverage"], item["coverage"])\n    return runtime, metadata, archive\n\n\ndef support_rows(records):\n    rows = []\n    for role in sorted({r["role"] for r in records}):\n        group = [r for r in records if r["role"] == role]\n        for cid, name in enumerate(NAMES):\n            gt = [g for r in group for g in r["gt"] if\n                  g["classe_id"] == cid and r["known_mask"] & (1 << cid)]\n            rows.append({"conjunto": role, "classe_id": cid, "classe": name,\n                         "imagens_cobertura_conhecida": sum(bool(r["known_mask"] & (1 << cid)) for r in group),\n                         "imagens_positivas_avaliaveis": sum(bool(r["positive_mask"] & (1 << cid)) for r in group),\n                         "caixas_avaliaveis": len(gt), "suporte_reduzido": len(gt) < 30})\n    return rows\n', 'exp10_partial.py': '"""Experimento 09: caixas humanas parciais, fontes equilibradas e Home VAL."""\nfrom __future__ import annotations\n\nimport hashlib\nimport inspect\nimport io\nimport json\nimport math\nimport shutil\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport ultralytics\nimport yaml\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Sampler\nfrom ultralytics.models.yolo.detect import DetectionTrainer\nfrom ultralytics.nn.tasks import DetectionModel\nfrom ultralytics.utils.loss import v8DetectionLoss\nfrom ultralytics.utils.torch_utils import unwrap_model\n\n\nVERSION = "10-openimages-80-20-1ep-3.0"\nULTRALYTICS_VERSION = "8.4.150"\nLOSS_SOURCE_SHA = "357053aa9a24ad2c4d541c3ada3a27232a10d679f89c7b4957b9a84a7da4ec1d"\n\nNOMES = [\n    "bed", "sofa", "chair", "table", "lamp", "tv",\n    "laptop", "wardrobe", "window", "door",\n    "potted plant", "photo frame"\n]\n\nALL_KNOWN = (1 << 12) - 1\n\n\ndef sha256_file(p):\n    h = hashlib.sha256()\n\n    with Path(p).open("rb") as f:\n        for b in iter(lambda: f.read(4 * 1024 * 1024), b""):\n            h.update(b)\n\n    return h.hexdigest()\n\n\ndef canonical_json(x):\n    return json.dumps(\n        x,\n        sort_keys=True,\n        ensure_ascii=False,\n        separators=(",", ":")\n    )\n\n\ndef save_json(p, x):\n    p = Path(p)\n    p.parent.mkdir(parents=True, exist_ok=True)\n\n    tmp = p.with_name(p.name + ".tmp")\n    tmp.write_text(\n        json.dumps(x, ensure_ascii=False, indent=2),\n        encoding="utf-8"\n    )\n    tmp.replace(p)\n\n\ndef check_api():\n    if ultralytics.__version__ != ULTRALYTICS_VERSION:\n        raise RuntimeError(\n            f"Instale ultralytics=={ULTRALYTICS_VERSION} "\n            "e reinicie o ambiente se necessário."\n        )\n\n    if sha256_file(inspect.getfile(v8DetectionLoss)) != LOSS_SOURCE_SHA:\n        raise RuntimeError(\n            "A implementação da perda difere da versão verificada; "\n            "interrompido antes do treino."\n        )\n\n\nclass BCEComCobertura(nn.Module):\n    """Zera perdas negativas de classes desconhecidas; conserva positivos."""\n\n    def __init__(self, original=None):\n        super().__init__()\n        self.original = original or nn.BCEWithLogitsLoss(reduction="none")\n        self.known = None\n\n    def forward(self, logits, target):\n        if self.known is None:\n            raise RuntimeError(\n                "Cobertura não foi transmitida à perda parcial."\n            )\n\n        known = self.known.to(\n            device=logits.device,\n            dtype=torch.bool\n        )\n\n        if known.shape != (logits.shape[0], logits.shape[-1]):\n            raise RuntimeError(\n                "Dimensões de cobertura incompatíveis com batch/classes."\n            )\n\n        permitted = known[:, None, :] | (target > 0)\n\n        return self.original(logits, target) * permitted.to(logits.dtype)\n\n\nclass DetectionLossParcial(v8DetectionLoss):\n    """Altera somente a BCE; caixa, DFL, assignment e ganhos são os padrões."""\n\n    def __init__(self, model):\n        super().__init__(model)\n\n        if (\n            self.nc != 12\n            or not isinstance(self.bce, nn.BCEWithLogitsLoss)\n            or self.bce.reduction != "none"\n        ):\n            raise RuntimeError(\n                "Cabeça ou perda de detecção não é a esperada "\n                "para este experimento."\n            )\n\n        self.bce = BCEComCobertura(self.bce)\n\n    def loss(self, preds, batch):\n        if "class_known" not in batch:\n            raise RuntimeError(\n                "Batch de treinamento sem cobertura; "\n                "não substituir por negativos presumidos."\n            )\n\n        self.bce.known = batch["class_known"]\n\n        try:\n            return super().loss(preds, batch)\n        finally:\n            self.bce.known = None\n\n\nclass SamplerDuasFontes(Sampler):\n    """Cada batch contém metade Home e metade OI; uma época cobre todos os OI."""\n\n    def __init__(self, home_indices, oi_indices, batch_size, seed=42):\n        self.home = list(home_indices)\n        self.oi = list(oi_indices)\n\n        if (\n            not self.home\n            or not self.oi\n            or batch_size < 2\n            or batch_size % 2\n        ):\n            raise ValueError(\n                "São necessárias duas fontes e um batch par >=2."\n            )\n\n        self.batch = int(batch_size)\n        self.half = self.batch // 2\n        self.batches = math.ceil(len(self.oi) / self.half)\n        self.seed = int(seed)\n        self.epoch = 0\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def __len__(self):\n        return self.batches * self.batch\n\n    def __iter__(self):\n        rng = np.random.default_rng(self.seed + self.epoch)\n\n        home, oi = [], []\n        needed = self.batches * self.half\n\n        while len(home) < needed:\n            home.extend(rng.permutation(self.home).tolist())\n\n        while len(oi) < needed:\n            oi.extend(rng.permutation(self.oi).tolist())\n\n        for start in range(0, needed, self.half):\n            batch = (\n                home[start:start + self.half]\n                + oi[start:start + self.half]\n            )\n            rng.shuffle(batch)\n            yield from batch\n\n\nclass LoaderFinito(DataLoader):\n    def reset(self):\n        # Um novo iterador é criado em cada época.\n        pass\n\n\ndef epoch_sampler(trainer):\n    sampler = getattr(trainer.train_loader, "sampler", None)\n\n    if isinstance(sampler, SamplerDuasFontes):\n        sampler.set_epoch(trainer.epoch)\n\n\nclass TrainerOpenImages(DetectionTrainer):\n    # Configurado antes de criar o trainer.\n    # Não é anexado ao DetectionModel salvo.\n    source_records = None\n\n    def __init__(self, *args, **kwargs):\n        check_api()\n        super().__init__(*args, **kwargs)\n        self.add_callback("on_train_epoch_start", epoch_sampler)\n\n    def get_dataloader(\n        self,\n        dataset_path,\n        batch_size=16,\n        rank=0,\n        mode="train"\n    ):\n        if mode != "train":\n            return super().get_dataloader(\n                dataset_path,\n                batch_size,\n                rank,\n                mode\n            )\n\n        if self.source_records is None:\n            raise RuntimeError(\n                "As fontes/coberturas ainda não foram configuradas."\n            )\n\n        if self.world_size > 1 or self.args.compile or self.args.rect:\n            raise RuntimeError(\n                "Este experimento usa uma GPU, "\n                "compile=False e rect=False."\n            )\n\n        for k in ["mosaic", "mixup", "cutmix", "copy_paste"]:\n            if float(getattr(self.args, k, 0)) != 0:\n                raise RuntimeError(\n                    f"{k} deve ficar em zero para preservar "\n                    "a cobertura por imagem."\n                )\n\n        dataset = self.build_dataset(\n            dataset_path,\n            mode,\n            batch_size\n        )\n\n        home, oi = [], []\n\n        for i, arquivo in enumerate(dataset.im_files):\n            key = Path(arquivo).resolve().as_posix()\n\n            if key not in self.source_records:\n                raise RuntimeError(\n                    f"Imagem sem proveniência/cobertura: {arquivo}"\n                )\n\n            source = self.source_records[key]["source"]\n\n            if source == "home":\n                home.append(i)\n            elif source == "openimages":\n                oi.append(i)\n            else:\n                raise RuntimeError(\n                    "Fonte de treinamento não reconhecida."\n                )\n\n        sampler = SamplerDuasFontes(\n            home,\n            oi,\n            batch_size,\n            self.args.seed\n        )\n\n        generator = torch.Generator().manual_seed(self.args.seed)\n\n        return LoaderFinito(\n            dataset,\n            batch_size=batch_size,\n            sampler=sampler,\n            num_workers=self.args.workers,\n            pin_memory=self.device.type == "cuda",\n            collate_fn=dataset.collate_fn,\n            generator=generator,\n            drop_last=False\n        )\n\n    def preprocess_batch(self, batch):\n        batch = super().preprocess_batch(batch)\n\n        masks = []\n\n        for arquivo in batch["im_file"]:\n            key = Path(arquivo).resolve().as_posix()\n\n            if key not in self.source_records:\n                raise RuntimeError(\n                    "Batch contém imagem fora do manifesto de treino."\n                )\n\n            bitmask = self.source_records[key]["known_mask"]\n\n            masks.append([\n                bool(bitmask & (1 << cid))\n                for cid in range(12)\n            ])\n\n        batch["class_known"] = torch.tensor(\n            masks,\n            dtype=torch.bool,\n            device=self.device\n        )\n\n        model = unwrap_model(self.model)\n        criterion = getattr(model, "criterion", None)\n\n        if criterion is None:\n            # Modelo padrão, já colocado no dispositivo.\n            # A EMA foi criada antes desta etapa e valida Home\n            # com a perda padrão de cobertura completa.\n            model.criterion = DetectionLossParcial(model)\n\n        elif not isinstance(criterion, DetectionLossParcial):\n            raise RuntimeError(\n                "O treino iniciou com uma perda diferente "\n                "da perda parcial."\n            )\n\n        return batch\n\n\ndef self_test_bce():\n    logits = torch.zeros(\n        (2, 3, 12),\n        requires_grad=True\n    )\n\n    targets = torch.zeros_like(logits)\n    targets[1, 0, 7] = 1\n\n    known = torch.ones(\n        (2, 12),\n        dtype=torch.bool\n    )\n    known[1, 7] = False\n    known[1, 10] = False\n\n    partial = BCEComCobertura()\n    partial.known = known\n    partial(logits, targets).sum().backward()\n\n    assert logits.grad[1, :, 10].eq(0).all()\n    assert logits.grad[1, 1:, 7].eq(0).all()\n    assert logits.grad[1, 0, 7].item() < 0\n    assert logits.grad[0].ne(0).all()\n\n    full = BCEComCobertura()\n    full.known = torch.ones(\n        (2, 12),\n        dtype=torch.bool\n    )\n\n    torch.testing.assert_close(\n        full(logits.detach(), targets),\n        nn.BCEWithLogitsLoss(reduction="none")(\n            logits.detach(),\n            targets\n        )\n    )\n\n    return {\n        "gradiente_negativo_desconhecido_zero": True,\n        "caixa_positiva_preservada": True,\n        "cobertura_completa_igual_bce_padrao": True\n    }\n\n\ndef read_safe(z, name):\n    path = PurePosixPath(name)\n\n    if path.is_absolute() or ".." in path.parts or "\\\\" in name:\n        raise RuntimeError(\n            f"Caminho inseguro dentro de pacote: {name}"\n        )\n\n    return z.read(name)\n\n\ndef validate_yolo(raw, nclasses=12, source="<anotação>"):\n    """\n    Ignora linhas vazias e valida as linhas preenchidas.\n    Em caso de erro, informa arquivo, linha e conteúdo.\n    """\n    result = []\n\n    try:\n        text = raw.decode("utf-8-sig")\n    except UnicodeDecodeError as exc:\n        raise RuntimeError(\n            f"Anotação YOLO não está em UTF-8: {source}"\n        ) from exc\n\n    for line_number, line in enumerate(text.splitlines(), 1):\n        if not line.strip():\n            continue\n\n        values = line.split()\n        location = f"{source}, linha {line_number}"\n\n        if len(values) != 5:\n            raise RuntimeError(\n                f"Anotação YOLO inválida em {location}: "\n                "esperados 5 campos "\n                "(classe x y largura altura), "\n                f"encontrados {len(values)}. "\n                f"Conteúdo: {line[:160]!r}"\n            )\n\n        try:\n            cid, *bbox = map(float, values)\n        except ValueError as exc:\n            raise RuntimeError(\n                f"Campo não numérico em {location}: "\n                f"{line[:160]!r}"\n            ) from exc\n\n        if (\n            not cid.is_integer()\n            or not 0 <= cid < nclasses\n            or not all(math.isfinite(x) for x in bbox)\n        ):\n            raise RuntimeError(\n                f"Classe ou caixa YOLO inválida em {location}: "\n                f"{line[:160]!r}"\n            )\n\n        x, y, w, h = bbox\n\n        if not (\n            0 <= x <= 1\n            and 0 <= y <= 1\n            and 0 < w <= 1\n            and 0 < h <= 1\n            and x - w / 2 >= -1e-6\n            and x + w / 2 <= 1 + 1e-6\n            and y - h / 2 >= -1e-6\n            and y + h / 2 <= 1 + 1e-6\n        ):\n            raise RuntimeError(\n                f"Caixa YOLO fora dos limites em {location}: "\n                f"{line[:160]!r}"\n            )\n\n        result.append(int(cid))\n\n    if not result:\n        raise RuntimeError(\n            f"Imagem desta preparação sem caixas: {source}"\n        )\n\n    return result\n\n\ndef write_checked(dest, raw, expected_sha):\n    if hashlib.sha256(raw).hexdigest() != expected_sha:\n        raise RuntimeError(\n            "Conteúdo do pacote diferente do SHA-256 no manifesto."\n        )\n\n    dest = Path(dest)\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    dest.write_bytes(raw)\n\n\ndef prepare_training(\n    zip_home,\n    oi_drive,\n    local,\n    results,\n    expected,\n    batches_oi=None,\n    home_train_count=1824,\n    home_val_count=456,\n    oi_count=100000,\n    package_fetcher=None\n):\n    """Snapshot Home TRAIN/VAL e de todos os lotes liberados para 100k."""\n\n    local = Path(local)\n    results = Path(results)\n    oi_drive = Path(oi_drive)\n\n    local.mkdir(parents=True, exist_ok=True)\n    results.mkdir(parents=True, exist_ok=True)\n\n    prep_cfg = json.loads(\n        (oi_drive / "configuracao_preparacao.json").read_text()\n    )\n\n    cfg_sha = hashlib.sha256(\n        canonical_json(prep_cfg).encode()\n    ).hexdigest()\n\n    if (\n        cfg_sha != expected["oi_config_sha256"]\n        or prep_cfg["nomes"] != NOMES\n    ):\n        raise RuntimeError(\n            "Preparação Open Images diferente da auditada no piloto."\n        )\n\n    home_zip_local = local / "home_sun_snapshot.zip"\n    shutil.copyfile(zip_home, home_zip_local)\n\n    if sha256_file(home_zip_local) != expected["home_zip_sha256"]:\n        raise RuntimeError(\n            "ZIP Home+SUN diferente do preservado no 07b/08."\n        )\n\n    records, home_val = {}, []\n    extract_home = local / "base_existente"\n\n    with zipfile.ZipFile(home_zip_local) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError(\n                "ZIP Home+SUN tem nomes repetidos."\n            )\n\n        manifest = json.loads(\n            read_safe(z, "verificacao_preparacao.json")\n        )\n\n        if (\n            manifest["test"]["imagens"] != 403\n            or manifest["test"]["sha256"] != expected["test_sha256"]\n        ):\n            raise RuntimeError(\n                "TEST preservado diferente do esperado."\n            )\n\n        if (\n            {int(k): v for k, v in manifest["classes"].items()}\n            != dict(enumerate(NOMES))\n        ):\n            raise RuntimeError(\n                "Classes/IDs da base Home incompatíveis."\n            )\n\n        root = manifest["dataset_dir"]\n\n        for item in manifest["arquivos"]:\n            name = item["arquivo"]\n            p = PurePosixPath(name)\n\n            in_home_train = (\n                name.startswith(f"{root}/images/train/raw_train/")\n                or name.startswith(f"{root}/labels/train/raw_train/")\n            )\n\n            in_home_val = (\n                name.startswith(f"{root}/images/val/raw_train/")\n                or name.startswith(f"{root}/labels/val/raw_train/")\n            )\n\n            code_or_meta = (\n                name.startswith("codigo/")\n                or (\n                    p.suffix in {\n                        ".json", ".csv", ".yaml", ".yml", ".txt"\n                    }\n                    and "/images/" not in name\n                    and "/labels/" not in name\n                )\n            )\n\n            if not (in_home_train or in_home_val or code_or_meta):\n                continue\n\n            raw = read_safe(z, name)\n            dest = extract_home / name\n            write_checked(dest, raw, item["sha256"])\n\n            if name.startswith(f"{root}/images/train/raw_train/"):\n                records[dest.resolve().as_posix()] = {\n                    "source": "home",\n                    "known_mask": ALL_KNOWN,\n                    "sha256": item["sha256"]\n                }\n\n            elif name.startswith(f"{root}/images/val/raw_train/"):\n                home_val.append(\n                    dest.resolve().as_posix()\n                )\n\n        listed = {\n            item["arquivo"]\n            for item in manifest["arquivos"]\n        }\n\n        # O hash do ZIP também autentica o código auxiliar\n        # que não esteja listado individualmente no manifesto.\n        for name in set(z.namelist()) - listed:\n            if name.startswith("codigo/") and not name.endswith("/"):\n                raw = read_safe(z, name)\n\n                write_checked(\n                    extract_home / name,\n                    raw,\n                    hashlib.sha256(raw).hexdigest()\n                )\n\n    home_dataset = extract_home / root\n    home_images = list(records)\n\n    if (\n        len(home_images) != home_train_count\n        or len(home_val) != home_val_count\n    ):\n        raise RuntimeError(\n            "Home TRAIN/VAL inesperados: "\n            f"{len(home_images)}/{len(home_val)}; "\n            f"esperados {home_train_count}/{home_val_count}."\n        )\n\n    for image in [*home_images, *home_val]:\n        label = Path(\n            image.replace("/images/", "/labels/")\n        ).with_suffix(".txt")\n\n        validate_yolo(\n            label.read_bytes(),\n            source=str(label)\n        )\n\n    packages = json.loads(\n        (oi_drive / "manifesto_pacotes.json").read_text()\n    )\n\n    if packages["config_sha256"] != cfg_sha:\n        raise RuntimeError(\n            "Manifesto dos lotes tem configuração diferente."\n        )\n\n    selected = sorted(\n        packages["lotes"],\n        key=lambda r: r["arquivo"]\n    )\n    if batches_oi is None:\n        batches_oi = len(selected)\n\n    if len(selected) != batches_oi:\n        raise RuntimeError(\n            "Quantidade de lotes difere da liberação para 100k."\n        )\n\n    oi_ids, oi_hashes, package_snapshot = set(), set(), []\n    oi_root = local / "openimages"\n    per_class = Counter()\n\n    for item in selected:\n        name = item["arquivo"]\n\n        if PurePosixPath(name).name != name:\n            raise RuntimeError(\n                "Nome de pacote não é um nome simples."\n            )\n\n        snapshot = local / name\n\n        if package_fetcher is None:\n            shutil.copyfile(oi_drive / "lotes" / name, snapshot)\n        else:\n            print(f"Pacote {len(package_snapshot) + 1}/{len(selected)}: {name}", flush=True)\n            package_fetcher(item, snapshot)\n\n        if (\n            sha256_file(snapshot) != item["zip_sha256"]\n            or snapshot.stat().st_size != item["zip_bytes"]\n        ):\n            raise RuntimeError(\n                f"Pacote alterado ou incompleto: {name}"\n            )\n\n        with zipfile.ZipFile(snapshot) as z:\n            if len(z.namelist()) != len(set(z.namelist())):\n                raise RuntimeError(\n                    "ZIP Open Images tem membros repetidos."\n                )\n\n            m = json.loads(\n                read_safe(z, "manifesto_lote.json")\n            )\n\n            if (\n                m["config_sha256"] != cfg_sha\n                or m["origem_split"] != "train"\n            ):\n                raise RuntimeError(\n                    "Lote não é da seleção do treino "\n                    "Open Images auditado."\n                )\n\n            for r in m["aceitas"]:\n                if r["id"] in oi_ids or r["sha256"] in oi_hashes:\n                    raise RuntimeError(\n                        "ID ou bytes duplicados nos lotes usados no treino."\n                    )\n\n                oi_ids.add(r["id"])\n                oi_hashes.add(r["sha256"])\n\n                im_raw = read_safe(z, r["imagem"])\n                label_raw = read_safe(z, r["label"])\n                cov_raw = read_safe(z, r["cobertura"])\n\n                for key, raw, hkey in [\n                    ("imagem", im_raw, "sha256"),\n                    ("label", label_raw, "label_sha256"),\n                    ("cobertura", cov_raw, "cobertura_sha256")\n                ]:\n                    write_checked(\n                        oi_root / r[key],\n                        raw,\n                        r[hkey]\n                    )\n\n                cls = validate_yolo(\n                    label_raw,\n                    source=f"{name}:{r[\'label\']}"\n                )\n\n                cov = json.loads(cov_raw)\n\n                if (\n                    cov["known_mask"] != r["known_mask"]\n                    or not isinstance(r["known_mask"], int)\n                    or not 0 <= r["known_mask"] <= ALL_KNOWN\n                ):\n                    raise RuntimeError(\n                        "Cobertura do lote inconsistente."\n                    )\n\n                known_list = (\n                    set(cov["classes_positivas_verificadas_exaustivas"])\n                    | set(cov["classes_negativas_verificadas"])\n                )\n\n                if sum(1 << cid for cid in known_list) != r["known_mask"]:\n                    raise RuntimeError(\n                        "Lista de classes conhecidas "\n                        "não corresponde à máscara."\n                    )\n\n                if set(cls) & set(cov["classes_negativas_verificadas"]):\n                    raise RuntimeError(\n                        "Caixa positiva contradiz ausência verificada."\n                    )\n\n                with Image.open(io.BytesIO(im_raw)) as im:\n                    im.load()\n\n                    if im.size != (r["width"], r["height"]):\n                        raise RuntimeError(\n                            "Dimensões da imagem diferem do manifesto."\n                        )\n\n                image = (\n                    oi_root / r["imagem"]\n                ).resolve().as_posix()\n\n                records[image] = {\n                    "source": "openimages", "known_mask": r["known_mask"], "sha256": r["sha256"],\n                    "oi_id": r["id"], "lote": name,\n                    "classe_primaria": int(r.get("classe_primaria_selecao", min(cls))),\n                    "classes_com_caixa": cls, "width": r["width"], "height": r["height"],\n                    "label_sha256": r["label_sha256"], "cobertura_sha256": r["cobertura_sha256"],\n                }\n\n                per_class.update(cls)\n\n        package_snapshot.append(item)\n\n        # Remove apenas a cópia local do ZIP do lote.\n        snapshot.unlink()\n\n    if len(oi_ids) != oi_count:\n        raise RuntimeError(\n            f"Treinamento esperava {oi_count} imagens OI, "\n            f"encontrou {len(oi_ids)}."\n        )\n\n    if set(home_val) & set(records):\n        raise RuntimeError(\n            "Imagem de validação aparece no treino."\n        )\n\n    val_hashes = {\n        sha256_file(x)\n        for x in home_val\n    }\n\n    if val_hashes & {\n        r["sha256"]\n        for r in records.values()\n    }:\n        raise RuntimeError(\n            "Imagem de treino coincide em bytes "\n            "com a validação Home."\n        )\n\n    # O 08 já realizou a triagem de contaminação.\n    # Aqui repetimos a verificação por bytes contra Home VAL.\n    train_list = local / "train.txt"\n    val_list = local / "home_val.txt"\n\n    train_list.write_text(\n        "\\n".join(sorted(records)) + "\\n",\n        encoding="utf-8"\n    )\n\n    val_list.write_text(\n        "\\n".join(sorted(home_val)) + "\\n",\n        encoding="utf-8"\n    )\n\n    data_yaml = local / "data_home_openimages.yaml"\n\n    data_yaml.write_text(\n        yaml.safe_dump(\n            {\n                "path": str(local.resolve()),\n                "train": str(train_list),\n                "val": str(val_list),\n                "names": dict(enumerate(NOMES))\n            },\n            sort_keys=False,\n            allow_unicode=True\n        ),\n        encoding="utf-8"\n    )\n\n    save_json(\n        local / "fontes_cobertura_treino.json",\n        records\n    )\n\n    summary = {\n        "versao": VERSION,\n        "home_train_imagens_unicas": len(home_images),\n        "home_val_imagens": len(home_val),\n        "openimages_imagens_unicas": len(oi_ids),\n        "treino_imagens_unicas": len(records),\n        "caixas_openimages": sum(per_class.values()),\n        "caixas_openimages_por_classe": dict(per_class),\n        "lotes_utilizados": package_snapshot,\n        "home_zip_sha256": expected["home_zip_sha256"],\n        "test_sha256": expected["test_sha256"],\n        "test_imagens": 403,\n        "test_extraido_ou_avaliado": False,\n        "sun_incluida_no_novo_treino": False,\n        "amostragem": (\n            "50% Home / 50% OI em cada batch; "\n            "Home reamostrado por índices, sem duplicar arquivos"\n        ),\n        "selecao_checkpoint": (\n            "mAP50-95 Home VAL; triagem de alertas após treino"\n        ),\n        "tratamento_parcial": (\n            "Zero gradiente BCE para negativos de classes desconhecidas; "\n            "caixas positivas preservadas"\n        ),\n        "config_preparacao_sha256": cfg_sha\n    }\n\n    save_json(\n        results / "resumo_dados_treino.json",\n        summary\n    )\n\n    save_json(\n        results / "lotes_utilizados.json",\n        package_snapshot\n    )\n\n    TrainerOpenImages.source_records = records\n\n    return {\n        "yaml": data_yaml,\n        "home_dataset": home_dataset,\n        "codigo": extract_home / "codigo",\n        "records": records,\n        "home_val": home_val,\n        "resumo": summary\n    }\n\n\ndef export_standard_checkpoint(source, dest):\n    """Confere que o checkpoint não depende das classes do treinador parcial."""\n\n    obj = torch.load(\n        source,\n        map_location="cpu",\n        weights_only=False\n    )\n\n    model = obj.get("ema") or obj.get("model")\n\n    if type(model) is not DetectionModel:\n        raise RuntimeError(\n            "O checkpoint não contém DetectionModel padrão."\n        )\n\n    model.criterion = None\n    model = model.float().eval()\n\n    if not all(\n        torch.isfinite(t).all()\n        for t in model.state_dict().values()\n        if t.is_floating_point()\n    ):\n        raise RuntimeError(\n            "Checkpoint com pesos não finitos."\n        )\n\n    obj["model"] = model\n    obj["ema"] = None\n    obj["optimizer"] = None\n\n    torch.save(obj, dest)\n\n    return {\n        "arquivo": str(dest),\n        "sha256": sha256_file(dest),\n        "modelo_padrao": True,\n        "criterion_serializada": False\n    }', 'exp12_protocol.py': '"""Orçamento e seleção do experimento 12, sem dependência de Torch."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport hashlib\nimport math\nimport numpy as np\n\nARMS = {\n    "A_home": {"home_per_batch": 16, "oi_per_batch": 0},\n    "B_home_replay": {"home_per_batch": 12, "oi_per_batch": 4},\n}\n\n\nclass BudgetSampler:\n    """Fluxos por fonte independentes; quantidade fixa de batches por época."""\n    def __init__(self, home, oi, batch_size, batches, oi_per_batch, seed=42):\n        self.home, self.oi = list(home), list(oi)\n        self.batch_size, self.batches = int(batch_size), int(batches)\n        self.oi_per_batch = int(oi_per_batch)\n        self.home_per_batch = self.batch_size - self.oi_per_batch\n        self.seed, self.epoch = int(seed), 0\n        if (not self.home or self.batches <= 0 or self.batch_size <= 0 or\n                not 0 <= self.oi_per_batch < self.batch_size or\n                (self.oi_per_batch and not self.oi) or set(self.home) & set(self.oi)):\n            raise ValueError("Fontes ou orçamento inválidos.")\n\n    def __len__(self):\n        return self.batches*self.batch_size\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def _stream(self, values, needed, source):\n        if not needed:\n            return []\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, source]))\n        result = []\n        while len(result) < needed:\n            result.extend(rng.permutation(values).tolist())\n        return result[:needed]\n\n    def __iter__(self):\n        home = self._stream(self.home, self.batches*self.home_per_batch, 0)\n        oi = self._stream(self.oi, self.batches*self.oi_per_batch, 1)\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, 2]))\n        for b in range(self.batches):\n            batch = home[b*self.home_per_batch:(b+1)*self.home_per_batch]\n            batch += oi[b*self.oi_per_batch:(b+1)*self.oi_per_batch]\n            rng.shuffle(batch)\n            yield from batch\n\n\ndef order_id(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef diverse_subset(rows, n=2000, seed=42, per_class=40):\n    if n <= 0 or n > len(rows) or len({r["id"] for r in rows}) != len(rows):\n        raise ValueError("Tamanho do replay ou identidade dos IDs inválidos.")\n    mapping = {r["id"]: r for r in rows}\n    groups = {c: sorted([r["id"] for r in rows if int(r["classe_primaria"]) == c],\n                        key=lambda iid: order_id(iid, seed)) for c in range(12)}\n    chosen = set()\n    for k in range(per_class):\n        for c in sorted(groups, key=lambda cid: (len(groups[cid]), cid)):\n            if k < len(groups[c]) and len(chosen) < n:\n                chosen.add(groups[c][k])\n    for iid in sorted(mapping, key=lambda x: order_id(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return [mapping[iid] for iid in sorted(chosen)]\n\n\ndef choose_replay(history_rows, n=2200, seed=42, per_class=40, forbidden_hashes=()):\n    batches = {f"lote_{i:06}.zip" for i in range(1, 6)}\n    rows = [r for r in history_rows if r["lote"] in batches]\n    if len(rows) != 5000 or any(r["split"] != "train" for r in rows):\n        raise ValueError("Os cinco lotes do piloto precisam conter 5.000 imagens só de treino.")\n    forbidden = set(forbidden_hashes)\n    return diverse_subset([r for r in rows if r["imagem_sha256"] not in forbidden],\n                          n, seed, per_class)\n\n\ndef validate_metric_block(block, label):\n    for k in ["precision", "recall", "f1"]:\n        x = block["alertas_por_objeto"][k]\n        if x is None or not math.isfinite(x):\n            raise ValueError(f"Métrica sem suporte ou não finita: {label}/{k}")\n    for k in ["alta_ref", "alta_sem_alerta"]:\n        if int(block[k]) < 0:\n            raise ValueError("Contagem inválida.")\n\n\ndef make_rule(baseline_home, candidate_external, dataset_sha, policy_sha):\n    validate_metric_block(baseline_home, "baseline_home")\n    validate_metric_block(candidate_external, "candidate_external")\n    return {\n        "versao": "12-regra-duas-validacoes-v1",\n        "home": {\n            "precision_maior_que": baseline_home["alertas_por_objeto"]["precision"],\n            "recall_minimo": baseline_home["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": baseline_home["alta_sem_alerta"],\n            "objetos_alerta_ref": baseline_home["alertas_por_objeto"]["tp"]+baseline_home["alertas_por_objeto"]["fn"],\n            "alta_ref": baseline_home["alta_ref"],\n        },\n        "externa": {\n            "precision_minima": candidate_external["alertas_por_objeto"]["precision"],\n            "recall_minimo": candidate_external["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": candidate_external["alta_sem_alerta"],\n            "objetos_alerta_ref": candidate_external["alertas_por_objeto"]["tp"]+candidate_external["alertas_por_objeto"]["fn"],\n            "alta_ref": candidate_external["alta_ref"],\n        },\n        "dataset_sha256": dataset_sha, "politica_sha256": policy_sha,\n        "conf_alertas": .25, "testes_consultados": False,\n        "tolerancia_numerica": 1e-12,\n        "criterio": "Home supera precisão do baseline, preserva recall e omissões altas; externa preserva precisão, recall e omissões altas do candidato 10.",\n        "desempate": "Precisão Home, recall Home, omissões altas Home, F1 externa, menos atualizações, identificador.",\n        "sem_elegivel": "Preservar baseline em uso; candidato 10 e todos os checkpoints continuam disponíveis.",\n    }\n\n\ndef judge(home, external, rule):\n    validate_metric_block(home, "home")\n    validate_metric_block(external, "external")\n    h, e = home["alertas_por_objeto"], external["alertas_por_objeto"]\n    rh, re = rule["home"], rule["externa"]\n    for observed, expected in [(h["tp"]+h["fn"], rh["objetos_alerta_ref"]),\n                               (e["tp"]+e["fn"], re["objetos_alerta_ref"]),\n                               (home["alta_ref"], rh["alta_ref"]),\n                               (external["alta_ref"], re["alta_ref"])]:\n        if observed != expected:\n            raise ValueError("Os denominadores de referência mudaram entre checkpoints.")\n    eps = rule["tolerancia_numerica"]\n    checks = {\n        "home_precision_superior": h["precision"] > rh["precision_maior_que"]+eps,\n        "home_recall_preservado": h["recall"] >= rh["recall_minimo"]-eps,\n        "home_altas_preservadas": home["alta_sem_alerta"] <= rh["alta_sem_alerta_maximo"],\n        "externa_precision_preservada": e["precision"] >= re["precision_minima"]-eps,\n        "externa_recall_preservado": e["recall"] >= re["recall_minimo"]-eps,\n        "externa_altas_preservadas": external["alta_sem_alerta"] <= re["alta_sem_alerta_maximo"],\n    }\n    return {**checks, "elegivel": all(checks.values()),\n            "retencao_externa": all(v for k, v in checks.items() if k.startswith("externa_"))}\n\n\ndef select_checkpoints(entries, rule):\n    if not entries:\n        raise ValueError("Nenhum checkpoint avaliado.")\n    assessed = []\n    for entry in entries:\n        checks = judge(entry["home"], entry["external"], rule)\n        assessed.append({**entry, "checks": checks})\n    eligible = [r for r in assessed if r["checks"]["elegivel"]]\n\n    def ranking(r):\n        h = r["home"]["alertas_por_objeto"]\n        return (-h["precision"], -h["recall"], r["home"]["alta_sem_alerta"],\n                -r["external"]["alertas_por_objeto"]["f1"], r["updates"], r["id"])\n\n    selected = min(eligible, key=ranking) if eligible else None\n    # Apenas para inspeção. A classificação exploratória não muda os requisitos.\n    retained = [r for r in assessed if r["checks"]["retencao_externa"]]\n    exploratory = min(retained, key=lambda r: (\n        -r["home"]["alertas_por_objeto"]["recall"],\n        r["home"]["alta_sem_alerta"],\n        -r["home"]["alertas_por_objeto"]["precision"],\n        -r["external"]["alertas_por_objeto"]["f1"],\n        r["updates"], r["id"])) if retained else None\n    return {"eligible": selected, "exploratory": exploratory, "assessed": assessed}\n', 'learn21_data.py': '"""Preparação automática de TRAIN ampliado e das validações já expostas."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip,hashlib,json,math,shutil,urllib.request,zipfile\nfrom pathlib import Path\nfrom PIL import Image\nimport yaml\nimport diag11_core as core\nimport diag11_data as d11\nfrom learn21_protocol import SPEC,digest,choose_candidates,balanced_pool\n\nQUARANTINE={\'home:homeobjects_yolo/images/train/raw_train/living_room_1p (200).jpg\',\n \'home:homeobjects_yolo/images/train/raw_train/living_room_1p (203).jpg\',\n \'home:homeobjects_yolo/images/train/raw_train/living_room_590.jpg\',\n \'home:homeobjects_yolo/images/train/raw_train/living_room_718.jpg\'}\n\ndef preparation_sha(config):\n    # Preserva a serialização canônica compacta da preparação histórica 10.\n    return hashlib.sha256(json.dumps(config,sort_keys=True,ensure_ascii=False,separators=(\',\',\':\')).encode()).hexdigest()\n\ndef safe_zip(path):\n    z=zipfile.ZipFile(path)\n    if len(z.namelist())!=len(set(z.namelist())):z.close();raise ValueError(\'ZIP com nomes repetidos.\')\n    for name in z.namelist():core.safe_member(name)\n    return z\n\ndef checked_write(root,name,raw,sha):\n    core.safe_member(name)\n    if hashlib.sha256(raw).hexdigest()!=sha:raise ValueError(\'Conteúdo alterado: \'+name)\n    p=Path(root)/name;p.parent.mkdir(parents=True,exist_ok=True)\n    if not p.is_file() or core.sha256_file(p)!=sha:p.write_bytes(raw)\n    return p\n\ndef public_weight(target,expected):\n    target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)\n    if target.is_file() and core.sha256_file(target)==expected[\'pretrained_sha256\']:return target\n    tmp=target.with_name(target.name+\'.part\')\n    with urllib.request.urlopen(expected[\'pretrained_url\'],timeout=60) as response,tmp.open(\'wb\') as f:\n        shutil.copyfileobj(response,f)\n    if core.sha256_file(tmp)!=expected[\'pretrained_sha256\']:\n        tmp.unlink(missing_ok=True);raise ValueError(\'O peso público YOLOv8s difere do arquivo verificado.\')\n    tmp.replace(target);return target\n\ndef source19(package,expected,output):\n    if core.sha256_file(package)!=expected[\'pacote_19\']:raise ValueError(\'Pacote 19 diferente do analisado.\')\n    with safe_zip(package) as z:\n        summary=json.loads(z.read(\'resumo_consolidacao_19.json\'))\n        if not summary[\'concluido\']:raise ValueError(\'Fonte 19 incompleta.\')\n        records=json.loads(gzip.decompress(z.read(\'dados_validacao_reutilizados.json.gz\')))\n        if Counter(r[\'role\'] for r in records)!={\'home_val\':456,\'val_nova\':1000}:raise ValueError(\'Papéis da validação alterados.\')\n        p=Path(output)/\'referencia19\';p.mkdir(parents=True,exist_ok=True)\n        for name in [\'comparacao_finais_e_referencias.csv\',\'texto_base_capitulo4.md\']:(p/name).write_bytes(z.read(name))\n        for cohort in [\'home_val_historica\',\'openimages_val_nova\']:\n            (p/(cohort+\'_candidato10.json\')).write_bytes(z.read(\'avaliacoes_reutilizadas/candidato_10/\'+cohort+\'/metricas.json\'))\n    return records\n\ndef dataset_protection(archive,expected):\n    if core.sha256_file(archive)!=expected[\'zip_novo\']:raise ValueError(\'ZIP externo diferente do preservado.\')\n    with safe_zip(archive) as z:meta=json.loads(z.read(\'manifesto_validacao_nova.json\'))\n    claimed=meta[\'dataset_sha256\']\n    if claimed!=expected[\'dataset_novo\'] or digest({k:v for k,v in meta.items() if k!=\'dataset_sha256\'})!=claimed:\n        raise ValueError(\'Manifesto externo alterado.\')\n    if Counter(r[\'role\'] for r in meta[\'arquivos\'])!={\'val_nova\':1000,\'reserva_futura\':300}:raise ValueError(\'Divisão externa inesperada.\')\n    # Da antiga reserva, somente identidade, bytes e pHash protegem a exclusão.\n    protected=[{k:r[k] for k in [\'id\',\'sha256\',\'phash\']} for r in meta[\'arquivos\']]\n    return meta,protected\n\ndef materialise_validation(archive,meta,development,root):\n    expected={r[\'id\']:r for r in development if r[\'role\']==\'val_nova\'};result=[]\n    with safe_zip(archive) as z:\n        for item in meta[\'arquivos\']:\n            if item[\'role\']!=\'val_nova\':continue\n            if item[\'id\'] not in expected:raise ValueError(\'ID externo fora da validação anterior.\')\n            r=expected[item[\'id\']]\n            for key in [\'sha256\',\'label_sha256\',\'known_mask\',\'width\',\'height\']:\n                if r[key]!=item[key]:raise ValueError(\'Referência externa alterada.\')\n            for field,hfield in [(\'image\',\'sha256\'),(\'label\',\'label_sha256\'),(\'coverage\',\'coverage_sha256\')]:\n                if not item[field].startswith({\'image\':\'images\',\'label\':\'labels\',\'coverage\':\'coverage\'}[field]+\'/val_nova/\'):\n                    raise ValueError(\'Arquivo fora da validação de desenvolvimento.\')\n                p=checked_write(root,\'external/\'+item[field],z.read(item[field]),item[hfield])\n                if field==\'image\':image=p\n            result.append({**r,\'image\':image,\'source\':\'validation\',\'role\':\'val_nova\'})\n    if len(result)!=1000:raise ValueError(\'Validação externa incompleta.\')\n    return sorted(result,key=lambda r:r[\'id\'])\n\ndef prepare_home(archive,expected,development,root,near,forbidden_sha):\n    if core.sha256_file(archive)!=expected[\'home_zip\']:raise ValueError(\'ZIP Home alterado.\')\n    previous={r[\'id\']:r for r in development if r[\'role\']==\'home_val\'}\n    train=[];val=[];excluded=[];all_hashes=set()\n    with safe_zip(archive) as z:\n        manifest=json.loads(z.read(\'verificacao_preparacao.json\'))\n        if manifest[\'test\'][\'sha256\']!=expected[\'test_home\'] or manifest[\'test\'][\'imagens\']!=403:raise ValueError(\'Home TEST diferente.\')\n        if {int(k):v for k,v in manifest[\'classes\'].items()}!=dict(enumerate(core.NAMES)):raise ValueError(\'Classes Home diferentes.\')\n        entries={r[\'arquivo\']:r for r in manifest[\'arquivos\']}\n        for name,item in sorted(entries.items()):\n            if \'/images/\' not in name or Path(name).suffix.lower() not in {\'.jpg\',\'.jpeg\',\'.png\',\'.bmp\',\'.webp\'}:continue\n            all_hashes.add(item[\'sha256\'])\n            role=\'train\' if \'/images/train/raw_train/\' in name else \'home_val\' if \'/images/val/raw_train/\' in name else None\n            if role is None:continue\n            iid=\'home:\'+name\n            if iid in QUARANTINE:excluded.append({\'id\':iid,\'motivo\':\'quarentena histórica\'});continue\n            label_name=name.replace(\'/images/\',\'/labels/\').rsplit(\'.\',1)[0]+\'.txt\'\n            token=hashlib.sha256(iid.encode()).hexdigest()[:24];folder=\'train\' if role==\'train\' else \'val\'\n            image=checked_write(root,f\'home/images/{folder}/{token}{Path(name).suffix}\',z.read(name),item[\'sha256\'])\n            label=checked_write(root,f\'home/labels/{folder}/{token}.txt\',z.read(label_name),entries[label_name][\'sha256\'])\n            with Image.open(image) as im:w,h=im.size;im.verify()\n            ph=d11.phash(image)\n            if role==\'train\' and (item[\'sha256\'] in forbidden_sha or near.find(ph)):\n                excluded.append({\'id\':iid,\'motivo\':\'coincidência com validação ou conjunto protegido\'});continue\n            r={\'id\':iid,\'image\':image,\'label\':label,\'name\':image.name,\'width\':w,\'height\':h,\'known_mask\':4095,\n                \'sha256\':item[\'sha256\'],\'label_sha256\':entries[label_name][\'sha256\'],\'phash\':ph,\n                \'gt\':core.parse_yolo(label.read_bytes(),w,h),\'role\':role,\'source\':\'home\',\'coorte\':\'home_val_historica\' if role==\'home_val\' else \'train\'}\n            if role==\'home_val\':\n                if iid not in previous or previous[iid][\'sha256\']!=r[\'sha256\'] or previous[iid][\'label_sha256\']!=r[\'label_sha256\']:raise ValueError(\'Home VAL alterada.\')\n                r[\'gt\']=previous[iid][\'gt\'];val.append(r)\n            else:train.append(r)\n    if len(val)!=456 or len(train)<500:raise ValueError(\'Home insuficiente após a separação automática.\')\n    return train,sorted(val,key=lambda r:r[\'id\']),excluded,all_hashes\n\ndef prepare_oi(packages,history,expected,root,drive,near,forbidden_ids,forbidden_sha,spec=SPEC,progress=None):\n    candidates=choose_candidates(history,forbidden_ids,forbidden_sha,spec);wanted={r[\'id\']:r for r in candidates}\n    records=[];excluded=[];seen=set();receipts=[]\n    lots=[item for item in packages[\'lotes\'] if item[\'arquivo\'] in {r[\'lote\'] for r in candidates}]\n    for li,item in enumerate(sorted(lots,key=lambda x:x[\'arquivo\']),1):\n        path=Path(root)/\'downloads\'/item[\'arquivo\']\n        print(f\'Lote de preparação {li}/{len(lots)}: {item["arquivo"]}\',flush=True)\n        drive.fetch(expected[\'oi_root\']+\'/lotes/\'+item[\'arquivo\'],path,item[\'zip_sha256\'])\n        if path.stat().st_size!=item[\'zip_bytes\']:raise ValueError(\'Tamanho do lote alterado.\')\n        with safe_zip(path) as z:\n            meta=json.loads(z.read(\'manifesto_lote.json\'))\n            if meta[\'config_sha256\']!=expected[\'preparacao\'] or meta[\'origem_split\']!=\'train\':raise ValueError(\'Lote fora do TRAIN.\')\n            for item2 in meta[\'aceitas\']:\n                iid=item2[\'id\']\n                if iid not in wanted:continue\n                prior=wanted[iid]\n                if iid in seen or prior[\'split\']!=\'train\' or prior[\'lote\']!=item[\'arquivo\'] or prior[\'imagem_sha256\']!=item2[\'sha256\'] or int(prior[\'known_mask\'])!=item2[\'known_mask\']:\n                    raise ValueError(\'Identidade ou divisão incompatível com a seleção.\')\n                seen.add(iid)\n                if iid in forbidden_ids or item2[\'sha256\'] in forbidden_sha:raise ValueError(\'TRAIN coincide com dado protegido.\')\n                raw={k:z.read(item2[k]) for k in [\'imagem\',\'label\',\'cobertura\']}\n                for field,hkey in [(\'imagem\',\'sha256\'),(\'label\',\'label_sha256\'),(\'cobertura\',\'cobertura_sha256\')]:\n                    if hashlib.sha256(raw[field]).hexdigest()!=item2[hkey]:raise ValueError(\'Arquivo do lote alterado.\')\n                cov=json.loads(raw[\'cobertura\']);pos=set(cov[\'classes_positivas_verificadas_exaustivas\']);neg=set(cov[\'classes_negativas_verificadas\'])\n                if pos&neg or sum(1<<c for c in pos|neg)!=item2[\'known_mask\'] or cov[\'known_mask\']!=item2[\'known_mask\']:raise ValueError(\'Cobertura parcial inválida.\')\n                image=checked_write(root,\'oi/images/train/oi_\'+iid+\'.jpg\',raw[\'imagem\'],item2[\'sha256\'])\n                label=checked_write(root,\'oi/labels/train/oi_\'+iid+\'.txt\',raw[\'label\'],item2[\'label_sha256\'])\n                coverage=checked_write(root,\'oi/coverage/train/oi_\'+iid+\'.json\',raw[\'cobertura\'],item2[\'cobertura_sha256\'])\n                with Image.open(image) as im:w,h=im.size;im.verify()\n                if (w,h)!=(item2[\'width\'],item2[\'height\']):raise ValueError(\'Dimensões do treino alteradas.\')\n                ph=d11.phash(image)\n                if near.find(ph):excluded.append({\'id\':iid,\'motivo\':\'semelhança pHash com dados protegidos ou treino já aceito\'});continue\n                gt=core.parse_yolo(raw[\'label\'],w,h)\n                if {g[\'classe_id\'] for g in gt}&neg or not pos<={g[\'classe_id\'] for g in gt}:raise ValueError(\'Cobertura parcial e caixas divergem.\')\n                near.add(ph,iid)\n                records.append({\'id\':\'oi:\'+iid,\'oi_id\':iid,\'name\':image.name,\'role\':\'train\',\'source\':\'openimages\',\'coorte\':\'train\',\n                    \'image\':image,\'label\':label,\'coverage\':coverage,\'width\':w,\'height\':h,\'known_mask\':item2[\'known_mask\'],\n                    \'sha256\':item2[\'sha256\'],\'label_sha256\':item2[\'label_sha256\'],\'coverage_sha256\':item2[\'cobertura_sha256\'],\n                    \'phash\':ph,\'gt\':gt,\'lote\':item[\'arquivo\']})\n        receipts.append({\'arquivo\':item[\'arquivo\'],\'sha256\':item[\'zip_sha256\']});path.unlink()\n        if progress:progress(li,len(lots))\n    if seen!=set(wanted):raise ValueError(\'Algum ID planejado não foi localizado nos lotes autenticados.\')\n    return balanced_pool(records,spec),excluded,receipts\n\ndef input_files(records,cohorts,root):\n    root=Path(root)\n    train=root/\'train.txt\';val=root/\'home_val.txt\'\n    train.write_text(\'\\n\'.join(str(Path(r[\'image\']).resolve()) for r in records)+\'\\n\')\n    val.write_text(\'\\n\'.join(str(Path(r[\'image\']).resolve()) for r in cohorts[\'home_val_historica\'])+\'\\n\')\n    target=root/\'data.yaml\';target.write_text(yaml.safe_dump({\'path\':str(root.resolve()),\'train\':str(train.resolve()),\'val\':str(val.resolve()),\'names\':dict(enumerate(core.NAMES))},sort_keys=False))\n    return target,{Path(r[\'image\']).resolve().as_posix():r for r in records}\n\ndef freeze_base(records,cohorts,root,plan_key,report):\n    root=Path(root)\n    def serial(r):\n        obj=core.json_clean(r)\n        for key in [\'image\',\'label\',\'coverage\']:\n            if key in r:obj[key]=Path(r[key]).relative_to(root).as_posix()\n        return obj\n    snapshot={\'version\':\'21-base-v1\',\'plan_key\':plan_key,\'train\':[serial(r) for r in records],\n        \'cohorts\':{c:[serial(r) for r in rows] for c,rows in cohorts.items()},\'report\':report}\n    core.save_json(root/\'base21_manifesto.json\',snapshot)\n    archive=root.parent/\'base_treino_21.zip\'\n    with zipfile.ZipFile(archive.with_suffix(\'.tmp\'),\'w\',zipfile.ZIP_STORED) as z:\n        def store(path,name):\n            # Datas dos arquivos não mudam a identidade da base numa nova sessão.\n            info=zipfile.ZipInfo(name,date_time=(1980,1,1,0,0,0));info.compress_type=zipfile.ZIP_STORED\n            info.external_attr=0o100644<<16\n            with Path(path).open(\'rb\') as source,z.open(info,\'w\',force_zip64=True) as target:shutil.copyfileobj(source,target,2**20)\n        store(root/\'base21_manifesto.json\',\'base21_manifesto.json\')\n        paths=set()\n        for r in records+[r for rows in cohorts.values() for r in rows]:\n            paths.update(Path(r[k]) for k in [\'image\',\'label\',\'coverage\'] if k in r)\n        for p in sorted(paths):store(p,p.relative_to(root).as_posix())\n    archive.with_suffix(\'.tmp\').replace(archive)\n    return archive,snapshot\n\ndef restore_base(archive,root,plan_key):\n    root=Path(root)\n    with safe_zip(archive) as z:\n        meta=json.loads(z.read(\'base21_manifesto.json\'))\n        if meta[\'plan_key\']!=plan_key:raise ValueError(\'Base preparada pertence a outro plano.\')\n        rows=meta[\'train\']+[r for group in meta[\'cohorts\'].values() for r in group]\n        for r in rows:\n            if r[\'role\'] not in {\'train\',\'home_val\',\'val_nova\'}:raise ValueError(\'Conjunto protegido presente na base preparada.\')\n            for field,hkey in [(\'image\',\'sha256\'),(\'label\',\'label_sha256\'),(\'coverage\',\'coverage_sha256\')]:\n                if field in r:checked_write(root,r[field],z.read(r[field]),r[hkey])\n    def hydrate(r):return {**r,**{k:root/r[k] for k in [\'image\',\'label\',\'coverage\'] if k in r}}\n    train=[hydrate(r) for r in meta[\'train\']];cohorts={c:[hydrate(r) for r in rows] for c,rows in meta[\'cohorts\'].items()}\n    return train,cohorts,meta[\'report\']\n', 'learn21_drive.py': '"""Drive, recibos atômicos e pacote leve; credenciais ficam fora dos resultados."""\nfrom __future__ import annotations\nimport base64\nimport configparser\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport zipfile\n\nimport diag11_core as core\n\nclass DigestMismatch(RuntimeError):\n    pass\n\nclass Drive:\n    def __init__(self, binary, config, root=\'gdrive_tcc:TCC_Visao_Computacional\'):\n        self.binary, self.config, self.root = str(binary), str(config), root\n\n    @classmethod\n    def from_kaggle(cls):\n        from kaggle_secrets import UserSecretsClient\n        root = Path(\'/tmp/tcc_rclone\')\n        root.mkdir(parents=True, exist_ok=True, mode=0o700)\n        root.chmod(0o700)\n        try:\n            secret = UserSecretsClient().get_secret(\'GDRIVE_RCLONE_CONFIG_B64\')\n        except Exception:\n            raise RuntimeError(\'Habilite o Secret existente GDRIVE_RCLONE_CONFIG_B64 em Add-ons → Secrets.\') from None\n        try:\n            raw = base64.b64decode(secret.strip(), validate=True)\n            cfg = configparser.ConfigParser(interpolation=None)\n            cfg.read_string(raw.decode(\'utf-8-sig\'))\n        except Exception:\n            raise RuntimeError(\'A configuração do Secret rclone não é válida.\') from None\n        if (not cfg.has_section(\'gdrive_tcc\') or cfg.get(\'gdrive_tcc\', \'type\', fallback=\'\') != \'drive\'\n                or not cfg.get(\'gdrive_tcc\', \'client_id\', fallback=\'\')\n                or not cfg.get(\'gdrive_tcc\', \'token\', fallback=\'\')\n                or cfg.get(\'gdrive_tcc\', \'scope\', fallback=\'\') != \'drive\'):\n            raise RuntimeError(\'O Secret precisa da conexão gdrive_tcc autorizada para leitura e gravação em Meu Drive.\')\n        cfg.remove_option(\'gdrive_tcc\', \'team_drive\')\n        config = root / \'gdrive_tcc.conf\'\n        with config.open(\'w\', encoding=\'utf-8\') as f:\n            cfg.write(f)\n        config.chmod(0o600)\n        del secret, raw, cfg\n        binary = root / \'rclone\'\n        if not binary.is_file():\n            executable = shutil.which(\'rclone\')\n            if not executable:\n                import os\n                prefix = [] if os.geteuid() == 0 else [\'sudo\']\n                subprocess.run(prefix + [\'apt-get\', \'update\', \'-qq\'], check=True)\n                subprocess.run(prefix + [\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', \'rclone\'], check=True)\n                executable = shutil.which(\'rclone\')\n            if not executable:\n                raise RuntimeError(\'Executável rclone indisponível após a instalação.\')\n            shutil.copy2(executable, binary)\n            binary.chmod(0o700)\n        return cls(binary, config)\n\n    def _call(self, args):\n        return subprocess.run([self.binary, \'--config\', self.config] + list(args)\n                              + [\'--retries\', \'2\', \'--low-level-retries\', \'3\',\n                                 \'--contimeout\', \'15s\', \'--timeout\', \'60s\'],\n                              capture_output=True, text=True)\n\n    def exists(self, relative):\n        result = self._call([\'lsjson\', self.root + \'/\' + relative, \'--stat\'])\n        if result.returncode == 0:\n            return True\n        if result.returncode in {3, 4}:\n            return False\n        raise RuntimeError(\'Falha ao consultar Drive (código \' + str(result.returncode) + \'): \' + relative)\n\n    def fetch(self, relative, target, expected_sha=None, optional=False):\n        target = Path(target)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        receipt = target.with_name(target.name + \'.download.json\')\n        if target.is_file() and target.stat().st_size:\n            local_sha = core.sha256_file(target)\n            prior = {}\n            if receipt.is_file():\n                try:\n                    prior = json.loads(receipt.read_text())\n                except (ValueError, OSError):\n                    pass\n            if ((expected_sha and local_sha == expected_sha)\n                    or (not expected_sha and prior.get(\'origem\') == relative and prior.get(\'sha256\') == local_sha)):\n                print(\'Entrada local conferida:\', relative, flush=True)\n                return target\n        if optional and not self.exists(relative):\n            return None\n        print(\'Baixando:\', relative, flush=True)\n        tmp = target.with_name(target.name + \'.part\')\n        result = self._call([\'copyto\', self.root + \'/\' + relative, str(tmp)])\n        if result.returncode:\n            raise RuntimeError(\'Download interrompido (código \' + str(result.returncode) + \'): \' + relative)\n        if not tmp.is_file() or not tmp.stat().st_size:\n            raise RuntimeError(\'Download vazio: \' + relative)\n        actual = core.sha256_file(tmp)\n        if expected_sha and actual != expected_sha:\n            tmp.replace(target.with_name(target.name + \'.rejeitado_\' + actual[:12]))\n            raise DigestMismatch(\'SHA-256 diferente da fonte autenticada: \' + relative)\n        tmp.replace(target)\n        core.save_json(receipt, {\'origem\': relative, \'sha256\': actual, \'bytes\': target.stat().st_size})\n        return target\n\n    def backup(self, local, relative, required=True):\n        print(\'Salvando resultados no Drive...\', flush=True)\n        result = self._call([\'copy\', str(local), self.root + \'/\' + relative,\n                             \'--exclude\', \'**/*.tmp\', \'--exclude\', \'**/*.part\', \'--exclude\', \'**/__pycache__/**\'])\n        if result.returncode and required:\n            raise RuntimeError(\'Falha no salvamento no Drive; resultados locais preservados. Código \' + str(result.returncode))\n        return result.returncode == 0\n\n    def restore(self, relative, local):\n        if not self.exists(relative):\n            return False\n        result = self._call([\'copy\', self.root + \'/\' + relative, str(local),\n                             \'--exclude\', \'*.zip\', \'--exclude\', \'**/*.tmp\', \'--exclude\', \'**/*.part\'])\n        if result.returncode:\n            raise RuntimeError(\'Não foi possível recuperar o resultado anterior; código \' + str(result.returncode))\n        return True\n', 'learn21_eval.py': '"""Precisão e recall por limiar na validação, com cobertura parcial preservada."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip,json\nfrom pathlib import Path\nimport numpy as np\nimport diag11_core as core\nfrom src import politica_alertas as policy\nfrom src.validacao_alertas import corresponder_objetos\nfrom learn21_protocol import SPEC\n\ndef profile(records,cache,threshold,spec=SPEC,details=False):\n    totals=np.zeros(3,dtype=np.int64);alert_totals=np.zeros(3,dtype=np.int64)\n    per=np.zeros((12,3),dtype=np.int64);support=np.zeros(12,dtype=np.int64);high=missing_high=unknown=0\n    images=[]\n    for r in records:\n        if r[\'role\'] not in {\'home_val\',\'val_nova\'}:raise ValueError(\'Somente validações de desenvolvimento podem selecionar pesos e limiares.\')\n        gt=core.with_policy([g for g in r[\'gt\'] if r[\'known_mask\']&(1<<g[\'classe_id\'])],r[\'width\'],r[\'height\'],policy)\n        predictions=core.with_policy([p for p in cache[r[\'id\']] if p[\'confianca\']>=threshold],r[\'width\'],r[\'height\'],policy)\n        pred=[p for p in predictions if r[\'known_mask\']&(1<<p[\'classe_id\'])];unknown+=len(predictions)-len(pred)\n        pairs,missing,extras=corresponder_objetos(gt,pred)\n        counts=np.array([len(pairs),len(missing),len(extras)],dtype=np.int64);totals+=counts\n        for g in gt:support[g[\'classe_id\']]+=1\n        for gi,pi,_ in pairs:per[gt[gi][\'classe_id\'],0]+=1\n        for gi in missing:per[gt[gi][\'classe_id\'],1]+=1\n        for pi in extras:per[pred[pi][\'classe_id\'],2]+=1\n        events=[(gt[gi],pred[pi]) for gi,pi,_ in pairs]+[(gt[gi],None) for gi in missing]+[(None,pred[pi]) for pi in extras]\n        ac=[0,0,0]\n        for g,p in events:\n            ga,pa=core.alert_on(g),core.alert_on(p)\n            ac[0]+=ga and pa;ac[1]+=ga and not pa;ac[2]+=not ga and pa\n            high+=bool(g and g[\'prioridade\']==\'alto\');missing_high+=bool(g and g[\'prioridade\']==\'alto\' and not pa)\n        alert_totals+=ac\n        if details:images.append({\'id\':r[\'id\'],\'coorte\':r[\'coorte\'],\'confianca\':threshold,\'tp_detector\':int(counts[0]),\'fn_detector\':int(counts[1]),\'fp_detector\':int(counts[2]),\n            \'tp_alerta\':ac[0],\'fn_alerta\':ac[1],\'fp_alerta\':ac[2],\n            \'mensagens\':\'\\n\'.join(policy.gerar_mensagem(p,\'medio\')+(\' Classe sem cobertura de referência.\' if not r[\'known_mask\']&(1<<p[\'classe_id\']) else \'\') for p in predictions if core.alert_on(p))})\n    det=core.alert_metrics(*totals);alerts=core.alert_metrics(*alert_totals)\n    score=min((det[\'precision\'] or 0)/spec[\'target_precision\'],(det[\'recall\'] or 0)/spec[\'target_recall\'])\n    rows=[{\'classe_id\':cid,\'classe\':name,\'gt_avaliaveis\':int(support[cid]),\'suporte_reduzido\':bool(support[cid]<30),**core.alert_metrics(*per[cid])} for cid,name in enumerate(core.NAMES)]\n    return {\'threshold\':threshold,\'images\':len(records),\'detector\':det,\'alertas\':alerts,\'alta_ref\':high,\'alta_sem_alerta\':missing_high,\n        \'predicoes_sem_cobertura\':unknown,\'target_met\':bool(det[\'precision\'] is not None and det[\'recall\'] is not None and det[\'precision\']>=spec[\'target_precision\'] and det[\'recall\']>=spec[\'target_recall\']),\n        \'fitness\':float(score+1e-6*(det[\'f1\'] or 0)),\'classes\':rows,\'image_rows\':images}\n\ndef choose(profiles):\n    if not profiles:raise ValueError(\'Grade de validação vazia.\')\n    return max(profiles,key=lambda p:(p[\'fitness\'],-p[\'threshold\']))\n\ndef capture(model,records,device,spec=SPEC):\n    if list(model.names.values())!=core.NAMES:raise ValueError(\'O modelo não tem as 12 classes esperadas, na ordem correta.\')\n    cache={};saturated=0\n    for i,r in enumerate(records,1):\n        if r[\'role\'] not in {\'home_val\',\'val_nova\'}:raise ValueError(\'Tentativa de consultar treino ou teste na seleção.\')\n        result=model.predict(str(r[\'image\']),conf=spec[\'inference_conf\'],imgsz=spec[\'imgsz\'],rect=True,\n            iou=spec[\'nms_iou\'],max_det=spec[\'max_det\'],half=False,batch=1,device=device,verbose=False,save=False)[0]\n        if Path(result.path).resolve()!=Path(r[\'image\']).resolve() or tuple(result.orig_shape)!=(r[\'height\'],r[\'width\']):raise ValueError(\'Previsão retornou outra imagem.\')\n        boxes=result.boxes\n        preds=[] if boxes is None else [{\'classe_id\':int(cid),\'classe\':core.NAMES[int(cid)],\'bbox\':[float(x) for x in box],\'confianca\':float(conf)}\n            for cid,box,conf in zip(boxes.cls.cpu().numpy(),boxes.xyxy.cpu().numpy(),boxes.conf.cpu().numpy())]\n        from conf14_evaluation import validate_predictions\n        validate_predictions(preds,r);cache[r[\'id\']]=preds;saturated+=len(preds)==spec[\'max_det\']\n        if i%250==0:print(f\'Validação {r["coorte"]}: {i}/{len(records)}\',flush=True)\n    return cache,saturated\n\ndef evaluate(model,cohorts,device,spec=SPEC,output=None):\n    caches={};profiles={};saturation={}\n    for cohort,records in cohorts.items():\n        caches[cohort],saturation[cohort]=capture(model,records,device,spec)\n        if cohort==spec[\'primary_cohort\']:\n            profiles[cohort]=[profile(records,caches[cohort],c,spec) for c in spec[\'thresholds\']]\n    selected=choose(profiles[spec[\'primary_cohort\']]);threshold=selected[\'threshold\']\n    summaries={c:profile(records,caches[c],threshold,spec,details=output is not None) for c,records in cohorts.items()}\n    if output is not None:\n        output=Path(output);output.mkdir(parents=True,exist_ok=True)\n        core.save_json(output/\'grade_limiares.json\',profiles[spec[\'primary_cohort\']])\n        for cohort,records in cohorts.items():\n            target=output/cohort;target.mkdir(exist_ok=True)\n            core.save_json(target/\'metricas.json\',summaries[cohort]);core.save_csv(target/\'metricas_por_classe.csv\',summaries[cohort][\'classes\'])\n            core.save_csv(target/\'metricas_por_imagem.csv\',summaries[cohort][\'image_rows\'])\n            with gzip.open(target/\'predicoes.jsonl.gz\',\'wt\',encoding=\'utf-8\',compresslevel=4) as f:\n                for r in records:f.write(json.dumps({\'id\':r[\'id\'],\'image_sha256\':r[\'sha256\'],\'label_sha256\':r[\'label_sha256\'],\'predicoes\':caches[cohort][r[\'id\']]},ensure_ascii=False)+\'\\n\')\n        core.save_json(output/\'limite_captura.json\',{\'confianca_captura\':spec[\'inference_conf\'],\'max_det\':spec[\'max_det\'],\'imagens_atingiram_max_det\':saturation})\n    return selected,summaries,profiles[spec[\'primary_cohort\']]\n', 'learn21_io.py': '"""Resultados leves e retomada verificada, preservando dois checkpoints de treino."""\nfrom pathlib import Path\nimport json,shutil,zipfile\nfrom datetime import datetime,timezone\nimport diag11_core as core\n\ndef backup(drive,output,remote):\n    # Checkpoints de origem são transitórios; os dois slots retomáveis e os pesos\n    # de inferência publicados são os arquivos usados para recuperação.\n    result=drive._call([\'copy\',str(output),drive.root+\'/\'+remote,\'--exclude\',\'**/*.tmp\',\'--exclude\',\'**/*.part\',\n        \'--exclude\',\'**/__pycache__/**\',\'--exclude\',\'treino/weights/**\',\'--exclude\',\'*.zip\'])\n    if result.returncode:print(\'Backup não concluído; os arquivos locais continuam preservados.\',flush=True)\n    return result.returncode==0\n\ndef restore(drive,output,remote,context):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if not drive.exists(remote):return None\n    result=drive._call([\'copy\',drive.root+\'/\'+remote,str(output),\'--exclude\',\'**/*.pt\',\'--exclude\',\'*.zip\',\'--exclude\',\'**/*.tmp\',\'--exclude\',\'treino/weights/**\'])\n    if result.returncode:raise RuntimeError(\'Não foi possível recuperar os metadados da execução anterior.\')\n    options=[]\n    for p in (output/\'retomada\').glob(\'slot_*.json\'):\n        try:\n            meta=json.loads(p.read_text())\n            if meta[\'context_key\']==context:options.append((meta,p))\n        except (ValueError,KeyError):pass\n    for meta,p in sorted(options,key=lambda x:x[0][\'epoch\'],reverse=True):\n        try:\n            drive.fetch(remote+\'/retomada/\'+meta[\'checkpoint\'],p.with_name(meta[\'checkpoint\']),meta[\'sha256\'])\n            best=meta[\'best\']\n            drive.fetch(remote+\'/\'+best[\'weight_relative\'],output/best[\'weight_relative\'],best[\'weight_sha256\'])\n            history_file=output/\'retomada\'/meta[\'history_checkpoint\']\n            drive.fetch(remote+\'/retomada/\'+meta[\'history_checkpoint\'],history_file,meta[\'history_sha256\'])\n        except Exception as exc:\n            print(\'Checkpoint remoto incompleto; tentando o anterior:\',type(exc).__name__,flush=True);continue\n        core.save_json(output/\'historico_treino_21.json\',json.loads(history_file.read_text()))\n        core.save_json(output/\'melhor_modelo_21.json\',best)\n        return meta\n    if options:raise RuntimeError(\'Nenhum checkpoint remoto completo passou na verificação; resultados anteriores foram preservados.\')\n    return None\n\ndef package(output):\n    output=Path(output);target=output/\'pacote_treinamento_21.zip\';tmp=target.with_suffix(\'.tmp\')\n    files=[];integrity=output/\'integridade_pacote_21.json\'\n    for p in sorted(output.rglob(\'*\')):\n        if not p.is_file() or p in {target,tmp,integrity} or p.suffix in {\'.tmp\',\'.part\',\'.jpg\',\'.jpeg\',\'.png\',\'.bmp\',\'.webp\'}:continue\n        rel=p.relative_to(output)\n        if \'__pycache__\' in rel.parts or \'retomada\' in rel.parts or rel.as_posix().startswith(\'treino/weights/\'):continue\n        if p.suffix==\'.pt\' and rel.as_posix()!=\'melhor_modelo.pt\':continue\n        files.append(p)\n    core.save_json(integrity,{\'algoritmo\':\'sha256\',\'arquivos\':{p.relative_to(output).as_posix():core.sha256_file(p) for p in files}})\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n        for p in files+[integrity]:z.write(p,p.relative_to(output).as_posix())\n    tmp.replace(target);return target\n\ndef failure(output,error,stage,drive=None,remote=None):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    core.save_json(output/\'resumo_treinamento_21.json\',{\'concluido\':False,\'status\':\'execucao_interrompida\',\'etapa\':stage,\n        \'erro\':{\'tipo\':type(error).__name__,\'mensagem\':str(error)[:1600]},\'preenchimento_manual_obrigatorio\':False,\n        \'modelo_substituido\':False,\'teste_final_novo_consultado\':False})\n    archive=package(output)\n    if drive and remote:\n        try:backup(drive,output,remote)\n        except Exception:pass\n    return archive\n', 'learn21_prepare.py': '"""Fontes conhecidas, base cacheada e preparação repetível sem rótulos manuais."""\nfrom pathlib import Path\nfrom collections import Counter\nimport json,shutil\nimport diag11_core as core\nimport diag11_data as d11\nimport learn21_data as data\nfrom learn21_protocol import SPEC,digest\n\ndef prepare(drive,expected,local,output,spec=SPEC):\n    local=Path(local);output=Path(output);entries=local/\'entradas\';root=local/\'base\';root.mkdir(parents=True,exist_ok=True)\n    plan={k:expected[k] for k in [\'home_zip\',\'test_home\',\'preparacao\',\'divisao_10\',\'manifesto_pacotes\',\'dataset_novo\',\'zip_novo\',\'pacote_19\']}\n    plan.update({k:spec[k] for k in [\'oi_pool_max\',\'oi_pool_min\',\'oi_candidates\',\'per_primary_class\',\'seed\']})\n    plan[\'versao_preparacao\']=\'21-v1\';plan_key=digest(plan);remote=\'datasets/treino21_\'+plan_key[:24]\n    package=drive.fetch(expected[\'pasta_19\']+\'/pacote_consolidacao_19.zip\',entries/\'pacote19.zip\',expected[\'pacote_19\'])\n    development=data.source19(package,expected,output)\n    prepared_meta=drive.fetch(remote+\'/base_pronta.json\',entries/\'base_pronta.json\',optional=True)\n    if prepared_meta:\n        meta=json.loads(Path(prepared_meta).read_text())\n        if meta[\'plan_key\']!=plan_key:raise ValueError(\'Base remota pronta pertence a outro plano.\')\n        archive=drive.fetch(remote+\'/base_treino_21.zip\',local/\'base_treino_21.zip\',meta[\'zip_sha256\'])\n        records,cohorts,report=data.restore_base(archive,root,plan_key)\n        core.save_json(output/\'preparacao_dados_21.json\',{**report,\'base_reutilizada\':True,\'base_zip_sha256\':meta[\'zip_sha256\']})\n        yaml,mapping=data.input_files(records,cohorts,root)\n        return records,cohorts,yaml,meta[\'zip_sha256\']\n    prep=drive.fetch(expected[\'oi_root\']+\'/configuracao_preparacao.json\',entries/\'configuracao_preparacao.json\')\n    cfg=json.loads(Path(prep).read_text())\n    if data.preparation_sha(cfg)!=expected[\'preparacao\'] or cfg[\'nomes\']!=core.NAMES:raise ValueError(\'Configuração Open Images alterada.\')\n    package_manifest=drive.fetch(expected[\'oi_root\']+\'/manifesto_pacotes.json\',entries/\'manifesto_pacotes.json\',expected[\'manifesto_pacotes\'])\n    packages=json.loads(Path(package_manifest).read_text())\n    if packages[\'config_sha256\']!=expected[\'preparacao\']:raise ValueError(\'Manifesto de lotes incompatível.\')\n    csv=drive.fetch(expected[\'pasta10\']+\'/divisao_openimages.csv\',entries/\'divisao.csv\')\n    js=drive.fetch(expected[\'pasta10\']+\'/divisao_openimages.json\',entries/\'divisao.json\')\n    history=d11.load_history(csv,js,expected[\'divisao_10\'])\n    archive_val=drive.fetch(expected[\'dataset_url\'],entries/\'validacao11.zip\',expected[\'zip_novo\'])\n    metadata,protected=data.dataset_protection(archive_val,expected)\n    near=d11.NearIndex()\n    for r in protected:near.add(int(r[\'phash\'],16),r[\'id\'])\n    for r in development:near.add(int(r[\'phash\']),r[\'id\'])\n    protected_sha={r[\'sha256\'] for r in development}|{r[\'sha256\'] for r in protected}|{r[\'imagem_sha256\'] for r in history[\'rows\'] if r[\'split\']==\'test\'}\n    protected_ids={r[\'id\'] for r in protected}|{r[\'id\'] for r in development}\n    external=data.materialise_validation(archive_val,metadata,development,root)\n    home_zip=drive.fetch(\'datasets/homeobjects_sunrgbd_validado.zip\',entries/\'home.zip\',expected[\'home_zip\'])\n    home,home_val,ex_home,home_hashes=data.prepare_home(home_zip,expected,development,root,near,protected_sha)\n    for r in home:near.add(r[\'phash\'],r[\'id\'])\n    oi,ex_oi,lots=data.prepare_oi(packages,history[\'rows\'],expected,root,drive,near,protected_ids,protected_sha|home_hashes,spec)\n    records=sorted(home+oi,key=lambda r:r[\'id\']);cohorts={\'home_val_historica\':home_val,\'openimages_val_nova\':external}\n    if {r[\'id\'] for r in records}&protected_ids:raise ValueError(\'ID protegido entrou na preparação de treino.\')\n    counts=Counter(g[\'classe_id\'] for r in records for g in r[\'gt\'])\n    absent=[core.NAMES[c] for c in range(12) if not counts[c]]\n    report={\'train_home\':len(home),\'train_openimages\':len(oi),\'train_total\':len(records),\'val_home\':len(home_val),\'val_external\':len(external),\n        \'selecionado_do_train_80k\':True,\'historico_test20k_extraido\':False,\'home_test_extraido\':False,\'reserva20_extraida\':False,\n        \'teste_final_novo_consultado\':False,\'base_reutilizada\':False,\'quarentena_home\':sorted(data.QUARANTINE),\n        \'exclusoes_home\':ex_home,\'exclusoes_oi\':ex_oi,\'caixas_por_classe\':{core.NAMES[c]:counts[c] for c in range(12)},\n        \'classes_sem_exemplo\':absent,\'lotes_lidos\':lots,\'plan_key\':plan_key,\n        \'contexto\':\'Seleção por anotações humanas existentes, sem previsões ou relabeling; preserva cobertura parcial.\'}\n    core.save_json(output/\'preparacao_dados_21.json\',report)\n    if absent:raise ValueError(\'Não há exemplos de treino para as classes: \'+\', \'.join(absent))\n    archive,_=data.freeze_base(records,cohorts,root,plan_key,report);base_sha=core.sha256_file(archive)\n    ready=local/\'base_pronta.json\';core.save_json(ready,{\'plan_key\':plan_key,\'zip_sha256\':base_sha,\'plan\':plan})\n    # Dois arquivos; o recibo só é publicado após a cópia completa do ZIP.\n    print(\'Salvando a base preparada para retomadas...\',flush=True)\n    result=drive._call([\'copyto\',str(archive),drive.root+\'/\'+remote+\'/base_treino_21.zip\'])\n    if not result.returncode:drive._call([\'copyto\',str(ready),drive.root+\'/\'+remote+\'/base_pronta.json\'])\n    else:print(\'Base pronta localmente. A cópia de cache no Drive não foi concluída.\',flush=True)\n    core.save_json(output/\'base_pronta_21.json\',{\'plan_key\':plan_key,\'zip_sha256\':base_sha,\'remote\':remote})\n    yaml,mapping=data.input_files(records,cohorts,root)\n    return records,cohorts,yaml,base_sha\n', 'learn21_protocol.py': '"""Objetivos e orçamento da nova linha de desenvolvimento; sem teste final."""\nfrom __future__ import annotations\nimport hashlib,json\n\nSPEC={\'version\':\'21-yolov8s-supervisao-ampliada-v1\',\'model\':\'yolov8s\',\'classes\':12,\'epochs\':40,\'patience\':8,\n    \'imgsz\':768,\'batch\':8,\'home_per_batch\':2,\'oi_per_batch\':6,\'oi_pool_max\':12000,\'oi_pool_min\':5000,\n    \'oi_candidates\':15000,\'per_primary_class\':1250,\'seed\':21002026,\'optimizer\':\'AdamW\',\'lr0\':.001,\'lrf\':.01,\n    \'warmup_epochs\':3.,\'weight_decay\':.0005,\'cos_lr\':True,\'nbs\':32,\'amp\':False,\n    \'mosaic\':0.,\'mixup\':0.,\'cutmix\':0.,\'copy_paste\':0.,\'degrees\':0.,\'translate\':.1,\'scale\':.3,\n    \'fliplr\':.5,\'flipud\':0.,\'hsv_h\':.015,\'hsv_s\':.4,\'hsv_v\':.3,\'workers\':2,\n    \'oi_uniform_fraction\':.75,\'session_hours\':8.,\'target_precision\':.80,\'target_recall\':.70,\n    \'thresholds\':[.05,.10,.15,.20,.25,.30,.40,.50,.60,.70,.80,.90],\n    \'primary_cohort\':\'openimages_val_nova\',\'inference_conf\':.05,\'nms_iou\':.7,\'match_iou\':.5,\'max_det\':300,\n    \'fitness\':\'max_limiar(min(P/0.80,R/0.70)+0.000001*F1), detecção externa com cobertura; empate pelo menor limiar\',\n    \'selection\':\'melhor fitness externo durante o treino; empate pela época anterior\',\n    \'final_test_consulted\':False,\'historical_300_used_for_tuning\':False}\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False).encode()).hexdigest()\n\ndef order_id(iid,seed):return hashlib.sha256(f\'{seed}|{iid}\'.encode()).hexdigest(),iid\n\ndef choose_candidates(history,forbidden_ids,forbidden_hashes,spec=SPEC):\n    from collections import defaultdict\n    eligible=[r for r in history if r[\'split\']==\'train\' and r[\'id\'] not in forbidden_ids and r[\'imagem_sha256\'] not in forbidden_hashes]\n    groups=defaultdict(list)\n    for row in eligible:groups[int(row[\'classe_primaria\'])].append(row)\n    for cid in groups:groups[cid].sort(key=lambda r:order_id(r[\'id\'],spec[\'seed\']))\n    chosen={}\n    for k in range(spec[\'per_primary_class\']):\n        for cid in sorted(groups,key=lambda c:(len(groups[c]),c)):\n            if len(chosen)>=spec[\'oi_candidates\']:break\n            if k<len(groups[cid]):chosen[groups[cid][k][\'id\']]=groups[cid][k]\n    for row in sorted(eligible,key=lambda r:order_id(r[\'id\'],spec[\'seed\'])):\n        if len(chosen)>=spec[\'oi_candidates\']:break\n        chosen.setdefault(row[\'id\'],row)\n    if len(chosen)<spec[\'oi_pool_min\']:raise ValueError(\'Menos de 5 mil exemplos elegíveis no TRAIN histórico.\')\n    return sorted(chosen.values(),key=lambda r:r[\'id\'])\n\ndef balanced_pool(records,spec=SPEC):\n    from collections import defaultdict\n    groups=defaultdict(list)\n    for row in records:\n        for cid in {g[\'classe_id\'] for g in row[\'gt\']}:groups[cid].append(row)\n    for cid in groups:groups[cid].sort(key=lambda r:order_id(r[\'id\'],spec[\'seed\']+1))\n    chosen={};limit=min(spec[\'oi_pool_max\'],len(records));k=0\n    while len(chosen)<limit:\n        progress=False\n        for cid in sorted(groups,key=lambda c:(len(groups[c]),c)):\n            if k<len(groups[cid]):\n                row=groups[cid][k];chosen.setdefault(row[\'id\'],row);progress=True\n                if len(chosen)>=limit:break\n        if not progress:break\n        k+=1\n    if len(chosen)<spec[\'oi_pool_min\']:raise ValueError(\'A preparação não encontrou supervisão suficiente para a nova rodada.\')\n    return sorted(chosen.values(),key=lambda r:r[\'id\'])\n', 'learn21_report.py': '"""Resultados de desenvolvimento; meta simultânea sem promessa de teste final."""\nfrom pathlib import Path\nimport html,json,shutil\nimport diag11_core as core\nfrom learn21_protocol import SPEC,digest\n\ndef pct(value):return \'indefinido\' if value is None else f\'{100*value:.2f}%\'.replace(\'.\',\',\')\ndef table(rows):\n    if not rows:return \'<p>Sem registros.</p>\'\n    cols=list(rows[0]);return \'<div class="scroll"><table><thead><tr>\'+\'\'.join(\'<th>\'+html.escape(c)+\'</th>\' for c in cols)+\'</tr></thead><tbody>\'+\'\'.join(\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(r.get(c,\'\')))+\'</td>\' for c in cols)+\'</tr>\' for r in rows)+\'</tbody></table></div>\'\n\ndef build(output,run,selected,summaries,profiles,context,expected,spec=SPEC):\n    output=Path(output);prep=json.loads((output/\'preparacao_dados_21.json\').read_text())\n    best=json.loads((output/\'melhor_modelo_21.json\').read_text())\n    source=output/best[\'weight_relative\']\n    if core.sha256_file(source)!=best[\'weight_sha256\']:raise ValueError(\'Peso escolhido mudou antes do relatório.\')\n    shutil.copy2(source,output/\'melhor_modelo.pt\')\n    ready=bool(run[\'concluido\']);target=bool(selected[\'target_met\'])\n    status=\'treino_pausado_retomavel\' if not ready else \'treino_concluido_meta_atingida_validacao\' if target else \'treino_concluido_meta_nao_atingida\'\n    rows=[];comparison=[];classes=[]\n    for cohort,s in summaries.items():\n        row={\'modelo\':\'yolov8s_21\',\'coorte\':cohort,\'confianca\':selected[\'threshold\'],\'imgsz\':spec[\'imgsz\'],\n            **s[\'detector\'],\'gt_avaliaveis\':s[\'detector\'][\'tp\']+s[\'detector\'][\'fn\'],\n            **{\'alertas_\'+k:v for k,v in s[\'alertas\'].items()},\'alta_ref\':s[\'alta_ref\'],\'alta_sem_alerta\':s[\'alta_sem_alerta\'],\n            \'meta_80_70\':s[\'target_met\']}\n        rows.append(row)\n        classes += [{\'coorte\':cohort,\'modelo\':\'yolov8s_21\',\'confianca\':selected[\'threshold\'],**c} for c in s[\'classes\']]\n        prior=output/\'referencia19\'/(cohort+\'_candidato10.json\')\n        if prior.is_file():\n            r=json.loads(prior.read_text());d=r[\'detector_no_limiar\']\n            comparison.append({\'modelo\':\'candidato_10\',\'coorte\':cohort,\'confianca\':.25,\'imgsz\':640,\n                \'precision\':d[\'precision\'],\'recall\':d[\'recall\'],\'f1\':d[\'f1\']})\n        comparison.append({k:row[k] for k in [\'modelo\',\'coorte\',\'confianca\',\'imgsz\',\'precision\',\'recall\',\'f1\']})\n    grid=[{\'confianca\':r[\'threshold\'],**r[\'detector\'],\'fitness\':r[\'fitness\'],\'meta_80_70\':r[\'target_met\']} for r in profiles]\n    core.save_csv(output/\'comparacao_desenvolvimento_21.csv\',comparison)\n    core.save_csv(output/\'resultados_21.csv\',rows);core.save_csv(output/\'classes_21.csv\',classes);core.save_csv(output/\'grade_limiares_21.csv\',grid)\n    summary={\'version\':spec[\'version\'],\'concluido\':ready,\'status\':status,\'context_key\':context,\'training\':run,\n        \'meta_precision\':spec[\'target_precision\'],\'meta_recall\':spec[\'target_recall\'],\'meta_atingida_validacao_externa\':target,\n        \'meta_atingida_teste_final\':None,\'teste_final_novo_consultado\':False,\'reserva20_usada_treino_ou_selecao\':False,\n        \'modelo\':\'yolov8s\',\'pretrained_url\':expected[\'pretrained_url\'],\'pretrained_sha256\':expected[\'pretrained_sha256\'],\n        \'melhor_epoch_desenvolvimento\':best[\'epoch\'],\'peso_final_sha256\':best[\'weight_sha256\'],\'confianca_escolhida\':selected[\'threshold\'],\n        \'resultados\':rows,\'treino_home\':prep[\'train_home\'],\'treino_oi\':prep[\'train_openimages\'],\n        \'selecao\':\'Peso escolhido pelas validações expostas; limiar recalculado no peso exportado, ainda em desenvolvimento.\',\n        \'modelo_substituido\':False,\'preenchimento_manual_obrigatorio\':False,\n        \'limites\':[\'Atingir 80/70 na validação não comprova o resultado em imagens novas.\',\n            \'Rótulos parciais: somente classes conhecidas entram nas métricas; não é precisão de todas as mensagens.\',\n            \'Mudanças simultâneas em capacidade, resolução, amostragem e orçamento; não isolam a causa de eventual ganho.\',\n            \'O sistema não foi validado para distância, perigo físico ou segurança de locomoção.\']}\n    core.save_json(output/\'resumo_treinamento_21.json\',summary)\n    shown=[{\'Modelo\':r[\'modelo\'],\'Conjunto\':r[\'coorte\'],\'Confiança\':r[\'confianca\'],\'Entrada\':r[\'imgsz\'],\n        \'Precisão detector\':pct(r[\'precision\']),\'Recall detector\':pct(r[\'recall\']),\'F1 detector\':pct(r[\'f1\'])} for r in comparison]\n    class_shown=[{\'Conjunto\':r[\'coorte\'],\'Classe\':r[\'classe\'],\'GT\':r[\'gt_avaliaveis\'],\'P\':pct(r[\'precision\']),\'R\':pct(r[\'recall\']),\n        \'TP / FN / FP\':f"{r[\'tp\']} / {r[\'fn\']} / {r[\'fp\']}",\'Suporte <30\':\'sim\' if r[\'suporte_reduzido\'] else \'não\'} for r in classes]\n    message=\'Treino pausado por limite de tempo. Execute Run All novamente para retomar.\' if not ready else \'Meta 80% de precisão e 70% de recall atingida na validação externa. Falta nova avaliação final.\' if target else \'Treino concluído. A meta conjunta 80/70 não foi atingida na validação externa.\'\n    text=f\'\'\'# Rodada 21 — YOLOv8s e supervisão ampliada\n\n{message}\n\nForam usados {prep[\'train_home\']} exemplos Home e {prep[\'train_openimages\']} Open Images, selecionados do TRAIN histórico. As 12 classes e as anotações humanas foram preservadas. Os quatro arquivos Home da quarentena histórica permaneceram excluídos. Foram aplicadas exclusões por coincidência com conjuntos protegidos e semelhança perceptual, documentadas na preparação.\n\nO modelo partiu de pesos públicos do YOLOv8s. O plano prevê até {spec[\'epochs\']} épocas, AdamW com taxa inicial {spec[\'lr0\']}, redução cossenoidal, entrada {spec[\'imgsz\']}, batch {spec[\'batch\']} e cobertura parcial da classificação. Misturas de imagens foram desativadas para preservar essa cobertura. A amostragem destina {spec[\'home_per_batch\']} posições a Home e {spec[\'oi_per_batch\']} a Open Images por batch; em OI, combina passagem uniforme e repetição por classe. Uma época é um orçamento amostrado e não garante exposição a todos os IDs.\n\nForam concluídas {run[\'epochs\']} épocas. O peso selecionado no desenvolvimento veio da época {best[\'epoch\']}. O limiar no peso exportado foi {selected[\'threshold\']}. O critério combina o atendimento simultâneo de precisão 80% e recall 70% na validação externa, com desempate por F1 e menor limiar. Não foram descartadas classes para atingir a meta.\n\n\'\'\'\n    text+=\'| Modelo | Validação | Confiança | Precisão detector | Recall detector | F1 detector |\\n| --- | --- | ---: | ---: | ---: | ---: |\\n\'\n    for r in comparison:text+=f"| {r[\'modelo\']} | {r[\'coorte\']} | {r[\'confianca\']} | {pct(r[\'precision\'])} | {pct(r[\'recall\'])} | {pct(r[\'f1\'])} |\\n"\n    text+=f\'\\nAs validações contêm {prep["val_home"]} imagens Home e {prep["val_external"]} externas já expostas ao desenvolvimento. As 300 imagens da avaliação 20 não foram usadas para treino ou seleção nesta rodada. O resultado permanece de desenvolvimento; outra reserva será necessária para confirmar o desempenho final da nova versão. As métricas de detecção e alertas estão separadas.\\n\'\n    (output/\'texto_resultados_desenvolvimento_21.md\').write_text(text,encoding=\'utf-8\')\n    page=\'\'\'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Treinamento 21</title><style>body{font:16px/1.6 system-ui;margin:0;background:#eef2f6;color:#162739}main{max-width:1450px;margin:auto;padding:28px}.note{padding:20px;background:white;border:1px solid #cbd5e1;border-radius:10px}.scroll{overflow:auto}table{border-collapse:collapse;width:100%;background:white}th,td{padding:9px;border:1px solid #cbd5e1;text-align:left}h1,h2{line-height:1.25}</style><main><p>TCC · Nova linha de desenvolvimento</p><h1>YOLOv8s: precisão e recuperação de objetos</h1>\'\'\'\n    page+=\'<div class="note"><strong>\'+html.escape(message)+\'</strong><p>\'+f"{prep[\'train_home\']} Home + {prep[\'train_openimages\']} Open Images. Épocas concluídas: {run[\'epochs\']}. Melhor época: {best[\'epoch\']}."+\'</p></div>\'\n    page+=\'<h2>Comparação nas validações de desenvolvimento</h2><p>O candidato 10 usou entrada 640 e confiança 0,25. A versão 21 usa entrada 768 e limiar escolhido na validação; a comparação caracteriza as configurações completas.</p>\'+table(shown)\n    page+=\'<h2>Detecção por classe</h2>\'+table(class_shown)+\'<h2>Limitações</h2><ul>\'+\'\'.join(\'<li>\'+html.escape(v)+\'</li>\' for v in summary[\'limites\'])+\'</ul>\'\n    page+=\'<p>Resultados em pacote_treinamento_21.zip. Os checkpoints de retomada permanecem no Drive; o pacote inclui um peso de inferência e os resultados.</p></main></html>\'\n    (output/\'relatorio_treinamento_21.html\').write_text(page,encoding=\'utf-8\')\n    return summary\n', 'learn21_training.py': '"""Treino longo, amostragem de classes e checkpoint retomável por época."""\nfrom __future__ import annotations\nfrom collections import Counter\nfrom copy import deepcopy\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport gc,json,math,shutil,time\nimport random\nfrom contextlib import contextmanager\nimport numpy as np\nimport torch\nfrom torch.utils.data import Sampler\nfrom ultralytics import YOLO\nfrom ultralytics.utils.torch_utils import unwrap_model\nimport diag11_core as core\nfrom exp10_partial import TrainerOpenImages,LoaderFinito,export_standard_checkpoint\nfrom learn21_protocol import SPEC,digest\nimport learn21_eval as evaluation\n\n@contextmanager\ndef isolated_validation():\n    py=random.getstate();np_state=np.random.get_state();cpu=torch.get_rng_state()\n    cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n    threads=torch.get_num_threads();det=torch.are_deterministic_algorithms_enabled();warn=torch.is_deterministic_algorithms_warn_only_enabled()\n    cudnn=(torch.backends.cudnn.benchmark,torch.backends.cudnn.deterministic)\n    try:yield\n    finally:\n        random.setstate(py);np.random.set_state(np_state);torch.set_rng_state(cpu)\n        if cuda is not None:torch.cuda.set_rng_state_all(cuda)\n        torch.set_num_threads(threads);torch.use_deterministic_algorithms(det,warn_only=warn)\n        torch.backends.cudnn.benchmark,torch.backends.cudnn.deterministic=cudnn\n\nclass ClassSourceSampler(Sampler):\n    def __init__(self,home,oi,classes,batch,home_per_batch,seed=42,uniform_fraction=.75):\n        self.home=list(home);self.oi=list(oi);self.classes=classes;self.batch=batch;self.nh=home_per_batch;self.no=batch-home_per_batch\n        if not self.home or not self.oi or self.no<=0 or self.nh<=0:raise ValueError(\'Amostragem exige duas fontes.\')\n        self.seed=seed;self.uniform=uniform_fraction;self.epoch=0;self.batches=math.ceil(len(oi)/self.no)\n        self.by_class={c:[i for i in oi if c in classes[i]] for c in range(12)}\n        self.available=[c for c,v in self.by_class.items() if v]\n        if not self.available:raise ValueError(\'Os rótulos processados de Open Images ficaram sem caixas válidas.\')\n    def set_epoch(self,epoch):self.epoch=int(epoch)\n    def __len__(self):return self.batches*self.batch\n    def __iter__(self):\n        rng=np.random.default_rng(np.random.SeedSequence([self.seed,self.epoch]));needed=self.batches*self.no\n        uniform=[]\n        while len(uniform)<needed:uniform.extend(rng.permutation(self.oi).tolist())\n        oi=[]\n        for i in range(needed):\n            if rng.random()<self.uniform:oi.append(uniform[i])\n            else:oi.append(int(rng.choice(self.by_class[int(rng.choice(self.available))])))\n        home=[]\n        while len(home)<self.batches*self.nh:home.extend(rng.permutation(self.home).tolist())\n        for b in range(self.batches):\n            batch=home[b*self.nh:(b+1)*self.nh]+oi[b*self.no:(b+1)*self.no];rng.shuffle(batch);yield from batch\n\nclass DevValidator:\n    def __init__(self,trainer):\n        self.trainer=trainer;self.args=deepcopy(trainer.args)\n        self.metrics=SimpleNamespace(keys=[\'dev/precision\',\'dev/recall\',\'dev/f1\',\'dev/threshold\',\'dev/target_met\'])\n    def __call__(self,trainer):\n        # A cópia impede que fusão/predict altere o modelo ou a EMA do treino.\n        with isolated_validation():\n            wrapper=YOLO(\'yolov8s.yaml\');wrapper.model=deepcopy(trainer.ema.ema).float().eval();wrapper.predictor=None;wrapper.task=\'detect\'\n            try:selected,summaries,profiles=evaluation.evaluate(wrapper,trainer.dev_cohorts,trainer.device,trainer.spec)\n            finally:del wrapper;gc.collect()\n        trainer.pending={\'selected\':selected,\'summaries\':summaries,\'profiles\':profiles}\n        return {\'dev/precision\':selected[\'detector\'][\'precision\'] or 0.,\'dev/recall\':selected[\'detector\'][\'recall\'] or 0.,\n            \'dev/f1\':selected[\'detector\'][\'f1\'] or 0.,\'dev/threshold\':selected[\'threshold\'],\'dev/target_met\':int(selected[\'target_met\']),\'fitness\':selected[\'fitness\']}\n\nclass LongTrainer(TrainerOpenImages):\n    spec=SPEC;dev_cohorts=None;output=None;context_key=None;backup_callback=None\n    def __init__(self,*args,**kwargs):\n        self.session_started=time.monotonic();self.epoch_started=None;self.sources=Counter();self.unique=set();self.updates_seen=0\n        self.pending=None;self.pause_reason=None\n        super().__init__(*args,**kwargs)\n        path=Path(self.output)/\'historico_treino_21.json\'\n        self.history=json.loads(path.read_text()) if path.is_file() else []\n        self.add_callback(\'on_train_epoch_start\',self.begin_epoch)\n        self.add_callback(\'on_model_save\',self.commit_epoch)\n    def get_dataloader(self,dataset_path,batch_size=16,rank=0,mode=\'train\'):\n        if mode!=\'train\':return super().get_dataloader(dataset_path,batch_size,rank,mode)\n        if batch_size!=self.spec[\'batch\'] or self.world_size>1:raise ValueError(\'Batch ou número de GPUs alterado.\')\n        for key in [\'mosaic\',\'mixup\',\'cutmix\',\'copy_paste\']:\n            if float(getattr(self.args,key,0))!=0:raise ValueError(\'Mistura de imagens incompatível com a cobertura parcial.\')\n        dataset=self.build_dataset(dataset_path,mode,batch_size);home=[];oi=[];classes={}\n        for i,path in enumerate(dataset.im_files):\n            record=self.source_records.get(Path(path).resolve().as_posix())\n            if not record or record[\'role\']!=\'train\':raise ValueError(\'Loader recebeu imagem fora do TRAIN.\')\n            (home if record[\'source\']==\'home\' else oi).append(i)\n            # Rótulos efetivamente processados pelo YOLO, incluindo seus filtros.\n            classes[i]=set(int(x) for x in dataset.labels[i][\'cls\'].reshape(-1))\n        sampler=ClassSourceSampler(home,oi,classes,batch_size,self.spec[\'home_per_batch\'],self.spec[\'seed\'],self.spec[\'oi_uniform_fraction\'])\n        return LoaderFinito(dataset,batch_size=batch_size,sampler=sampler,num_workers=self.args.workers,\n            pin_memory=self.device.type==\'cuda\',collate_fn=dataset.collate_fn,drop_last=False,\n            generator=torch.Generator().manual_seed(self.spec[\'seed\']))\n    def get_validator(self):\n        self.loss_names=(\'box_loss\',\'cls_loss\',\'dfl_loss\');return DevValidator(self)\n    def begin_epoch(self,trainer):\n        self.train_loader.sampler.set_epoch(self.epoch);self.sources.clear();self.unique.clear();self.epoch_started=time.monotonic()\n    def preprocess_batch(self,batch):\n        result=super().preprocess_batch(batch)\n        for name in batch[\'im_file\']:\n            r=self.source_records[Path(name).resolve().as_posix()];self.sources[r[\'source\']]+=1;self.unique.add(r[\'id\'])\n        return result\n    def optimizer_step(self):\n        if not torch.isfinite(self.loss.detach()).all():raise FloatingPointError(\'Loss não finita; checkpoint anterior preservado.\')\n        super().optimizer_step();self.updates_seen+=1\n    def validate(self):\n        metrics=self.validator(self);fitness=metrics.pop(\'fitness\')\n        if not math.isfinite(fitness):raise FloatingPointError(\'Fitness não finito.\')\n        if self.best_fitness is None or fitness>self.best_fitness:self.best_fitness=fitness\n        return metrics,fitness\n    def _handle_nan_recovery(self,epoch):\n        if self.loss is not None and not torch.isfinite(self.loss.detach()).all():raise FloatingPointError(\'Treino não finito; retome pelo último checkpoint válido.\')\n        return False\n    def save_model(self):\n        if any(not torch.isfinite(v).all() for v in self.ema.ema.state_dict().values() if v.is_floating_point()):\n            raise FloatingPointError(\'EMA não finita; o checkpoint anterior será preservado.\')\n        return super().save_model()\n    def final_eval(self):\n        # A rotina padrão remove o optimizer de last.pt. A versão retomável deve\n        # permanecer completa, inclusive quando a pausa ocorre por tempo.\n        return None\n    def commit_epoch(self,trainer):\n        if self.pending is None:raise RuntimeError(\'Época sem avaliação de desenvolvimento.\')\n        output=Path(self.output);epoch=self.epoch+1\n        previous=[r for r in self.history if r[\'epoch\']<epoch]\n        best_old=max(previous,key=lambda r:(r[\'fitness\'],-r[\'epoch\'])) if previous else None\n        best_now=best_old is None or self.fitness>best_old[\'fitness\']\n        best_info=json.loads((output/\'melhor_modelo_21.json\').read_text()) if (output/\'melhor_modelo_21.json\').is_file() else None\n        if best_now:\n            target=output/\'pesos\'/f\'epoca_{epoch:03d}_inferencia.pt\';target.parent.mkdir(exist_ok=True)\n            export_standard_checkpoint(self.last,target)\n            best_info={\'epoch\':epoch,\'fitness\':float(self.fitness),\'threshold\':self.pending[\'selected\'][\'threshold\'],\n                \'weight_relative\':target.relative_to(output).as_posix(),\'weight_sha256\':core.sha256_file(target),\'context_key\':self.context_key}\n            core.save_json(output/\'melhor_modelo_21.json\',best_info)\n        row={\'epoch\':epoch,\'fitness\':float(self.fitness),\'selected\':self.pending[\'selected\'],\n            \'home\':self.pending[\'summaries\'][\'home_val_historica\'],\'exposicoes_fontes\':dict(self.sources),\n            \'imagens_unicas_expostas\':len(self.unique),\'optimizer_steps_nesta_sessao\':self.updates_seen,\n            \'segundos_epoca_com_validacao\':time.monotonic()-self.epoch_started}\n        self.history=previous+[row];core.save_json(output/\'historico_treino_21.json\',self.history)\n        best_epoch=max(self.history,key=lambda r:(r[\'fitness\'],-r[\'epoch\']))[\'epoch\']\n        if epoch-best_epoch>=self.spec[\'patience\']:self.stop=True;self.pause_reason=\'early_stopping\'\n        if time.monotonic()-self.session_started>=self.spec[\'session_hours\']*3600 and epoch<self.spec[\'epochs\'] and self.pause_reason is None:\n            self.stop=True;self.pause_reason=\'limite_tempo_sessao\'\n        if epoch>=self.spec[\'epochs\']:self.pause_reason=\'epochs_concluidas\'\n        slot=epoch%2;folder=output/\'retomada\';folder.mkdir(exist_ok=True)\n        target=folder/f\'slot_{slot}.pt\';tmp=target.with_suffix(\'.tmp\');shutil.copy2(self.last,tmp);tmp.replace(target)\n        history_slot=folder/f\'historico_slot_{slot}.json\';core.save_json(history_slot,self.history)\n        core.save_json(folder/f\'slot_{slot}.json\',{\'epoch\':epoch,\'sha256\':core.sha256_file(target),\n            \'context_key\':self.context_key,\'checkpoint\':target.name,\'best\':best_info,\'pause_reason\':self.pause_reason,\n            \'history_checkpoint\':history_slot.name,\'history_sha256\':core.sha256_file(history_slot)})\n        p=self.pending[\'selected\'];print(f\'Época {epoch}/{self.spec["epochs"]}: P={p["detector"]["precision"]}, R={p["detector"]["recall"]}, confiança={p["threshold"]}, meta={p["target_met"]}\',flush=True)\n        print(f\'Tempo desta época com validação: {row["segundos_epoca_com_validacao"]/60:.1f} min. Restante sem parada antecipada: {(self.spec["epochs"]-epoch)*row["segundos_epoca_com_validacao"]/3600:.1f} h.\',flush=True)\n        if self.backup_callback:self.backup_callback()\n        self.pending=None\n\ndef resume_candidate(output,context_key):\n    valid=[]\n    for path in (Path(output)/\'retomada\').glob(\'slot_*.json\'):\n        try:\n            meta=json.loads(path.read_text());weight=path.with_name(meta[\'checkpoint\'])\n            if meta[\'context_key\']!=context_key:raise ValueError(\'Contexto de retomada diferente.\')\n            best=meta[\'best\'];best_weight=Path(output)/best[\'weight_relative\']\n            history=path.with_name(meta[\'history_checkpoint\'])\n            if (weight.is_file() and core.sha256_file(weight)==meta[\'sha256\']\n                    and best[\'context_key\']==context_key and best_weight.is_file()\n                    and core.sha256_file(best_weight)==best[\'weight_sha256\']\n                    and history.is_file() and core.sha256_file(history)==meta[\'history_sha256\']):valid.append((meta,weight))\n        except (ValueError,OSError,KeyError):continue\n    return max(valid,key=lambda x:x[0][\'epoch\']) if valid else None\n\ndef train(initial,data_yaml,records,cohorts,output,context_key,device,backup=None,spec=SPEC):\n    output=Path(output);output.mkdir(exist_ok=True)\n    source_map={Path(r[\'image\']).resolve().as_posix():r for r in records}\n    for r in records:\n        if r[\'role\']!=\'train\':raise ValueError(\'Conjunto não autorizado no treinamento.\')\n    prior=resume_candidate(output,context_key)\n    if prior:\n        # Uma interrupção durante a publicação da época pode deixar histórico\n        # ou ponteiro mais novos que o último slot completo.\n        core.save_json(output/\'melhor_modelo_21.json\',prior[0][\'best\'])\n        hp=output/\'retomada\'/prior[0][\'history_checkpoint\']\n        core.save_json(output/\'historico_treino_21.json\',json.loads(hp.read_text()))\n    elif list((output/\'retomada\').glob(\'slot_*.json\')):\n        raise RuntimeError(\'Há registros de retomada, mas nenhum checkpoint completo foi validado. Os arquivos foram preservados.\')\n    if prior and (prior[0][\'epoch\']>=spec[\'epochs\'] or prior[0].get(\'pause_reason\')==\'early_stopping\'):\n        return {\'status\':\'treino_concluido_reutilizado\',\'concluido\':True,\'epochs\':prior[0][\'epoch\']}\n    class ConfiguredTrainer(LongTrainer):pass\n    ConfiguredTrainer.source_records=source_map;ConfiguredTrainer.spec=spec;ConfiguredTrainer.dev_cohorts=cohorts\n    ConfiguredTrainer.output=output;ConfiguredTrainer.context_key=context_key;ConfiguredTrainer.backup_callback=staticmethod(backup) if backup else None\n    kwargs={k:spec[k] for k in [\'epochs\',\'patience\',\'imgsz\',\'batch\',\'optimizer\',\'lr0\',\'lrf\',\'warmup_epochs\',\'weight_decay\',\'cos_lr\',\'nbs\',\'amp\',\n        \'mosaic\',\'mixup\',\'cutmix\',\'copy_paste\',\'degrees\',\'translate\',\'scale\',\'fliplr\',\'flipud\',\'hsv_h\',\'hsv_s\',\'hsv_v\',\'workers\',\'seed\']}\n    kwargs.update(data=str(data_yaml),device=device,save_dir=str(output/\'treino\'),exist_ok=True,project=str(output),name=\'treino\',\n        cache=False,rect=False,compile=False,plots=False,save=True,save_period=-1,close_mosaic=0,\n        val=True,pretrained=True,channels_last=False,deterministic=True,verbose=False)\n    if prior:kwargs[\'resume\']=str(prior[1]);model=YOLO(str(prior[1]))\n    else:model=YOLO(str(initial))\n    try:\n        model.train(trainer=ConfiguredTrainer,**kwargs)\n        trainer=model.trainer;paused=trainer.pause_reason==\'limite_tempo_sessao\'\n        return {\'status\':\'pausado_retomavel\' if paused else \'treino_concluido\',\'concluido\':not paused,\n            \'epochs\':trainer.epoch+1,\'motivo\':trainer.pause_reason,\'retomou_de_epoch\':prior[0][\'epoch\'] if prior else None}\n    finally:\n        del model;gc.collect()\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n', 'src/__init__.py': '', 'src/politica_alertas.py': '"""Parâmetros e funções da V2 original, sem reestimar com o TEST."""\nimport math\n\nNOMES_PT = {"bed": "cama", "sofa": "sofá", "chair": "cadeira", "table": "mesa",\n            "wardrobe": "guarda-roupa", "potted plant": "planta em vaso"}\nVERBOS_PT = {"bed": "detectada", "sofa": "detectado", "chair": "detectada", "table": "detectada",\n             "wardrobe": "detectado", "potted plant": "detectada"}\nORDEM_PRIORIDADE = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef avaliar_bbox(classe, bbox, largura, altura, versao="v2"):\n    if versao not in {"v1", "v2"}:\n        raise ValueError("Versão deve ser v1 ou v2.")\n    x1, y1, x2, y2 = map(float, bbox)\n    if largura <= 0 or altura <= 0 or not all(math.isfinite(v) for v in [x1, y1, x2, y2]):\n        raise ValueError("Dimensões ou coordenadas inválidas.")\n    if x2 <= x1 or y2 <= y1:\n        raise ValueError("Caixa sem área positiva.")\n    posicao = classificar_posicao((x1 + x2) / 2, largura)\n    tamanho, area = classificar_tamanho((x2 - x1) * (y2 - y1), largura * altura)\n    vertical, base = classificar_posicao_vertical(y2, altura)\n    resultado = (classificar_risco_v2(classe, posicao, tamanho, vertical) if versao == "v2"\n                 else classificar_risco_v1(classe, posicao, tamanho))\n    nivel = resultado["nivel_risco"]\n    return {"prioridade": nivel if nivel in {"baixo", "medio", "alto"} else "sem_prioridade",\n            "status_original": nivel, "pontuacao": resultado["pontuacao"],\n            "posicao": posicao, "tamanho": tamanho, "regiao_vertical": vertical,\n            "area_relativa": area, "base_vertical_relativa": base,\n            "ajuste_vertical": resultado.get("ajuste_vertical", 0)}\n\n\ndef gerar_mensagem(objeto, nivel_minimo="medio"):\n    if nivel_minimo not in {"medio", "alto"}:\n        raise ValueError("Nível mínimo deve ser medio ou alto.")\n    prioridade = objeto["prioridade"]\n    if ORDEM_PRIORIDADE.get(prioridade, 0) < ORDEM_PRIORIDADE[nivel_minimo]:\n        return ""\n    posicao = {"centro": "central", "esquerda": "esquerda", "direita": "direita"}[objeto["posicao"]]\n    classe = objeto["classe"]\n    return (f"Atenção: {NOMES_PT[classe]} {VERBOS_PT[classe]} na região {posicao} da imagem. "\n            f"Prioridade {\'alta\' if prioridade == \'alto\' else \'média\'}.")\n\n\ndef gerar_mensagens_tabela(df, coluna="nivel_risco_v2"):\n    if df.empty:\n        return []\n    candidatos = df.loc[df[coluna].isin(["medio", "alto"])].copy()\n    candidatos["_ordem"] = candidatos[coluna].map(ORDEM_PRIORIDADE)\n    candidatos = candidatos.sort_values(["_ordem", "ocupacao_imagem"], ascending=[False, False])\n    return [gerar_mensagem({"classe": r["classe"], "posicao": r["posicao"], "prioridade": r[coluna]})\n            for _, r in candidatos.iterrows()]\n\nPESO_OBJETO = {\n    "bed": 3,\n    "sofa": 3,\n    "chair": 3,\n    "table": 3,\n    "wardrobe": 3,\n    "potted plant": 2,\n}\n\nCLASSES_CONTEXTUAIS = {\n    "lamp",\n    "door",\n}\n\nCLASSES_IGNORADAS = {\n    "tv",\n    "laptop",\n    "window",\n    "photo frame",\n}\n\n\nLIMITE_ESQUERDA = 1 / 3\nLIMITE_DIREITA = 2 / 3\n\nLIMITE_AREA_MEDIA = 0.05\nLIMITE_AREA_GRANDE = 0.20\n\nLIMITE_VERTICAL_SUPERIOR = 0.60\nLIMITE_VERTICAL_INFERIOR = 0.80\n\nPESO_POSICAO = {\n    "esquerda": 1,\n    "centro": 3,\n    "direita": 1,\n}\n\nPESO_TAMANHO = {\n    "pequeno": 1,\n    "medio": 2,\n    "grande": 3,\n}\n\nAJUSTE_VERTICAL = {\n    "superior": -2,\n    "intermediaria": -1,\n    "inferior": 0,\n}\n\ndef classificar_posicao(\n    centro_x: float,\n    largura_imagem: int,\n) -> str:\n    centro_normalizado = centro_x / largura_imagem\n\n    if centro_normalizado < LIMITE_ESQUERDA:\n        return "esquerda"\n\n    if centro_normalizado < LIMITE_DIREITA:\n        return "centro"\n\n    return "direita"\n\n\ndef classificar_tamanho(\n    area_box: float,\n    area_imagem: float,\n):\n    proporcao = area_box / area_imagem\n\n    if proporcao < LIMITE_AREA_MEDIA:\n        return "pequeno", proporcao\n\n    if proporcao < LIMITE_AREA_GRANDE:\n        return "medio", proporcao\n\n    return "grande", proporcao\n\n\ndef classificar_posicao_vertical(\n    y2: float,\n    altura_imagem: int,\n):\n    proporcao_vertical = y2 / altura_imagem\n\n    if proporcao_vertical < LIMITE_VERTICAL_SUPERIOR:\n        return "superior", proporcao_vertical\n\n    if proporcao_vertical < LIMITE_VERTICAL_INFERIOR:\n        return "intermediaria", proporcao_vertical\n\n    return "inferior", proporcao_vertical\n\ndef classificar_risco_v1(\n    classe: str,\n    posicao: str,\n    tamanho: str,\n):\n    if classe in CLASSES_IGNORADAS:\n        return {\n            "pontuacao": 0,\n            "nivel_risco": "ignorado",\n            "motivo": "classe sem relevância direta para locomoção",\n        }\n\n    if classe in CLASSES_CONTEXTUAIS:\n        return {\n            "pontuacao": None,\n            "nivel_risco": "contextual",\n            "motivo": "classe exige informação adicional de contexto",\n        }\n\n    if classe not in PESO_OBJETO:\n        return {\n            "pontuacao": None,\n            "nivel_risco": "nao_mapeado",\n            "motivo": "classe não configurada",\n        }\n\n    pontuacao = (\n        PESO_OBJETO[classe]\n        + PESO_POSICAO[posicao]\n        + PESO_TAMANHO[tamanho]\n    )\n\n    if pontuacao <= 4:\n        nivel = "baixo"\n    elif pontuacao <= 6:\n        nivel = "medio"\n    else:\n        nivel = "alto"\n\n    return {\n        "pontuacao": pontuacao,\n        "nivel_risco": nivel,\n        "motivo": (\n            f"{classe} + posição {posicao} + tamanho {tamanho}"\n        ),\n    }\n\ndef classificar_risco_v2(\n    classe: str,\n    posicao: str,\n    tamanho: str,\n    posicao_vertical: str,\n):\n    avaliacao_v1 = classificar_risco_v1(\n        classe=classe,\n        posicao=posicao,\n        tamanho=tamanho,\n    )\n\n    if avaliacao_v1["nivel_risco"] in {\n        "ignorado",\n        "contextual",\n        "nao_mapeado",\n    }:\n        return {\n            "pontuacao": avaliacao_v1["pontuacao"],\n            "nivel_risco": avaliacao_v1["nivel_risco"],\n            "ajuste_vertical": 0,\n            "motivo": avaliacao_v1["motivo"],\n        }\n\n    ajuste_vertical = AJUSTE_VERTICAL[posicao_vertical]\n\n    pontuacao = max(\n        1,\n        avaliacao_v1["pontuacao"] + ajuste_vertical,\n    )\n\n    if pontuacao <= 4:\n        nivel = "baixo"\n    elif pontuacao <= 6:\n        nivel = "medio"\n    else:\n        nivel = "alto"\n\n    return {\n        "pontuacao": pontuacao,\n        "nivel_risco": nivel,\n        "ajuste_vertical": ajuste_vertical,\n        "motivo": (\n            f"{classe} + posição {posicao} "\n            f"+ tamanho {tamanho} "\n            f"+ região vertical {posicao_vertical}"\n        ),\n    }\n', 'src/validacao_alertas.py': '"""Compara detectores na validação Home, com a V2 e os parâmetros do 05d."""\nfrom __future__ import annotations\n\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .politica_alertas import avaliar_bbox\n\nIMGSZ = 640\nCONF_ALERTA = 0.25\nIOU_NMS = 0.70\nIOU_CORRESPONDENCIA = 0.50\nMAX_DETECCOES = 300\n\n\ndef ler_objetos_yolo(caminho, largura, altura, nomes):\n    objetos = []\n    for n, linha in enumerate(Path(caminho).read_text(encoding="utf-8-sig").splitlines(), 1):\n        if not linha.strip():\n            continue\n        partes = linha.split()\n        if len(partes) != 5:\n            raise ValueError(f"Label inválido: {caminho}, linha {n}")\n        valores = [float(p) for p in partes]\n        if not all(math.isfinite(v) for v in valores):\n            raise ValueError(f"Label não finito: {caminho}, linha {n}")\n        id_float, xc, yc, w, h = valores\n        c = int(id_float)\n        bordas = [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]\n        if id_float != c or c not in nomes or w <= 0 or h <= 0 or min(bordas) < -0.0001 or max(bordas) > 1.0001:\n            raise ValueError(f"Classe ou caixa inválida: {caminho}, linha {n}")\n        x1, y1, x2, y2 = np.clip(bordas, 0, 1)\n        objetos.append({"classe_id": c, "classe": nomes[c],\n                        "bbox": [float(x1 * largura), float(y1 * altura),\n                                 float(x2 * largura), float(y2 * altura)]})\n    return objetos\n\n\ndef carregar_home_val(dataset, nomes):\n    dataset = Path(dataset)\n    raiz_imagens = dataset / "images/val"\n    registros = []\n    for imagem in sorted(raiz_imagens.rglob("*")):\n        if not imagem.is_file() or imagem.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:\n            continue\n        relativo = imagem.relative_to(raiz_imagens)\n        if relativo.parts[0] == "sunrgbd":\n            continue\n        label = dataset / "labels/val" / relativo.with_suffix(".txt")\n        with Image.open(imagem) as im:\n            largura, altura = im.size\n            im.verify()\n        registros.append({"imagem": imagem, "nome": relativo.as_posix(),\n                          "largura": largura, "altura": altura,\n                          "gt": ler_objetos_yolo(label, largura, altura, nomes)})\n    if not registros:\n        raise ValueError("Validação Home vazia.")\n    return registros\n\n\ndef iou_matriz(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)), dtype=float)\n    g = np.asarray([o["bbox"] for o in gt], dtype=float)\n    p = np.asarray([o["bbox"] for o in pred], dtype=float)\n    inicio = np.maximum(g[:, None, :2], p[None, :, :2])\n    fim = np.minimum(g[:, None, 2:], p[None, :, 2:])\n    inter = np.prod(np.maximum(0.0, fim - inicio), axis=2)\n    ag = np.prod(np.maximum(0.0, g[:, 2:] - g[:, :2]), axis=1)\n    ap = np.prod(np.maximum(0.0, p[:, 2:] - p[:, :2]), axis=1)\n    uniao = ag[:, None] + ap[None, :] - inter\n    return np.divide(inter, uniao, out=np.zeros_like(inter), where=uniao > 0)\n\n\ndef corresponder_objetos(gt, pred):\n    matriz = iou_matriz(gt, pred)\n    candidatos = []\n    for gi, g in enumerate(gt):\n        for pi, p in enumerate(pred):\n            if g["classe_id"] == p["classe_id"] and matriz[gi, pi] >= IOU_CORRESPONDENCIA:\n                candidatos.append((float(matriz[gi, pi]), gi, pi))\n    candidatos.sort(key=lambda item: (-item[0], item[1], item[2]))\n    usados_g, usados_p, pares = set(), set(), []\n    for iou, gi, pi in candidatos:\n        if gi not in usados_g and pi not in usados_p:\n            pares.append((gi, pi, iou))\n            usados_g.add(gi)\n            usados_p.add(pi)\n    return pares, [i for i in range(len(gt)) if i not in usados_g], [i for i in range(len(pred)) if i not in usados_p]\n\n\ndef contagem_eventos(gt, pred, largura, altura):\n    """Um FN/FP pode decorrer de detecção ausente/extra ou mudança de prioridade."""\n    g = [{**o, **avaliar_bbox(o["classe"], o["bbox"], largura, altura)} for o in gt]\n    p = [{**o, **avaliar_bbox(o["classe"], o["bbox"], largura, altura)} for o in pred]\n    pares, faltantes, extras = corresponder_objetos(g, p)\n    eventos = [(g[gi], p[pi]) for gi, pi, _ in pares]\n    eventos += [(g[gi], None) for gi in faltantes]\n    eventos += [(None, p[pi]) for pi in extras]\n    contagem = {"tp": 0, "fn": 0, "fp": 0, "alta_sem_alerta": 0}\n    for ref, prev in eventos:\n        a = bool(ref and ref["prioridade"] in {"medio", "alto"})\n        b = bool(prev and prev["prioridade"] in {"medio", "alto"})\n        contagem["tp"] += int(a and b)\n        contagem["fn"] += int(a and not b)\n        contagem["fp"] += int(not a and b)\n        contagem["alta_sem_alerta"] += int(bool(ref and ref["prioridade"] == "alto" and not b))\n    return contagem\n\n\ndef avaliar_alertas(modelo, registros, nomes, device=0):\n    modelo_nomes = {int(k): str(v) for k, v in modelo.names.items()}\n    if modelo_nomes != nomes:\n        raise ValueError("Mapa de classes do modelo difere do projeto.")\n    contagem = {"tp": 0, "fn": 0, "fp": 0, "alta_sem_alerta": 0}\n    for numero, r in enumerate(registros, 1):\n        resultado = modelo.predict(str(r["imagem"]), conf=CONF_ALERTA, imgsz=IMGSZ,\n                                   iou=IOU_NMS, max_det=MAX_DETECCOES,\n                                   device=device, verbose=False, save=False)[0]\n        caixas = resultado.boxes\n        pred = [] if caixas is None else [\n            {"classe_id": int(c), "classe": nomes[int(c)], "bbox": [float(v) for v in b]}\n            for c, b in zip(caixas.cls.cpu().numpy(), caixas.xyxy.cpu().numpy())]\n        eventos = contagem_eventos(r["gt"], pred, r["largura"], r["altura"])\n        for k, v in eventos.items():\n            contagem[k] += v\n        if numero % 100 == 0:\n            print(f"Validação dos alertas: {numero}/{len(registros)}")\n    tp, fn, fp = contagem["tp"], contagem["fn"], contagem["fp"]\n    dividir = lambda n, d: n / d if d else None\n    return {"imagens": len(registros), **contagem,\n            "precision_alertas": dividir(tp, tp + fp),\n            "recall_alertas": dividir(tp, tp + fn),\n            "f1_alertas": dividir(2 * tp, 2 * tp + fp + fn),\n            "referencia": "V2 fixa nas anotações; não valida risco físico independentemente",\n            "conf": CONF_ALERTA, "iou_nms": IOU_NMS,\n            "iou_correspondencia": IOU_CORRESPONDENCIA}\n'}
CODIGO_SHA256={'conf14_data.py': 'f73232836278c08edf0f955ae8fe50a33cb1d4eabe2b2cc35a5f410f095bae66', 'conf14_evaluation.py': '3e64c612ffcd7cb75fdf05707affc45cafa91acd1882ab311425c2752ed01656', 'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'exp10_partial.py': '86a36f93aa643853f1268dcc133bed7e7344dea269b9d1d58cf54095538fffae', 'exp12_protocol.py': '257d09b0b2aafe94da28bdcb59859778ffb31e25601db75d1058092b943b5759', 'learn21_data.py': 'ebf38d56464e163adfde767003de7a9540fce865fae441e938f27e63ccf404d6', 'learn21_drive.py': '25300e50e08260a7a0c0601fb65941605d042c97ec23cbb46811e4e0900f14eb', 'learn21_eval.py': 'd000391c866a02e26f30805845884584c15d7aa05fc8acd2e00aa15f2ac07338', 'learn21_io.py': '7dd94a3c139cc375f178d13297c2cc9599261a4bbad83075e63fd324538f10e5', 'learn21_prepare.py': 'afc81b71ce818a3cc63c36d8033e02cc7e56ad7aee1153f90c0378f903737378', 'learn21_protocol.py': 'f7ec4e9f72c040b154f51fbcfd5c1ef55398bb754558c3ae6eff8fccfd8ef8e3', 'learn21_report.py': '85a7ba6c75ed2d1e493811cac5582415a6920d1c453ad2a3f79e8c6f99ae3116', 'learn21_training.py': '7a02dc145309aaa2f799274d08318ccb100190781a9d5ebf4cdf6f195e7369a5', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/politica_alertas.py': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'src/validacao_alertas.py': '1da5dff21a6c6f6ec53d041247ce18d012008c4b26023b52c3fd2745f4c4105d'}
PASTA_CODIGO=PASTA_LOCAL/'codigo';PASTA_CODIGO.mkdir(exist_ok=True)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    if hashlib.sha256(fonte.encode()).hexdigest()!=CODIGO_SHA256[nome]:raise RuntimeError('Código embutido alterado: '+nome)
    compile(fonte,nome,'exec')
    for raiz in [PASTA_CODIGO,PASTA_BOOT/'codigo']:
        destino=raiz/nome;destino.parent.mkdir(parents=True,exist_ok=True);destino.write_text(fonte,encoding='utf-8')
sys.path.insert(0,str(PASTA_CODIGO))
import diag11_core as core
import exp10_partial as parcial
import learn21_data as dados21
import learn21_prepare as preparacao21
import learn21_training as treino21
import learn21_eval as avaliacao21
import learn21_report as relatorio21
import learn21_io as arquivos21
from learn21_protocol import SPEC,digest
from learn21_drive import Drive
from src import politica_alertas as politica
from src import validacao_alertas as associacao
if core.sha256_file(politica.__file__)!=EXPECTED['politica'] or core.sha256_file(associacao.__file__)!=EXPECTED['associador']:
    raise RuntimeError('A política ou o associador de referência foram alterados.')
parcial.check_api();parcial.self_test_bce()
def progresso(etapa):
    global ULTIMA_ETAPA
    agora=time.monotonic();TEMPOS.append({'etapa':etapa,'segundos':agora-ULTIMA_ETAPA,'acumulado_segundos':agora-INICIO});ULTIMA_ETAPA=agora
    core.save_json(PASTA_RESULTADOS/'tempos_execucao_21.json',{'exclui_pip':True,'etapas':TEMPOS})
    print(f'{etapa} | {(agora-INICIO)/60:.1f} min acumulados',flush=True)
def salvar_checkpoint():
    return arquivos21.backup(DRIVE,PASTA_RESULTADOS,'resultados/'+NOME_EXPERIMENTO)
def registrar_falha(exc,etapa):
    pacote=arquivos21.failure(PASTA_RESULTADOS,exc,etapa,DRIVE,'resultados/'+NOME_EXPERIMENTO)
    print('Execução interrompida. Pacote de diagnóstico disponível:',pacote,flush=True)
    display(FileLink(str(pacote)))
core.save_json(PASTA_BOOT/'plano_treinamento_21.json',SPEC)
progresso('Ambiente, política e perda parcial conferidos')


## 2. Conectar ao Drive e conferir o pré-treino

Reutiliza a conexão das rodadas anteriores. Baixa o peso público YOLOv8s e
confere o SHA-256 antes de carregá-lo. O pré-treino parte do modelo público,
não dos pesos nano anteriores. Nenhum modelo em produção é substituído.


In [ ]:
try:
    DRIVE=Drive.from_kaggle()
    if shutil.disk_usage(PASTA_LOCAL).free<12*1024**3:
        raise RuntimeError('São necessários pelo menos 12 GiB livres para preparação, lotes e checkpoints.')
    PESO_INICIAL=dados21.public_weight(PASTA_LOCAL/'yolov8s.pt',EXPECTED)
    print('Peso público YOLOv8s conferido:',core.sha256_file(PESO_INICIAL))
    progresso('Drive conectado e peso público conferido')
except Exception as exc:
    registrar_falha(exc,'2-conexao-e-pretreino');raise


## 3. Preparar automaticamente o treino e as validações

Reutiliza os rótulos humanos já existentes. A seleção percorre metadados dos
80 mil IDs de TRAIN, procura diversidade por classe e prepara até 12 mil
imagens elegíveis de Open Images. O notebook lê os lotes necessários um por
vez; não treina nas 100 mil imagens nem extrai o TEST histórico de 20 mil.

Home TRAIN mantém as exclusões e a quarentena histórica. Home VAL e a
validação externa de mil imagens mantêm suas referências. Identidade, hash
e semelhança perceptual protegem a separação. Da antiga reserva de 300, apenas
identidade/hash/pHash do manifesto servem para exclusão; suas imagens não são
extraídas nem avaliadas. O treino não cria rótulos por previsão.

Uma cópia da base preparada é enviada ao Drive. Nas próximas sessões, ela é
recuperada e conferida, evitando repetir a leitura de todos os lotes.


In [ ]:
try:
    REGISTROS,COORTES,DATA_YAML,BASE_SHA=preparacao21.prepare(DRIVE,EXPECTED,PASTA_LOCAL,PASTA_BOOT,SPEC)
    PREPARACAO=json.loads((PASTA_BOOT/'preparacao_dados_21.json').read_text())
    print('Treino:',PREPARACAO['train_home'],'Home +',PREPARACAO['train_openimages'],'Open Images')
    print('Validações:',{c:len(rows) for c,rows in COORTES.items()})
    progresso('Preparação automática concluída')
except Exception as exc:
    registrar_falha(exc,'3-preparacao-automatica');raise


## 4. Recuperar a execução compatível

A execução é identificada pela base preparada, pelo pré-treino, plano, código
e ambiente. Um checkpoint só é usado após conferir seu contexto e seus bytes,
incluindo o peso de inferência correspondente. Dois slots permitem recuperar
a época anterior caso o último envio tenha sido interrompido. Um checkpoint
local completo tem prioridade sobre a cópia remota.


In [ ]:
try:
    CONTEXTO={'versao':SPEC['version'],'base_zip_sha256':BASE_SHA,'pretrained_sha256':EXPECTED['pretrained_sha256'],
        'fontes_sha256':CODIGO_SHA256,'plano':SPEC,'ambiente':AMBIENTE}
    CONTEXT_KEY=digest(CONTEXTO);NOME_EXPERIMENTO='treino21_'+CONTEXT_KEY[:24]
    PASTA_RESULTADOS=Path('/kaggle/working/resultados_tcc')/NOME_EXPERIMENTO
    PASTA_RESULTADOS.mkdir(parents=True,exist_ok=True)
    if treino21.resume_candidate(PASTA_RESULTADOS,CONTEXT_KEY) is None:
        arquivos21.restore(DRIVE,PASTA_RESULTADOS,'resultados/'+NOME_EXPERIMENTO,CONTEXT_KEY)
    shutil.copytree(PASTA_BOOT,PASTA_RESULTADOS,dirs_exist_ok=True,
        ignore=shutil.ignore_patterns('*.zip','resumo_treinamento_21.json'))
    core.save_json(PASTA_RESULTADOS/'configuracao_treinamento_21.json',{'context_key':CONTEXT_KEY,**CONTEXTO})
    core.save_json(PASTA_RESULTADOS/'fontes_esperadas_21.json',EXPECTED)
    ANTERIOR=treino21.resume_candidate(PASTA_RESULTADOS,CONTEXT_KEY)
    print('Retomada a partir da época:',ANTERIOR[0]['epoch'] if ANTERIOR else 'início, pesos públicos')
    print('Pasta:',PASTA_RESULTADOS)
    salvar_checkpoint()
    progresso('Contexto e retomada conferidos')
except Exception as exc:
    registrar_falha(exc,'4-contexto-e-retomada');raise


## 5. Treinar YOLOv8s e acompanhar a validação externa

Cada batch tem duas posições Home e seis Open Images. Em Open Images,
75% da escolha é uniforme e 25% repete classes para ampliar exposição às
menos frequentes. Uma época é um orçamento amostrado; não significa que todo
ID apareceu uma vez. O relatório registra exposição por fonte e IDs únicos.

São até 40 épocas com AdamW, redução da taxa e entrada 768. Misturas de imagens
estão desativadas para preservar a cobertura parcial. A cada época, o modelo
é validado; pesos e confiança são escolhidos pelo atendimento conjunto de
precisão 80% e recall 70% na validação externa, mantendo as 12 classes.
Home VAL é acompanhado como resultado secundário.

O checkpoint completo, incluindo estado do otimizador, é enviado ao Drive
após cada época. Interromper uma época pode exigir repeti-la na retomada.


In [ ]:
try:
    EXECUCAO=treino21.train(PESO_INICIAL,DATA_YAML,REGISTROS,COORTES,PASTA_RESULTADOS,
        CONTEXT_KEY,DEVICE,backup=salvar_checkpoint,spec=SPEC)
    core.save_json(PASTA_RESULTADOS/'execucao_treinamento_21.json',EXECUCAO)
    progresso('Sessão de treinamento encerrada: '+EXECUCAO['status'])
except Exception as exc:
    registrar_falha(exc,'5-treinamento');raise


## 6. Conferir o melhor peso exportado e gerar os resultados

Carrega o peso de inferência escolhido, confere os bytes e recalcula suas
métricas. A confiança também é conferida nesse peso serializado. Reporta
precisão, recall e F1 do detector, suporte e erros por classe, além dos alertas
V2 separadamente. Classes sem cobertura de referência não são tratadas como
negativas nem contam como acerto.

Se a sessão pausou, os resultados são provisórios e o status permanece
**pausado**. Se o treino terminou sem atingir a meta conjunta, isso aparece
explicitamente no relatório. Esta etapa não usa as 300 imagens antigas como
um novo teste independente.


In [ ]:
try:
    MELHOR=json.loads((PASTA_RESULTADOS/'melhor_modelo_21.json').read_text())
    PESO_MELHOR=PASTA_RESULTADOS/MELHOR['weight_relative']
    if MELHOR['context_key']!=CONTEXT_KEY or core.sha256_file(PESO_MELHOR)!=MELHOR['weight_sha256']:
        raise RuntimeError('O peso escolhido não pertence a esta execução ou foi alterado.')
    from ultralytics import YOLO
    MODELO_FINAL=YOLO(str(PESO_MELHOR))
    SELECIONADO,METRICAS,GRADE=avaliacao21.evaluate(MODELO_FINAL,COORTES,DEVICE,SPEC,
        output=PASTA_RESULTADOS/'avaliacao_melhor_modelo')
    del MODELO_FINAL
    torch.cuda.empty_cache()
    RESUMO=relatorio21.build(PASTA_RESULTADOS,EXECUCAO,SELECIONADO,METRICAS,GRADE,CONTEXT_KEY,EXPECTED,SPEC)
    d=SELECIONADO['detector']
    print('VALIDAÇÃO EXTERNA — precisão:',d['precision'],'| recall:',d['recall'],'| confiança:',SELECIONADO['threshold'])
    if not EXECUCAO['concluido']:
        print('TREINO 21 PAUSADO COM RETOMADA. Execute Run All novamente para continuar.')
    elif SELECIONADO['target_met']:
        print('TREINO 21 CONCLUÍDO — META ATINGIDA NA VALIDAÇÃO. Falta avaliação final independente.')
    else:
        print('TREINO 21 CONCLUÍDO — META CONJUNTA 80/70 NÃO ATINGIDA NA VALIDAÇÃO.')
    progresso('Melhor peso exportado avaliado e relatório gerado')
except Exception as exc:
    registrar_falha(exc,'6-avaliacao-e-relatorio');raise


## 7. Salvar e baixar o pacote

O pacote contém um peso de inferência, configuração, fontes, histórico,
métricas, previsões compactas e relatório. As imagens e os checkpoints de
retomada não entram no pacote; estes últimos são preservados no Drive.

**Artefato: `pacote_treinamento_21.zip`.** Se o status for pausado, Run All retoma
automaticamente. Se for concluído, o relatório informa se a meta foi atingida
e os resultados poderão ser analisados antes do próximo experimento.


In [ ]:
try:
    progresso('Preparando o pacote final desta sessão')
    PACOTE=arquivos21.package(PASTA_RESULTADOS)
    BACKUP_OK=salvar_checkpoint()
    ENVIO=DRIVE._call(['copyto',str(PACOTE),DRIVE.root+'/resultados/'+NOME_EXPERIMENTO+'/'+PACOTE.name])
    if BACKUP_OK and ENVIO.returncode==0:
        print('Resultados e pacote salvos no Drive.')
    else:
        print('O envio ao Drive não foi concluído. Baixe o pacote pelo link e preserve esta saída do Kaggle.')
    print('STATUS:',RESUMO['status'])
    print('Pacote:',PACOTE,'| bytes:',PACOTE.stat().st_size)
    display(FileLink(str(PACOTE)))
    display(FileLink(str(PASTA_RESULTADOS/'relatorio_treinamento_21.html')))
    display(FileLink(str(PASTA_RESULTADOS/'melhor_modelo.pt')))
except Exception as exc:
    registrar_falha(exc,'7-pacote-e-salvamento');raise
